##### About this notebook:

In [1]:
#---------------------------------------------------------------------------------------------------------------------------------------------------
# Author:             Erick Rico Esparza
# Dates:              Mar 12 - Mar 23, 2026
# Description:        Seasonal synthesis, lagged evolution, and light synoptic regime extraction for extreme PM episodes over the Valley of Mexico.
# Organization:       Tampere University / Institute of Atmospheric Sciences and Climate Change (ICAyCC-UNAM)
#---------------------------------------------------------------------------------------------------------------------------------------------------

# Stage 4

## 1. Libraries and setup

### 1.1 Importing libraries

In [74]:
# --- Core
import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from xarray.coding.variables import SerializationWarning

# --- Statistics / analysis
from scipy.stats import ttest_ind, ttest_1samp
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# --- Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
import calendar

# --- Mapping
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

# --- Warnings / display
warnings.filterwarnings("ignore", category=SerializationWarning)

pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

### 1.2 Plotting style and figure defaults

In [3]:
# --- Global plotting style
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.labelweight": "regular",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Default map projection
MAP_PROJ = ccrs.PlateCarree()

# --- Shared map styling helper
def apply_geo_format(
    ax,
    extent=None,
    add_coastlines=True,
    add_borders=True,
    add_states=False,
    add_land=False,
    add_ocean=False,
    gridline_lw=0.5,
    coastline_lw=0.8,
    border_lw=0.6,
    state_lw=0.4
):
    """
    Apply a consistent geographic style to Cartopy axes.
    """
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    if add_land:
        ax.add_feature(cfeature.LAND, facecolor="0.96", zorder=0)
    if add_ocean:
        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    if add_coastlines:
        ax.coastlines(resolution="10m", linewidth=coastline_lw)
    if add_borders:
        ax.add_feature(cfeature.BORDERS, linewidth=border_lw, edgecolor="0.25")
    if add_states:
        ax.add_feature(cfeature.STATES, linewidth=state_lw, edgecolor="0.4")

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=gridline_lw,
        color="gray",
        alpha=0.4,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 5))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 5))

    return ax


# --- Shared title helper
def add_panel_title(ax, main_title, subtitle=None, fontsize=11):
    """
    Add a bold main title and an optional subtitle to an axis.
    """
    if subtitle:
        ax.set_title(f"{main_title}\n{subtitle}", fontsize=fontsize, fontweight="bold")
    else:
        ax.set_title(main_title, fontsize=fontsize, fontweight="bold")


# --- Shared figure export helper
def savefig(fig, outpath, close=False, facecolor="white"):
    """
    Save figure with consistent export settings.
    """
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight", facecolor=facecolor)
    if close:
        plt.close(fig)

### 1.3 Global parameters and notebook options

In [4]:
# --- Stage identity
STAGE_NAME = "Stage 4"
NOTEBOOK_NAME = "notebook_stage4"

# --- Pollutants
POLLUTANTS = ["PM10", "PM2.5"]

# --- Focus months and seasonal logic
FOCUS_MONTHS = [2, 5, 7]   # February, May, July
FOCUS_MONTH_NAMES = {m: calendar.month_name[m] for m in FOCUS_MONTHS}

MONTH_TO_SEASON = {
    1: "DJF", 2: "DJF", 12: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON"
}

# --- Event / lag settings
EVENT_QUANTILE = "p90"
CLEAN_QUANTILE = "p10"

DAY0_DEFINITION = "episode_onset"   # Stage 4 policy: day 0 = start day of p90 episode
LAG_DAYS = np.arange(-3, 3)         # [-3, -2, -1, 0, +1, +2]
LAG_LABELS = [f"{d:+d}" if d != 0 else "0" for d in LAG_DAYS]

# --- Episode persistence classification
PERSISTENT_MIN_DAYS = 2

# --- Clustering settings
CLUSTER_FIELD = "Z500_anomaly"
CLUSTER_K_VALUES = [2, 3, 4]
DEFAULT_N_CLUSTERS = 3
KMEANS_RANDOM_STATE = 42
KMEANS_N_INIT = 50
KMEANS_MAX_ITER = 500

# --- Study-domain settings
# Using the same broad Mexico-centered domain style from prior stages.
DOMAIN_MAIN = {
    "lon_min": -120.0,
    "lon_max": -85.0,
    "lat_min": 12.0,
    "lat_max": 33.0,
}

DOMAIN_EXTENT = [
    DOMAIN_MAIN["lon_min"],
    DOMAIN_MAIN["lon_max"],
    DOMAIN_MAIN["lat_min"],
    DOMAIN_MAIN["lat_max"],
]

# --- Valley of Mexico reference box
DOMAIN_VOM = {
    "lon_min": -100.9,
    "lon_max": -97.4,
    "lat_min": 18.3,
    "lat_max": 20.7,
}

# --- CDMX reference coordinates
CDMX_REF = {"lon": -99.13, "lat": 19.43}

# --- Significance settings
ALPHA = 0.05

# --- Fields expected in Stage 4
FIELD_NAMES = {
    "H500": "hgt500",
    "U500": "uwnd500",
    "V500": "vwnd500",
    "H850": "hgt850",
    "U850": "uwnd850",
    "V850": "vwnd850",
    "PRCP": "precip",
    "SH850": "sh850",      # placeholder; to be downloaded later
}

# --- Output folders (actual paths will be created in Section 2)
OUTPUT_SUBDIRS = ["data", "figures", "tables"]

# --- Notebook behavior flags
EXPORT_FIGURES = True
EXPORT_TABLES = True
EXPORT_DATA = True

RUN_KMEANS = True
RUN_LAG_COMPOSITES = True
RUN_MECHANISM_CHECK = True

# --- Summary
print("Stage configuration loaded:")
print(f"  Stage: {STAGE_NAME}")
print(f"  Pollutants: {POLLUTANTS}")
print(f"  Focus months: {FOCUS_MONTHS} -> {[FOCUS_MONTH_NAMES[m] for m in FOCUS_MONTHS]}")
print(f"  Day 0 definition: {DAY0_DEFINITION}")
print(f"  Lag window: {LAG_DAYS.tolist()}")
print(f"  Candidate k values: {CLUSTER_K_VALUES}")
print(f"  Default k: {DEFAULT_N_CLUSTERS}")
print(f"  Main domain: {DOMAIN_EXTENT}")

Stage configuration loaded:
  Stage: Stage 4
  Pollutants: ['PM10', 'PM2.5']
  Focus months: [2, 5, 7] -> ['February', 'May', 'July']
  Day 0 definition: episode_onset
  Lag window: [-3, -2, -1, 0, 1, 2]
  Candidate k values: [2, 3, 4]
  Default k: 3
  Main domain: [-120.0, -85.0, 12.0, 33.0]


## 2. Paths, I/O rules, and folder structure

This notebook lives inside `Schedule/Stage 4/`.
All core inputs (CSV + NetCDF files) are located **one level above** (i.e., in `Schedule/`). Some other useful files are located inside `Schedule/Stage 2/outputs/tables` and `Schedule/Stage 3/outputs/tables`.

Using `Path` objects and relative paths to keep the project portable.


### 2.1 Working directory and relative paths

In [5]:
# --- Working directories
HERE = Path.cwd()                  # expected: .../Schedule/Stage 4
SCHEDULE_DIR = HERE.parent         # expected: .../Schedule

print(f"HERE:         {HERE}")
print(f"SCHEDULE_DIR: {SCHEDULE_DIR}")

if "Stage 4" not in HERE.name:
    warnings.warn(
        f"You are running from '{HERE.name}'. Expected to run from inside a folder named like 'Stage 4'. "
        "Relative paths may be incorrect."
    )

# --- Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Derived Valley of Mexico box for plotting
SW_lon = DOMAIN_VOM["lon_min"]
SW_lat = DOMAIN_VOM["lat_min"]
NE_lon = DOMAIN_VOM["lon_max"]
NE_lat = DOMAIN_VOM["lat_max"]

VOM_BOX = (
    SW_lon,
    SW_lat,
    NE_lon - SW_lon,
    NE_lat - SW_lat
)

print("VOM_BOX defined from DOMAIN_VOM:")
print(VOM_BOX)

HERE:         c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4
SCHEDULE_DIR: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule
VOM_BOX defined from DOMAIN_VOM:
(-100.9, 18.3, 3.5, 2.3999999999999986)


### 2.2 Input data sources

In [227]:
# ---------------------------------------------------------
# A) Core Schedule-level inputs
# ---------------------------------------------------------
PM_CSV = SCHEDULE_DIR / "pm_cdmx_citymean_daily_2012_2024.csv"

# --- 500 hPa
HGT500_NC = SCHEDULE_DIR / "hgt500_mex_2012_2024.nc"
U500_NC   = SCHEDULE_DIR / "uwnd500_mex_2012_2024.nc"
V500_NC   = SCHEDULE_DIR / "vwnd500_mex_2012_2024.nc"

# --- Daily precipitation
PRECIP_NC = SCHEDULE_DIR / "precip_sfc_mex_2012_2024.nc"

# --- 850 hPa
HGT850_NC = SCHEDULE_DIR / "hgt850_mex_2012_2024.nc"
U850_NC   = SCHEDULE_DIR / "uwnd850_mex_2012_2024.nc"
V850_NC   = SCHEDULE_DIR / "vwnd850_mex_2012_2024.nc"

# --- 850 hPa moisture (to be downloaded later)
# keeping this as a placeholder for now; expected variable family: specific humidity
SH850_NC  = HERE / "shum850_mex_2012_2024.nc"


# ---------------------------------------------------------
# B) Reused Stage 2 outputs
# ---------------------------------------------------------
STAGE2_DIR = SCHEDULE_DIR / "Stage 2"
STAGE2_OUT = STAGE2_DIR / "outputs"
STAGE2_TAB = STAGE2_OUT / "tables"

# --- Event flags / thresholds
PM_P90_FLAGS_CSV = STAGE2_TAB / "pm_daily_with_p90_flags_2012_2024.csv"
P90_THR_CSV      = STAGE2_TAB / "p90_thresholds_by_calendar_month_2012_2024.csv"

# --- Episode-level event tables (key Stage 4 inputs)
EPISODES_PM10_CSV = STAGE2_TAB / "episodes_runs_PM10_p90_with_timefields.csv"
EPISODES_PM25_CSV = STAGE2_TAB / "episodes_runs_PM25_p90_with_timefields.csv"


# ---------------------------------------------------------
# C) Reused Stage 3 outputs
# ---------------------------------------------------------
STAGE3_DIR = SCHEDULE_DIR / "Stage 3"
STAGE3_OUT = STAGE3_DIR / "outputs"
STAGE3_TAB = STAGE3_OUT / "tables"
STAGE3_DAT = STAGE3_OUT / "data"

# --- Stage 3 summary tables used for focus-window selection
STAGE3_P10_THR_CSV = STAGE3_TAB / "stage3_p10_thresholds_by_calendar_month.csv"

STAGE3_NCTRL_PM10_CSV = STAGE3_TAB / "stage3_N_control_p90_vs_p10_PM10.csv"
STAGE3_NCTRL_PM25_CSV = STAGE3_TAB / "stage3_N_control_p90_vs_p10_PM25.csv"
STAGE3_NCTRL_ALL_CSV  = STAGE3_TAB / "stage3_N_control_p90_vs_p10_ALL.csv"

STAGE3_SIGFRAC_MONTH_CSV = STAGE3_TAB / "stage3_frac_significant_area_by_month.csv"
STAGE3_SIGRANK_CSV       = STAGE3_TAB / "stage3_sigfrac_p90_minus_p10_ranking.csv"
STAGE3_SIGTOP5_CSV       = STAGE3_TAB / "stage3_sigfrac_p90_minus_p10_top5.csv"

STAGE3_WHO_MONTH_CSV     = STAGE3_TAB / "stage3_WHO_exceedance_frequency_magnitude_load_by_month.csv"
STAGE3_PRECIP_CLIM_CSV   = STAGE3_TAB / "stage3_precip_climatology_vom_mmday.csv"

# --- Stage 3 monthly baseline climatology in NetCDF
STAGE3_HCLIM_NC = STAGE3_DAT / "stage3_H_clim_mon.nc"
STAGE3_UCLIM_NC = STAGE3_DAT / "stage3_U_clim_mon.nc"
STAGE3_VCLIM_NC = STAGE3_DAT / "stage3_V_clim_mon.nc"

print("Input paths defined.")

Input paths defined.


### 2.3 Output folder structure

In [7]:
OUT_ROOT = HERE / "outputs"
OUT_DATA = OUT_ROOT / "data"
OUT_FIG  = OUT_ROOT / "figures"
OUT_TAB  = OUT_ROOT / "tables"

for folder in [OUT_ROOT, OUT_DATA, OUT_FIG, OUT_TAB]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output folders ready:")
print(f"  OUT_ROOT: {OUT_ROOT}")
print(f"  OUT_DATA: {OUT_DATA}")
print(f"  OUT_FIG : {OUT_FIG}")
print(f"  OUT_TAB : {OUT_TAB}")

Output folders ready:
  OUT_ROOT: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs
  OUT_DATA: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\data
  OUT_FIG : c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures
  OUT_TAB : c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables


### 2.4 Naming conventions for exported products

In [8]:
IO_RULES = """
I/O rules for Stage 4
---------------------
Inputs
- Core inputs are read from one level above the notebook location (Schedule/).
- Reused products are read from:
    - Stage 2/outputs/tables/
    - Stage 3/outputs/tables/
    - Stage 3/outputs/data/

Outputs
- All Stage 4 products are saved inside:
    - Stage 4/outputs/data/
    - Stage 4/outputs/figures/
    - Stage 4/outputs/tables/

Output types
- figures -> PNG, presentation-ready, dpi=300
- tables  -> CSV
- data    -> NetCDF or CSV, depending on product

Notebook cleanliness
- Does not print full large DataFrames by default.
- After each export, will print only a short confirmation:
    'saved: <path>'

Filename conventions
- figures:
    stage4_<topic>_<detail>.png

- tables:
    stage4_<topic>_<detail>.csv

- data:
    stage4_<topic>_<detail>.nc
    stage4_<topic>_<detail>.csv

Examples
- stage4_kmeans_Z500_onsetdays_k3_PM25.png
- stage4_lag_composites_February_PM10.png
- stage4_mechanismcheck_May_PM25.csv
- stage4_regime_counts_by_month.csv
"""

print(IO_RULES)


I/O rules for Stage 4
---------------------
Inputs
- Core inputs are read from one level above the notebook location (Schedule/).
- Reused products are read from:
    - Stage 2/outputs/tables/
    - Stage 3/outputs/tables/
    - Stage 3/outputs/data/

Outputs
- All Stage 4 products are saved inside:
    - Stage 4/outputs/data/
    - Stage 4/outputs/figures/
    - Stage 4/outputs/tables/

Output types
- figures -> PNG, presentation-ready, dpi=300
- tables  -> CSV
- data    -> NetCDF or CSV, depending on product

Notebook cleanliness
- Does not print full large DataFrames by default.
- After each export, will print only a short confirmation:
    'saved: <path>'

Filename conventions
- figures:
    stage4_<topic>_<detail>.png

- tables:
    stage4_<topic>_<detail>.csv

- data:
    stage4_<topic>_<detail>.nc
    stage4_<topic>_<detail>.csv

Examples
- stage4_kmeans_Z500_onsetdays_k3_PM25.png
- stage4_lag_composites_February_PM10.png
- stage4_mechanismcheck_May_PM25.csv
- stage4_regime_cou

### 2.5 Existence check for all critical paths

In [9]:
paths = {
    # --- Working dirs
    "HERE": HERE,
    "SCHEDULE_DIR": SCHEDULE_DIR,

    # --- Core schedule-level files
    "PM_CSV": PM_CSV,
    "HGT500_NC": HGT500_NC,
    "U500_NC": U500_NC,
    "V500_NC": V500_NC,
    "PRECIP_NC": PRECIP_NC,
    "HGT850_NC": HGT850_NC,
    "U850_NC": U850_NC,
    "V850_NC": V850_NC,
    "SH850_NC": SH850_NC,

    # --- Stage 2
    "STAGE2_TAB": STAGE2_TAB,
    "PM_P90_FLAGS_CSV": PM_P90_FLAGS_CSV,
    "P90_THR_CSV": P90_THR_CSV,
    "EPISODES_PM10_CSV": EPISODES_PM10_CSV,
    "EPISODES_PM25_CSV": EPISODES_PM25_CSV,

    # --- Stage 3
    "STAGE3_TAB": STAGE3_TAB,
    "STAGE3_DAT": STAGE3_DAT,
    "STAGE3_P10_THR_CSV": STAGE3_P10_THR_CSV,
    "STAGE3_NCTRL_PM10_CSV": STAGE3_NCTRL_PM10_CSV,
    "STAGE3_NCTRL_PM25_CSV": STAGE3_NCTRL_PM25_CSV,
    "STAGE3_NCTRL_ALL_CSV": STAGE3_NCTRL_ALL_CSV,
    "STAGE3_SIGFRAC_MONTH_CSV": STAGE3_SIGFRAC_MONTH_CSV,
    "STAGE3_SIGRANK_CSV": STAGE3_SIGRANK_CSV,
    "STAGE3_SIGTOP5_CSV": STAGE3_SIGTOP5_CSV,
    "STAGE3_WHO_MONTH_CSV": STAGE3_WHO_MONTH_CSV,
    "STAGE3_PRECIP_CLIM_CSV": STAGE3_PRECIP_CLIM_CSV,
    "STAGE3_HCLIM_NC": STAGE3_HCLIM_NC,
    "STAGE3_UCLIM_NC": STAGE3_UCLIM_NC,
    "STAGE3_VCLIM_NC": STAGE3_VCLIM_NC,

    # --- Outputs
    "OUT_ROOT": OUT_ROOT,
    "OUT_DATA": OUT_DATA,
    "OUT_FIG": OUT_FIG,
    "OUT_TAB": OUT_TAB,
}

for name, path in paths.items():
    print(f"{name:<26} -> {path} | exists={path.exists()}")

HERE                       -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4 | exists=True
SCHEDULE_DIR               -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule | exists=True
PM_CSV                     -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\pm_cdmx_citymean_daily_2012_2024.csv | exists=True
HGT500_NC                  -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\hgt500_mex_2012_2024.nc | exists=True
U500_NC                    -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\uwnd500_mex_2012_2024.nc | exists=True
V500_NC                    -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\vwnd500_mex_2012_2024.nc | exists=True
PRECIP_NC                  -> c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\precip_sfc_mex_2012_2024.nc

## 3. Load Stage 3 inputs and working PM/event tables

### 3.1 Load daily PM and event-definition tables

In [10]:
# --- Base daily PM table
df_pm = pd.read_csv(PM_CSV)

# Parse dates
df_pm["DATE"] = pd.to_datetime(df_pm["DATE"])
df_pm = df_pm.sort_values("DATE").set_index("DATE")

# Ensure numeric PM columns
for col in ["PM10", "PM2.5"]:
    if col in df_pm.columns:
        df_pm[col] = pd.to_numeric(df_pm[col], errors="coerce")

print("df_pm loaded:")
print(f"  shape: {df_pm.shape}")
print(f"  date range: {df_pm.index.min().date()} -> {df_pm.index.max().date()}")
print(f"  columns: {list(df_pm.columns)}")
display(df_pm.head(3))


# --- Stage 2 daily p90-flag table
df_flags = pd.read_csv(PM_P90_FLAGS_CSV)

# Parse date column if present
if "DATE" in df_flags.columns:
    df_flags["DATE"] = pd.to_datetime(df_flags["DATE"])
    df_flags = df_flags.sort_values("DATE").set_index("DATE")

print("\ndf_flags loaded:")
print(f"  shape: {df_flags.shape}")
if isinstance(df_flags.index, pd.DatetimeIndex):
    print(f"  date range: {df_flags.index.min().date()} -> {df_flags.index.max().date()}")
print(f"  columns: {list(df_flags.columns)}")
display(df_flags.head(3))

df_pm loaded:
  shape: (4555, 2)
  date range: 2012-01-01 -> 2024-12-31
  columns: ['PM10', 'PM2.5']


,PM10,PM2.5
DATE,,
2012-01-01,100.14,66.43
2012-01-02,19.29,6.14
2012-01-03,38.00,17.43



df_flags loaded:
  shape: (4555, 6)
  date range: 2012-01-01 -> 2024-12-31
  columns: ['PM10', 'PM2.5', 'thr_p90_PM10', 'thr_p90_PM2.5', 'evt_PM10_p90', 'evt_PM2.5_p90']


,PM10,PM2.5,thr_p90_PM10,thr_p90_PM2.5,evt_PM10_p90,evt_PM2.5_p90
DATE,,,,,,
2012-01-01,100.14,66.43,78.89,41.00,1,1
2012-01-02,19.29,6.14,78.89,41.00,0,0
2012-01-03,38.00,17.43,78.89,41.00,0,0


### 3.2 Load monthly thresholds and event flags from previous stages

In [11]:
# --- Stage 2 p90 thresholds by calendar month
df_thr_p90 = pd.read_csv(P90_THR_CSV)

# --- Stage 3 p10 thresholds by calendar month
df_thr_p10 = pd.read_csv(STAGE3_P10_THR_CSV)

# --- Stage 3 N-control summaries
df_nctrl_pm10 = pd.read_csv(STAGE3_NCTRL_PM10_CSV)
df_nctrl_pm25 = pd.read_csv(STAGE3_NCTRL_PM25_CSV)
df_nctrl_all  = pd.read_csv(STAGE3_NCTRL_ALL_CSV)

# --- Stage 3 focus-window selection tables
df_sigfrac_month = pd.read_csv(STAGE3_SIGFRAC_MONTH_CSV)
df_sigrank       = pd.read_csv(STAGE3_SIGRANK_CSV)
df_sigtop5       = pd.read_csv(STAGE3_SIGTOP5_CSV)
df_who_month     = pd.read_csv(STAGE3_WHO_MONTH_CSV)
df_precip_clim   = pd.read_csv(STAGE3_PRECIP_CLIM_CSV)

print("Threshold and Stage 3 summary tables loaded.\n")

print("df_thr_p90:")
print(f"  shape: {df_thr_p90.shape}")
print(f"  columns: {list(df_thr_p90.columns)}")
display(df_thr_p90.head())

print("\ndf_thr_p10:")
print(f"  shape: {df_thr_p10.shape}")
print(f"  columns: {list(df_thr_p10.columns)}")
display(df_thr_p10.head())

print("\ndf_sigfrac_month:")
print(f"  shape: {df_sigfrac_month.shape}")
print(f"  columns: {list(df_sigfrac_month.columns)}")
display(df_sigfrac_month.head())

print("\ndf_who_month:")
print(f"  shape: {df_who_month.shape}")
print(f"  columns: {list(df_who_month.columns)}")
display(df_who_month.head())

Threshold and Stage 3 summary tables loaded.

df_thr_p90:
  shape: (12, 4)
  columns: ['month', 'month_name', 'p90_PM10', 'p90_PM2.5']


,month,month_name,p90_PM10,p90_PM2.5
0,1,Jan,78.89,41.00
1,2,Feb,75.97,35.70
2,3,Mar,68.29,32.75
3,4,Apr,69.63,37.13
4,5,May,72.88,40.26



df_thr_p10:
  shape: (12, 4)
  columns: ['month', 'month_name', 'thr_p10_PM10', 'thr_p10_PM2.5']


,month,month_name,thr_p10_PM10,thr_p10_PM2.5
0,1,January,34.33,14.13
1,2,February,35.86,14.30
2,3,March,36.00,15.43
3,4,April,35.43,17.33
4,5,May,31.33,18.04



df_sigfrac_month:
  shape: (24, 4)
  columns: ['month', 'month_name', 'pollutant', 'frac_area_p_lt_0p05']


,month,month_name,pollutant,frac_area_p_lt_0p05
0,1,January,PM2.5,0.39
1,2,February,PM2.5,0.49
2,3,March,PM2.5,0.50
3,4,April,PM2.5,0.35
4,5,May,PM2.5,0.35



df_who_month:
  shape: (12, 8)
  columns: ['month', 'month_name', 'PM10_exceed_pct', 'PM10_excess_mag_mean', 'PM10_excess_load_sum', 'PM25_exceed_pct', 'PM25_excess_mag_mean', 'PM25_excess_load_sum']


,month,month_name,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum
0,1,January,74.26,19.80,"5,444.50",87.67,14.87,"4,863.52"
1,2,February,72.80,17.19,"4,418.24",88.67,11.72,"3,666.80"
2,3,March,72.82,13.80,"3,987.90",91.52,10.11,"3,700.20"
3,4,April,69.03,15.01,"3,902.51",96.59,12.70,"4,634.77"
4,5,May,57.80,15.55,"3,297.54",97.04,14.28,"5,155.66"


### 3.3 Load Stage 2 episode-run tables for p90 events

In [12]:
# --- Episode-level tables (key Stage 4 inputs)
df_ep_pm10 = pd.read_csv(EPISODES_PM10_CSV)
df_ep_pm25 = pd.read_csv(EPISODES_PM25_CSV)

# Parse datetime fields
episode_date_cols = ["start", "end"]

for df_ep in [df_ep_pm10, df_ep_pm25]:
    for col in episode_date_cols:
        if col in df_ep.columns:
            df_ep[col] = pd.to_datetime(df_ep[col])

# Sort by episode start
if "start" in df_ep_pm10.columns:
    df_ep_pm10 = df_ep_pm10.sort_values("start").reset_index(drop=True)

if "start" in df_ep_pm25.columns:
    df_ep_pm25 = df_ep_pm25.sort_values("start").reset_index(drop=True)

# --- Add day-0 field explicitly for Stage 4 logic
df_ep_pm10["day0"] = df_ep_pm10["start"]
df_ep_pm25["day0"] = df_ep_pm25["start"]

# --- Add persistence class
df_ep_pm10["persistence_class"] = np.where(df_ep_pm10["duration_days"] >= PERSISTENT_MIN_DAYS, "persistent", "one_day")
df_ep_pm25["persistence_class"] = np.where(df_ep_pm25["duration_days"] >= PERSISTENT_MIN_DAYS, "persistent", "one_day")

print("Episode-run tables loaded.\n")

print("df_ep_pm10:")
print(f"  shape: {df_ep_pm10.shape}")
print(f"  columns: {list(df_ep_pm10.columns)}")
display(df_ep_pm10.head())

print("\ndf_ep_pm25:")
print(f"  shape: {df_ep_pm25.shape}")
print(f"  columns: {list(df_ep_pm25.columns)}")
display(df_ep_pm25.head())

Episode-run tables loaded.

df_ep_pm10:
  shape: (250, 9)
  columns: ['episode_id', 'start', 'end', 'duration_days', 'start_year', 'start_month', 'start_season', 'day0', 'persistence_class']


,episode_id,start,end,duration_days,start_year,start_month,start_season,day0,persistence_class
0,1,2012-01-01,2012-01-01,1,2012,1,DJF,2012-01-01,one_day
1,2,2012-01-20,2012-01-21,2,2012,1,DJF,2012-01-20,persistent
2,3,2012-01-23,2012-01-24,2,2012,1,DJF,2012-01-23,persistent
3,4,2012-03-01,2012-03-03,3,2012,3,MAM,2012-03-01,persistent
4,5,2012-03-06,2012-03-09,4,2012,3,MAM,2012-03-06,persistent



df_ep_pm25:
  shape: (286, 9)
  columns: ['episode_id', 'start', 'end', 'duration_days', 'start_year', 'start_month', 'start_season', 'day0', 'persistence_class']


,episode_id,start,end,duration_days,start_year,start_month,start_season,day0,persistence_class
0,1,2012-01-01,2012-01-01,1,2012,1,DJF,2012-01-01,one_day
1,2,2012-01-21,2012-01-21,1,2012,1,DJF,2012-01-21,one_day
2,3,2012-01-23,2012-01-23,1,2012,1,DJF,2012-01-23,one_day
3,4,2012-02-25,2012-02-26,2,2012,2,DJF,2012-02-25,persistent
4,5,2012-03-02,2012-03-03,2,2012,3,MAM,2012-03-02,persistent


### 3.4 Integrity checks for reused inputs

In [13]:
# ---------------------------------------------------------
# A) Helper for compact integrity checks
# ---------------------------------------------------------
def summarize_table(df, name, date_col=None):
    print(f"\n{name}")
    print("-" * len(name))
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print("missing values (top 10):")
    print(df.isna().sum().sort_values(ascending=False).head(10))

    if date_col is not None and date_col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[date_col]):
            print(f"{date_col} range: {df[date_col].min()} -> {df[date_col].max()}")


# ---------------------------------------------------------
# B) Core integrity checks
# ---------------------------------------------------------
summarize_table(df_pm.reset_index(), "Base daily PM table", date_col="DATE")
summarize_table(df_flags.reset_index(), "Daily p90 flags table", date_col="DATE")
summarize_table(df_thr_p90, "Monthly p90 threshold table")
summarize_table(df_thr_p10, "Monthly p10 threshold table")
summarize_table(df_ep_pm10, "PM10 episode table", date_col="start")
summarize_table(df_ep_pm25, "PM2.5 episode table", date_col="start")
summarize_table(df_sigfrac_month, "Stage 3 monthly significance-area summary")
summarize_table(df_sigrank, "Stage 3 significance ranking table")
summarize_table(df_who_month, "Stage 3 WHO monthly summary")
summarize_table(df_precip_clim, "Stage 3 precipitation climatology summary")


# ---------------------------------------------------------
# C) Logic checks for Stage 4 event framework
# ---------------------------------------------------------
required_episode_cols = [
    "episode_id", "start", "end", "duration_days",
    "start_year", "start_month", "start_season", "day0", "persistence_class"
]

for name, df_ep in [("PM10", df_ep_pm10), ("PM2.5", df_ep_pm25)]:
    missing_cols = [c for c in required_episode_cols if c not in df_ep.columns]
    print(f"\n{name} episode table required columns missing: {missing_cols if missing_cols else 'None'}")

    if "duration_days" in df_ep.columns:
        print(f"{name} duration_days summary:")
        print(df_ep["duration_days"].describe())

    if "persistence_class" in df_ep.columns:
        print(f"{name} persistence_class counts:")
        print(df_ep["persistence_class"].value_counts(dropna=False))

    if "start_month" in df_ep.columns:
        print(f"{name} start_month counts:")
        print(df_ep["start_month"].value_counts().sort_index())


# ---------------------------------------------------------
# D) Consistency checks for selected focus months
# ---------------------------------------------------------
focus_pm10 = df_ep_pm10[df_ep_pm10["start_month"].isin(FOCUS_MONTHS)].copy()
focus_pm25 = df_ep_pm25[df_ep_pm25["start_month"].isin(FOCUS_MONTHS)].copy()

print("\nFocus-month episode counts (Stage 4 months):")
print("PM10:")
print(focus_pm10["start_month"].value_counts().sort_index())

print("\nPM2.5:")
print(focus_pm25["start_month"].value_counts().sort_index())


# ---------------------------------------------------------
# E) Merge-ready onset tables for later sections
# ---------------------------------------------------------
evt_pm10_onsets = df_ep_pm10[[
    "episode_id", "day0", "start", "end", "duration_days",
    "start_year", "start_month", "start_season", "persistence_class"
]].copy()

evt_pm25_onsets = df_ep_pm25[[
    "episode_id", "day0", "start", "end", "duration_days",
    "start_year", "start_month", "start_season", "persistence_class"
]].copy()

print("\nOnset tables prepared for later sections:")
print(f"  evt_pm10_onsets: {evt_pm10_onsets.shape}")
print(f"  evt_pm25_onsets: {evt_pm25_onsets.shape}")
display(evt_pm10_onsets.head(3))
display(evt_pm25_onsets.head(3))


Base daily PM table
-------------------
shape: (4555, 3)
columns: ['DATE', 'PM10', 'PM2.5']
missing values (top 10):
DATE     0
PM10     0
PM2.5    0
dtype: int64
DATE range: 2012-01-01 00:00:00 -> 2024-12-31 00:00:00

Daily p90 flags table
---------------------
shape: (4555, 7)
columns: ['DATE', 'PM10', 'PM2.5', 'thr_p90_PM10', 'thr_p90_PM2.5', 'evt_PM10_p90', 'evt_PM2.5_p90']
missing values (top 10):
DATE             0
PM10             0
PM2.5            0
thr_p90_PM10     0
thr_p90_PM2.5    0
evt_PM10_p90     0
evt_PM2.5_p90    0
dtype: int64
DATE range: 2012-01-01 00:00:00 -> 2024-12-31 00:00:00

Monthly p90 threshold table
---------------------------
shape: (12, 4)
columns: ['month', 'month_name', 'p90_PM10', 'p90_PM2.5']
missing values (top 10):
month         0
month_name    0
p90_PM10      0
p90_PM2.5     0
dtype: int64

Monthly p10 threshold table
---------------------------
shape: (12, 4)
columns: ['month', 'month_name', 'thr_p10_PM10', 'thr_p10_PM2.5']
missing values (top 10

,episode_id,day0,start,end,duration_days,start_year,start_month,start_season,persistence_class
0,1,2012-01-01,2012-01-01,2012-01-01,1,2012,1,DJF,one_day
1,2,2012-01-20,2012-01-20,2012-01-21,2,2012,1,DJF,persistent
2,3,2012-01-23,2012-01-23,2012-01-24,2,2012,1,DJF,persistent


,episode_id,day0,start,end,duration_days,start_year,start_month,start_season,persistence_class
0,1,2012-01-01,2012-01-01,2012-01-01,1,2012,1,DJF,one_day
1,2,2012-01-21,2012-01-21,2012-01-21,1,2012,1,DJF,one_day
2,3,2012-01-23,2012-01-23,2012-01-23,1,2012,1,DJF,one_day


## 4. Load reanalysis fields and preprocess analysis-ready datasets

### 4.1 Load 500 hPa fields

In [14]:
# =========================================================
# 4.1 Load 500 hPa fields
# =========================================================

def _find_coord_name(ds, candidates):
    """Return the first matching coordinate/dimension/variable name from candidates."""
    all_names = list(ds.coords) + list(ds.dims) + list(ds.variables)
    for cand in candidates:
        if cand in all_names:
            return cand
    return None


def _standardize_latlon_time(ds):
    """
    Standardize dataset coordinate names to time / lat / lon whenever possible.
    """
    rename_map = {}

    time_name = _find_coord_name(ds, ["time", "Time", "valid_time", "date"])
    lat_name  = _find_coord_name(ds, ["lat", "latitude", "y"])
    lon_name  = _find_coord_name(ds, ["lon", "longitude", "x"])

    if time_name and time_name != "time":
        rename_map[time_name] = "time"
    if lat_name and lat_name != "lat":
        rename_map[lat_name] = "lat"
    if lon_name and lon_name != "lon":
        rename_map[lon_name] = "lon"

    if rename_map:
        ds = ds.rename(rename_map)

    return ds


def _is_1d_coord(ds, name):
    return (name in ds.coords) and (ds[name].ndim == 1)


def _convert_lon_to_180_if_needed(ds):
    """
    Convert longitude from 0..360 to -180..180 for both 1-D and 2-D lon coordinates.
    """
    if "lon" not in ds.coords:
        return ds

    lon = ds["lon"]

    try:
        lon_max = float(np.nanmax(lon.values))
    except Exception:
        return ds

    if lon_max > 180:
        lon_new = (((lon + 180) % 360) - 180)
        ds = ds.assign_coords(lon=lon_new)

        # Only sort if lon is truly 1-D
        if ds["lon"].ndim == 1:
            ds = ds.sortby("lon")

    return ds


def _subset_domain(ds, lon_min, lon_max, lat_min, lat_max):
    """
    Subset dataset to DOMAIN_MAIN.
    Works for:
    - 1-D lat/lon coordinates
    - 2-D lat/lon coordinates
    """
    if ("lat" not in ds.coords) or ("lon" not in ds.coords):
        return ds

    lat = ds["lat"]
    lon = ds["lon"]

    # --- Case 1: standard 1-D lat/lon
    if lat.ndim == 1 and lon.ndim == 1:
        lat_vals = lat.values

        if lat_vals[0] < lat_vals[-1]:
            lat_slice = slice(lat_min, lat_max)
        else:
            lat_slice = slice(lat_max, lat_min)

        ds = ds.sel(lat=lat_slice, lon=slice(lon_min, lon_max))
        return ds

    # --- Case 2: curvilinear / 2-D lat-lon
    mask = (
        (lon >= lon_min) & (lon <= lon_max) &
        (lat >= lat_min) & (lat <= lat_max)
    )
    ds = ds.where(mask, drop=True)
    return ds


def _guess_data_var(ds, preferred_keywords=None):
    """
    Guess the main data variable from a dataset.
    Priority:
    1) variable names containing preferred keywords
    2) first variable with time in dims
    3) first data_var
    """
    preferred_keywords = preferred_keywords or []
    data_vars = list(ds.data_vars)

    if len(data_vars) == 1:
        return data_vars[0]

    for v in data_vars:
        vlow = v.lower()
        if any(k in vlow for k in preferred_keywords):
            return v

    for v in data_vars:
        dims = set(ds[v].dims)
        if "time" in dims:
            return v

    return data_vars[0]


def _open_field_dataset(path, preferred_keywords=None):
    """
    Open a NetCDF, standardize coords, subset domain, and guess main variable.
    Safe for both 1-D and 2-D lat/lon.
    """
    ds = xr.open_dataset(path)
    ds = _standardize_latlon_time(ds)
    ds = _convert_lon_to_180_if_needed(ds)

    # Only sort coordinates if they are 1-D
    if _is_1d_coord(ds, "lat"):
        ds = ds.sortby("lat")
    if _is_1d_coord(ds, "lon"):
        ds = ds.sortby("lon")
    if _is_1d_coord(ds, "time"):
        ds = ds.sortby("time")

    ds = _subset_domain(
        ds,
        DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
        DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]
    )

    var = _guess_data_var(ds, preferred_keywords=preferred_keywords)
    return ds, var

In [15]:
# --- 500 hPa
ds_h500, var_h500 = _open_field_dataset(HGT500_NC, preferred_keywords=["hgt", "z", "gh"])
ds_u500, var_u500 = _open_field_dataset(U500_NC, preferred_keywords=["uwnd", "u"])
ds_v500, var_v500 = _open_field_dataset(V500_NC, preferred_keywords=["vwnd", "v"])

print("500 hPa datasets loaded:")
print(f"  H500 -> var='{var_h500}', dims={ds_h500[var_h500].dims}, shape={ds_h500[var_h500].shape}")
print(f"  U500 -> var='{var_u500}', dims={ds_u500[var_u500].dims}, shape={ds_u500[var_u500].shape}")
print(f"  V500 -> var='{var_v500}', dims={ds_v500[var_v500].dims}, shape={ds_v500[var_v500].shape}")

if "lat" in ds_h500.coords:
    print(f"  lat dims: {ds_h500['lat'].dims}, shape={ds_h500['lat'].shape}")
if "lon" in ds_h500.coords:
    print(f"  lon dims: {ds_h500['lon'].dims}, shape={ds_h500['lon'].shape}")

500 hPa datasets loaded:
  H500 -> var='hgt', dims=('time', 'y', 'x'), shape=(4749, 90, 140)
  U500 -> var='uwnd', dims=('time', 'y', 'x'), shape=(4749, 90, 140)
  V500 -> var='vwnd', dims=('time', 'y', 'x'), shape=(4749, 90, 140)
  lat dims: ('y', 'x'), shape=(90, 140)
  lon dims: ('y', 'x'), shape=(90, 140)


In [16]:
print(ds_h500)
print(ds_u500)
print(ds_v500)

<xarray.Dataset> Size: 239MB
Dimensions:  (time: 4749, y: 90, x: 140)
Coordinates:
  * time     (time) datetime64[ns] 38kB 2012-01-01 2012-01-02 ... 2024-12-31
  * y        (y) float32 360B 9.739e+04 1.299e+05 ... 2.954e+06 2.987e+06
  * x        (x) float32 560B 3.96e+06 3.993e+06 ... 8.44e+06 8.473e+06
    level    float32 4B 500.0
    lat      (y, x) float32 50kB 10.99 11.03 11.07 11.1 ... 30.93 30.82 30.72
    lon      (y, x) float32 50kB -119.6 -119.3 -119.1 ... -78.67 -78.37 -78.07
Data variables:
    hgt      (time, y, x) float32 239MB nan nan nan nan nan ... nan nan nan nan
<xarray.Dataset> Size: 239MB
Dimensions:  (time: 4749, y: 90, x: 140)
Coordinates:
  * time     (time) datetime64[ns] 38kB 2012-01-01 2012-01-02 ... 2024-12-31
  * y        (y) float32 360B 9.739e+04 1.299e+05 ... 2.954e+06 2.987e+06
  * x        (x) float32 560B 3.96e+06 3.993e+06 ... 8.44e+06 8.473e+06
    level    float32 4B 500.0
    lat      (y, x) float32 50kB 10.99 11.03 11.07 11.1 ... 30.93 30.82 30.

### 4.2 Load daily precipitation

In [17]:
ds_pr, var_pr = _open_field_dataset(PRECIP_NC, preferred_keywords=["precip", "prate", "rain", "apcp", "pcp"])

print("Precipitation dataset loaded:")
print(f"  PRCP -> var='{var_pr}', dims={ds_pr[var_pr].dims}, shape={ds_pr[var_pr].shape}")

Precipitation dataset loaded:
  PRCP -> var='precip_mmday', dims=('time', 'y', 'x'), shape=(4749, 90, 140)


### 4.3 Load 850 hPa fields

In [18]:
ds_h850, var_h850 = _open_field_dataset(HGT850_NC, preferred_keywords=["hgt", "z", "gh"])
ds_u850, var_u850 = _open_field_dataset(U850_NC, preferred_keywords=["uwnd", "u"])
ds_v850, var_v850 = _open_field_dataset(V850_NC, preferred_keywords=["vwnd", "v"])

print("850 hPa datasets loaded:")
print(f"  H850 -> var='{var_h850}', dims={ds_h850[var_h850].dims}, shape={ds_h850[var_h850].shape}")
print(f"  U850 -> var='{var_u850}', dims={ds_u850[var_u850].dims}, shape={ds_u850[var_u850].shape}")
print(f"  V850 -> var='{var_v850}', dims={ds_v850[var_v850].dims}, shape={ds_v850[var_v850].shape}")

850 hPa datasets loaded:
  H850 -> var='hgt', dims=('time', 'y', 'x'), shape=(4749, 90, 140)
  U850 -> var='uwnd', dims=('time', 'y', 'x'), shape=(4749, 90, 140)
  V850 -> var='vwnd', dims=('time', 'y', 'x'), shape=(4749, 90, 140)


### 4.4 Load 850 hPa moisture field

In [228]:
ds_sh850 = None
var_sh850 = None

if SH850_NC.exists():
    ds_sh850, var_sh850 = _open_field_dataset(SH850_NC, preferred_keywords=["shum", "spfh", "q", "sh"])
    print("850 hPa moisture dataset loaded:")
    print(f"  SH850 -> var='{var_sh850}', dims={ds_sh850[var_sh850].dims}, shape={ds_sh850[var_sh850].shape}")
else:
    print("SH850_NC not found yet. Moisture field will be added later once downloaded.")

850 hPa moisture dataset loaded:
  SH850 -> var='shum', dims=('time', 'y', 'x'), shape=(4749, 90, 140)


### 4.5 Temporal alignment and preprocessing

In [20]:
# --- Extract DataArrays with standardized names
H500 = ds_h500[var_h500].rename("H500")
U500 = ds_u500[var_u500].rename("U500")
V500 = ds_v500[var_v500].rename("V500")

PRCP = ds_pr[var_pr].rename("PRCP")

# --- Match precipitation footprint to the reanalysis mask
PRCP = PRCP.where(np.isfinite(H500))

# --- Clean negative precipitation artifacts without destroying NaNs
PRCP = xr.where(PRCP < 0, 0, PRCP)

H850 = ds_h850[var_h850].rename("H850")
U850 = ds_u850[var_u850].rename("U850")
V850 = ds_v850[var_v850].rename("V850")

if ds_sh850 is not None:
    SH850 = ds_sh850[var_sh850].rename("SH850")
else:
    SH850 = None

In [21]:
# --- Keep only date (drop sub-daily time if present)
def _normalize_time_daily(da):
    if "time" not in da.coords:
        return da
    da = da.copy()
    da = da.assign_coords(time=pd.to_datetime(da["time"].values).normalize())
    return da

In [22]:
H500 = _normalize_time_daily(H500)
U500 = _normalize_time_daily(U500)
V500 = _normalize_time_daily(V500)

PRCP = _normalize_time_daily(PRCP)

H850 = _normalize_time_daily(H850)
U850 = _normalize_time_daily(U850)
V850 = _normalize_time_daily(V850)

if SH850 is not None:
    SH850 = _normalize_time_daily(SH850)

# --- Inner-join time intersection across required fields
required_arrays = [H500, U500, V500, PRCP, H850, U850, V850]
aligned_required = xr.align(*required_arrays, join="inner")

H500, U500, V500, PRCP, H850, U850, V850 = aligned_required

if SH850 is not None:
    H500, U500, V500, PRCP, H850, U850, V850, SH850 = xr.align(
        H500, U500, V500, PRCP, H850, U850, V850, SH850,
        join="inner"
    )

# --- Also align to PM daily date range
pm_time = pd.DatetimeIndex(df_pm.index).normalize()
time_mask = np.isin(pd.to_datetime(H500["time"].values).normalize(), pm_time)

H500 = H500.sel(time=time_mask)
U500 = U500.sel(time=time_mask)
V500 = V500.sel(time=time_mask)
PRCP = PRCP.sel(time=time_mask)
H850 = H850.sel(time=time_mask)
U850 = U850.sel(time=time_mask)
V850 = V850.sel(time=time_mask)

if SH850 is not None:
    SH850 = SH850.sel(time=time_mask)

print("Temporal alignment complete.")
print(f"H500 time range: {pd.to_datetime(H500.time.values[0]).date()} -> {pd.to_datetime(H500.time.values[-1]).date()}")
print(f"Common daily samples: {H500.sizes['time']}")

# --- Shape summary
def summarize_da(da, name):
    print(f"{name:<6} | dims={da.dims} | shape={da.shape}")
    if "lat" in da.coords:
        print(f"        lat dims={da['lat'].dims}, shape={da['lat'].shape}")
    if "lon" in da.coords:
        print(f"        lon dims={da['lon'].dims}, shape={da['lon'].shape}")

summarize_da(H500, "H500")
summarize_da(U500, "U500")
summarize_da(V500, "V500")
summarize_da(PRCP, "PRCP")
summarize_da(H850, "H850")
summarize_da(U850, "U850")
summarize_da(V850, "V850")
if SH850 is not None:
    summarize_da(SH850, "SH850")

Temporal alignment complete.
H500 time range: 2012-01-01 -> 2024-12-31
Common daily samples: 4555
H500   | dims=('time', 'y', 'x') | shape=(4555, 90, 140)
        lat dims=('y', 'x'), shape=(90, 140)
        lon dims=('y', 'x'), shape=(90, 140)
U500   | dims=('time', 'y', 'x') | shape=(4555, 90, 140)
        lat dims=('y', 'x'), shape=(90, 140)
        lon dims=('y', 'x'), shape=(90, 140)
V500   | dims=('time', 'y', 'x') | shape=(4555, 90, 140)
        lat dims=('y', 'x'), shape=(90, 140)
        lon dims=('y', 'x'), shape=(90, 140)
PRCP   | dims=('time', 'y', 'x') | shape=(4555, 90, 140)
        lat dims=('y', 'x'), shape=(90, 140)
        lon dims=('y', 'x'), shape=(90, 140)
H850   | dims=('time', 'y', 'x') | shape=(4555, 90, 140)
        lat dims=('y', 'x'), shape=(90, 140)
        lon dims=('y', 'x'), shape=(90, 140)
U850   | dims=('time', 'y', 'x') | shape=(4555, 90, 140)
        lat dims=('y', 'x'), shape=(90, 140)
        lon dims=('y', 'x'), shape=(90, 140)
V850   | dims=('time

### 4.6 Derive anomalies and analysis-ready fields

In [23]:
def compute_daily_anomaly(da):
    """
    Compute anomalies relative to the climatological daily cycle
    (grouped by day of year over the full available period).

    Uses a simple day-of-year climatology. Leap day is kept if present.
    """
    clim = da.groupby("time.dayofyear").mean("time")
    anom = da.groupby("time.dayofyear") - clim
    return anom, clim

In [24]:
# --- 500 hPa anomalies (core clustering / synoptic field)
H500_ANOM, H500_CLIM_DOY = compute_daily_anomaly(H500)
U500_ANOM, U500_CLIM_DOY = compute_daily_anomaly(U500)
V500_ANOM, V500_CLIM_DOY = compute_daily_anomaly(V500)

# --- 850 hPa anomalies (mechanism check support)
H850_ANOM, H850_CLIM_DOY = compute_daily_anomaly(H850)
U850_ANOM, U850_CLIM_DOY = compute_daily_anomaly(U850)
V850_ANOM, V850_CLIM_DOY = compute_daily_anomaly(V850)

# --- Precip anomalies
PRCP_ANOM, PRCP_CLIM_DOY = compute_daily_anomaly(PRCP)

# --- Moisture anomalies (if available)
if SH850 is not None:
    SH850_ANOM, SH850_CLIM_DOY = compute_daily_anomaly(SH850)
else:
    SH850_ANOM, SH850_CLIM_DOY = None, None

# --- Pack analysis-ready fields in dictionaries
FIELDS_RAW = {
    "H500": H500,
    "U500": U500,
    "V500": V500,
    "PRCP": PRCP,
    "H850": H850,
    "U850": U850,
    "V850": V850,
}
if SH850 is not None:
    FIELDS_RAW["SH850"] = SH850

FIELDS_ANOM = {
    "H500": H500_ANOM,
    "U500": U500_ANOM,
    "V500": V500_ANOM,
    "PRCP": PRCP_ANOM,
    "H850": H850_ANOM,
    "U850": U850_ANOM,
    "V850": V850_ANOM,
}
if SH850_ANOM is not None:
    FIELDS_ANOM["SH850"] = SH850_ANOM

FIELDS_CLIM_DOY = {
    "H500": H500_CLIM_DOY,
    "U500": U500_CLIM_DOY,
    "V500": V500_CLIM_DOY,
    "PRCP": PRCP_CLIM_DOY,
    "H850": H850_CLIM_DOY,
    "U850": U850_CLIM_DOY,
    "V850": V850_CLIM_DOY,
}
if SH850_CLIM_DOY is not None:
    FIELDS_CLIM_DOY["SH850"] = SH850_CLIM_DOY

print("Analysis-ready anomaly fields prepared.\n")

for k, da in FIELDS_ANOM.items():
    print(f"{k:<6} anomaly -> dims={da.dims}, shape={da.shape}")

Analysis-ready anomaly fields prepared.

H500   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)
U500   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)
V500   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)
PRCP   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)
H850   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)
U850   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)
V850   anomaly -> dims=('time', 'y', 'x'), shape=(4555, 90, 140)


### 4.7 Bullet-proof check

In [25]:
for name, da in FIELDS_RAW.items():
    valid_frac = float(np.isfinite(da.values).mean())
    print(f"{name:<5} valid fraction = {valid_frac:.3f}")

H500  valid fraction = 0.786
U500  valid fraction = 0.786
V500  valid fraction = 0.786
PRCP  valid fraction = 0.786
H850  valid fraction = 0.786
U850  valid fraction = 0.786
V850  valid fraction = 0.786


In [26]:
for name, da in FIELDS_RAW.items():
    print(f"\n{name}")
    print(f"  min = {float(np.nanmin(da.values)):.3f}")
    print(f"  max = {float(np.nanmax(da.values)):.3f}")


H500
  min = 5367.405
  max = 6007.631

U500
  min = -30.429
  max = 54.218

V500
  min = -44.264
  max = 45.515

PRCP
  min = 0.000
  max = 331.511

H850
  min = 1315.088
  max = 1648.633

U850
  min = -24.855
  max = 23.955

V850
  min = -36.861
  max = 27.608


## 5. Stage 3 recap and focus-window selection

### 5.1 Recap of the Stage 3 monthly contrast story

In [27]:
# --- Month lookup
month_lookup = (
    df_who_month[["month", "month_name"]]
    .drop_duplicates()
    .sort_values("month")
    .reset_index(drop=True)
)

# --- Pivot Stage 3 significance-area summary to wide format
sig_pivot = (
    df_sigfrac_month
    .pivot(index=["month", "month_name"], columns="pollutant", values="frac_area_p_lt_0p05")
    .reset_index()
)

# Flatten column names if needed
sig_pivot.columns.name = None
sig_pivot = sig_pivot.rename(columns={
    "PM10": "sigfrac_PM10",
    "PM2.5": "sigfrac_PM25"
})

# --- Keep only columns we expect
expected_sig_cols = ["month", "month_name", "sigfrac_PM10", "sigfrac_PM25"]
for col in expected_sig_cols:
    if col not in sig_pivot.columns:
        sig_pivot[col] = np.nan

sig_pivot = sig_pivot[expected_sig_cols].sort_values("month").reset_index(drop=True)

# --- Merge with WHO monthly burden/exceedance table
df_stage3_monthly_story = sig_pivot.merge(
    df_who_month,
    on=["month", "month_name"],
    how="left"
)

# --- Add precip climatology
df_stage3_monthly_story = df_stage3_monthly_story.merge(
    df_precip_clim,
    on=["month", "month_name"],
    how="left"
)

# --- Add simple seasonal tag
df_stage3_monthly_story["season"] = df_stage3_monthly_story["month"].map(MONTH_TO_SEASON)

# --- Reorder columns for readability
ordered_cols = [
    "month", "month_name", "season",
    "sigfrac_PM10", "sigfrac_PM25",
    "PM10_exceed_pct", "PM10_excess_mag_mean", "PM10_excess_load_sum",
    "PM25_exceed_pct", "PM25_excess_mag_mean", "PM25_excess_load_sum",
    "precip_mmday"
]
df_stage3_monthly_story = df_stage3_monthly_story[ordered_cols]

print("Stage 3 monthly recap table prepared:")
print(f"  shape: {df_stage3_monthly_story.shape}")
display(df_stage3_monthly_story.round(3))

Stage 3 monthly recap table prepared:
  shape: (12, 12)


,month,month_name,season,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,precip_mmday
0,1,January,DJF,0.42,0.39,74.26,19.80,"5,444.50",87.67,14.87,"4,863.52",0.31
1,2,February,DJF,0.64,0.49,72.81,17.19,"4,418.24",88.67,11.71,"3,666.80",0.31
2,3,March,MAM,0.38,0.50,72.82,13.80,"3,987.90",91.52,10.11,"3,700.20",0.69
3,4,April,MAM,0.29,0.35,69.03,15.01,"3,902.51",96.59,12.70,"4,634.77",0.80
4,5,May,MAM,0.34,0.35,57.80,15.55,"3,297.54",97.04,14.28,"5,155.66",1.91
5,6,June,JJA,0.43,0.39,22.22,7.16,551.32,73.06,7.89,"2,058.33",4.38
6,7,July,JJA,0.02,0.30,14.82,5.23,308.32,78.14,6.18,"1,883.80",4.63
7,8,August,JJA,0.18,0.20,7.32,6.71,181.17,63.38,5.05,"1,221.80",4.97
8,9,September,SON,0.11,0.33,8.11,6.83,204.83,64.05,6.47,"1,495.43",5.16
9,10,October,SON,0.10,0.32,22.55,9.22,783.33,62.33,7.73,"1,809.60",2.45


### 5.2 Selection of high-priority months and seasonal windows

In [28]:
# --- Keep only Stage 4 focus months
df_focus_months = (
    df_stage3_monthly_story[df_stage3_monthly_story["month"].isin(FOCUS_MONTHS)]
    .copy()
    .sort_values("month")
    .reset_index(drop=True)
)

# --- Add short rationale labels
focus_rationale_map = {
    2: "Strongest and cleanest synoptic signal; late-winter benchmark case.",
    5: "Transition month with physical interest and notable PM2.5 burden.",
    7: "Wet-season contrast; weaker or altered synoptic control, especially for PM10."
}
df_focus_months["focus_rationale"] = df_focus_months["month"].map(focus_rationale_map)

# --- Episode counts from Stage 2 episode tables
pm10_focus_counts = (
    df_ep_pm10[df_ep_pm10["start_month"].isin(FOCUS_MONTHS)]
    .groupby("start_month")
    .size()
    .rename("PM10_episode_count")
)

pm25_focus_counts = (
    df_ep_pm25[df_ep_pm25["start_month"].isin(FOCUS_MONTHS)]
    .groupby("start_month")
    .size()
    .rename("PM25_episode_count")
)

df_focus_months = df_focus_months.merge(
    pm10_focus_counts, left_on="month", right_index=True, how="left"
).merge(
    pm25_focus_counts, left_on="month", right_index=True, how="left"
)

# Fill possible missing counts with 0
for col in ["PM10_episode_count", "PM25_episode_count"]:
    df_focus_months[col] = df_focus_months[col].fillna(0).astype(int)

# --- Add a simple Stage 4 role label
stage4_role_map = {
    2: "Primary high-signal case",
    5: "Primary transition case",
    7: "Wet-season contrast case",
}
df_focus_months["stage4_role"] = df_focus_months["month"].map(stage4_role_map)

# --- Keep a clean view for reporting
focus_cols = [
    "month", "month_name", "season", "stage4_role",
    "sigfrac_PM10", "sigfrac_PM25",
    "PM10_exceed_pct", "PM10_excess_load_sum",
    "PM25_exceed_pct", "PM25_excess_load_sum",
    "precip_mmday",
    "PM10_episode_count", "PM25_episode_count",
    "focus_rationale"
]
df_focus_months = df_focus_months[focus_cols]

print("Stage 4 focus months formally selected:")
display(df_focus_months.round(3))

# --- Exportable summary table for later use
stage4_focus_summary = df_focus_months.copy()

Stage 4 focus months formally selected:


,month,month_name,season,stage4_role,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_load_sum,precip_mmday,PM10_episode_count,PM25_episode_count,focus_rationale
0,2,February,DJF,Primary high-signal case,0.64,0.49,72.81,"4,418.24",88.67,"3,666.80",0.31,19,20,Strongest and cleanest synoptic signal; late-w...
1,5,May,MAM,Primary transition case,0.34,0.35,57.80,"3,297.54",97.04,"5,155.66",1.91,15,18,Transition month with physical interest and no...
2,7,July,JJA,Wet-season contrast case,0.02,0.30,14.82,308.32,78.14,"1,883.80",4.63,29,29,Wet-season contrast; weaker or altered synopti...


### 5.3 Expected physical narratives to be tested in Stage 4

In [29]:
# --- Stage 4 hypothesis / narrative table
df_stage4_hypotheses = pd.DataFrame([
    {
        "focus_window": "February",
        "stage4_role": "Primary high-signal case",
        "working_hypothesis": (
            "Extreme PM episodes in February are expected to occur under the clearest "
            "synoptic-control regime, with stronger polluted-versus-clean contrasts, "
            "greater spatial significance, and circulation patterns consistent with "
            "stagnation or reduced ventilation."
        ),
        "main_stage4_test": (
            "Test whether onset-day clustering and lag composites isolate a robust "
            "late-winter polluted prototype and its preconditioning structure."
        )
    },
    {
        "focus_window": "May",
        "stage4_role": "Primary transition case",
        "working_hypothesis": (
            "May is expected to behave as a transitional month in which extreme PM "
            "episodes remain synoptically sensitive but reflect a different physical "
            "story than late winter, likely involving seasonal shifts in ventilation, "
            "lower-tropospheric structure, precipitation background, or source expression."
        ),
        "main_stage4_test": (
            "Test whether May events organize into distinct prototypes and whether "
            "their lagged evolution differs from the February benchmark."
        )
    },
    {
        "focus_window": "July",
        "stage4_role": "Wet-season contrast case",
        "working_hypothesis": (
            "July is expected to represent a weaker or altered synoptic-control regime, "
            "especially for PM10, as wet-season conditions increase precipitation influence "
            "and reduce the robustness of the upper-level polluted-versus-clean signal."
        ),
        "main_stage4_test": (
            "Test whether July behaves mainly as a contrast case showing reduced signal, "
            "different event structure, or stronger precipitation-related modulation."
        )
    }
])

print("Stage 4 physical narratives defined:")
display(df_stage4_hypotheses)

# --- text block reporting the Stage 4 narrative summary
STAGE4_NARRATIVE_SUMMARY = """
Stage 4 will test three linked physical stories:
1) February as the strongest late-winter synoptic-control benchmark;
2) May as a transitional month with altered but still meaningful PM-weather coupling;
3) July as a wet-season contrast where the signal weakens or shifts, especially for PM10.

These windows are selected based on Stage 3 significance-area patterns, WHO-related burden metrics,
episode counts from Stage 2, and the broader physical narrative established by the monthly contrasts.
""".strip()

print("\nStage 4 narrative summary:")
print(STAGE4_NARRATIVE_SUMMARY)

Stage 4 physical narratives defined:


,focus_window,stage4_role,working_hypothesis,main_stage4_test
0,February,Primary high-signal case,Extreme PM episodes in February are expected t...,Test whether onset-day clustering and lag comp...
1,May,Primary transition case,May is expected to behave as a transitional mo...,Test whether May events organize into distinct...
2,July,Wet-season contrast case,July is expected to represent a weaker or alte...,Test whether July behaves mainly as a contrast...



Stage 4 narrative summary:
Stage 4 will test three linked physical stories:
1) February as the strongest late-winter synoptic-control benchmark;
2) May as a transitional month with altered but still meaningful PM-weather coupling;
3) July as a wet-season contrast where the signal weakens or shifts, especially for PM10.

These windows are selected based on Stage 3 significance-area patterns, WHO-related burden metrics,
episode counts from Stage 2, and the broader physical narrative established by the monthly contrasts.


### 5.4 Exports for later use

In [31]:
if EXPORT_TABLES:
    out1 = OUT_TAB / "stage4_stage3_monthly_recap_table.csv"
    out2 = OUT_TAB / "stage4_focus_month_selection_table.csv"
    out3 = OUT_TAB / "stage4_working_hypotheses_table.csv"

    df_stage3_monthly_story.to_csv(out1, index=False)
    stage4_focus_summary.to_csv(out2, index=False)
    df_stage4_hypotheses.to_csv(out3, index=False)

    print(f"saved: {out1}")
    print(f"saved: {out2}")
    print(f"saved: {out3}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_stage3_monthly_recap_table.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_focus_month_selection_table.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_working_hypotheses_table.csv


## 6. Seasonal synthesis of the Stage 3 monthly story

### 6.1 From monthly diagnostics to seasonal narratives

In [32]:
# --- Seasonal aggregation from the Stage 3 monthly story
season_order = ["DJF", "MAM", "JJA", "SON"]

# Episode counts by season from Stage 2 onset tables
pm10_season_counts = (
    df_ep_pm10.groupby("start_season")
    .size()
    .rename("PM10_episode_count")
)

pm25_season_counts = (
    df_ep_pm25.groupby("start_season")
    .size()
    .rename("PM25_episode_count")
)

df_seasonal_story = (
    df_stage3_monthly_story
    .groupby("season", as_index=False)
    .agg({
        "sigfrac_PM10": "mean",
        "sigfrac_PM25": "mean",
        "PM10_exceed_pct": "mean",
        "PM10_excess_load_sum": "sum",
        "PM25_exceed_pct": "mean",
        "PM25_excess_load_sum": "sum",
        "precip_mmday": "mean"
    })
)

df_seasonal_story = df_seasonal_story.merge(
    pm10_season_counts, left_on="season", right_index=True, how="left"
).merge(
    pm25_season_counts, left_on="season", right_index=True, how="left"
)

df_seasonal_story["season"] = pd.Categorical(
    df_seasonal_story["season"],
    categories=season_order,
    ordered=True
)
df_seasonal_story = df_seasonal_story.sort_values("season").reset_index(drop=True)

display(df_seasonal_story.round(3))

# --- Export table
out_csv = OUT_TAB / "stage4_seasonal_story_summary.csv"
df_seasonal_story.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,season,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_load_sum,precip_mmday,PM10_episode_count,PM25_episode_count
0,DJF,0.46,0.43,73.86,"15,730.22",89.53,"14,169.03",0.34,70,74
1,MAM,0.34,0.40,66.55,"11,187.95",95.05,"13,490.63",1.13,53,63
2,JJA,0.21,0.30,14.79,"1,040.81",71.53,"5,163.93",4.66,68,76
3,SON,0.17,0.36,28.13,"3,744.00",71.01,"6,880.49",2.85,59,73


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_seasonal_story_summary.csv


In [69]:

# =========================================================
# figure: monthly-to-seasonal synthesis overview
# =========================================================

plot_df = df_stage3_monthly_story.copy()
plot_df["month_abbr"] = plot_df["month"].map(lambda m: calendar.month_abbr[m])

x = np.arange(len(plot_df))
focus_mask = plot_df["month"].isin(FOCUS_MONTHS)
focus_idx = [i for i, is_f in enumerate(focus_mask) if is_f]

fig, axes = plt.subplots(3, 1, figsize=(13, 10), dpi=250, sharex=True)
fig.subplots_adjust(top=0.89, bottom=0.17, left=0.10, right=0.90, hspace=0.56)
fig.suptitle("Stage 4 seasonal synthesis from Stage 3 monthly diagnostics", fontsize=15, fontweight="bold", y=0.975)

# Move subplot 1 slightly upward (subplot 2 stays fixed)
_p0 = axes[0].get_position()
axes[0].set_position([_p0.x0, _p0.y0 + 0.02, _p0.width, _p0.height])

# ---------------------------------------------------------
# panel 1: significance-area monthly story
# ---------------------------------------------------------
w = 0.36
axes[0].bar(
    x - w/2, plot_df["sigfrac_PM10"], width=w,
    label="PM10", edgecolor="black", linewidth=0.6
)
axes[0].bar(
    x + w/2, plot_df["sigfrac_PM25"], width=w,
    label="PM2.5", edgecolor="black", linewidth=0.6
)

for i, is_focus in enumerate(focus_mask):
    if is_focus:
        axes[0].axvspan(i - 0.5, i + 0.5, alpha=0.10)

# Value labels for focus months - panel 1
for i in focus_idx:
    v10 = plot_df["sigfrac_PM10"].iloc[i]
    v25 = plot_df["sigfrac_PM25"].iloc[i]
    axes[0].text(i - w/2, v10 + 0.012, f"{v10:.2f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    axes[0].text(i + w/2, v25 + 0.012, f"{v25:.2f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

axes[0].set_ylabel("Fraction of area\n(p < 0.05)")
axes[0].set_title(
    "Monthly robustness of polluted-versus-clean synoptic contrasts",
    fontweight="bold"
)
axes[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.20), frameon=True)
axes[0].grid(axis="y", linestyle="--", alpha=0.35)

# ---------------------------------------------------------
# panel 2: WHO exceedance frequency
# ---------------------------------------------------------
axes[1].plot(
    x, plot_df["PM10_exceed_pct"],
    marker="o", linewidth=2, label="PM10 exceedance frequency (%)"
)
axes[1].plot(
    x, plot_df["PM25_exceed_pct"],
    marker="s", linewidth=2, label="PM2.5 exceedance frequency (%)"
)

for i, is_focus in enumerate(focus_mask):
    if is_focus:
        axes[1].axvspan(i - 0.5, i + 0.5, alpha=0.10)

# Value labels for focus months - panel 2
for i in focus_idx:
    v10 = plot_df["PM10_exceed_pct"].iloc[i]
    v25 = plot_df["PM25_exceed_pct"].iloc[i]
    axes[1].text(i, v10 + 1.5, f"{v10:.0f}%", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    axes[1].text(i, v25 + 1.5, f"{v25:.0f}%", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

axes[1].set_ylabel("WHO exceedance\nfrequency (%)")
axes[1].set_title(
    "Monthly burden context: WHO exceedance frequency",
    fontweight="bold"
)
axes[1].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.20), frameon=True)
axes[1].grid(axis="y", linestyle="--", alpha=0.35)

# ---------------------------------------------------------
# panel 3: PM burden + precipitation background
# ---------------------------------------------------------
ax3 = axes[2]
ax3b = ax3.twinx()

# Move subplot 3 (and its twin) downward AFTER twinx() is created,
# so both axes move together
_p2 = ax3.get_position()
_new_p2 = [_p2.x0, _p2.y0 - 0.03, _p2.width, _p2.height]
ax3.set_position(_new_p2)
ax3b.set_position(_new_p2)

ax3.plot(
    x, plot_df["PM10_excess_load_sum"],
    marker="o", linewidth=2, label="PM10 excess load sum"
)
ax3.plot(
    x, plot_df["PM25_excess_load_sum"],
    marker="s", linewidth=2, label="PM2.5 excess load sum"
)

ax3b.bar(
    x, plot_df["precip_mmday"],
    width=0.55, alpha=0.25, color="#25DBB6", edgecolor="black", linewidth=0.5,
    label="Precipitation climatology (mm/day)"
)

for i, is_focus in enumerate(focus_mask):
    if is_focus:
        ax3.axvspan(i - 0.5, i + 0.5, alpha=0.10)

# Value labels for focus months - panel 3
for i in focus_idx:
    v10 = plot_df["PM10_excess_load_sum"].iloc[i]
    v25 = plot_df["PM25_excess_load_sum"].iloc[i]
    ax3.text(i - 0.2, v10 + 100, f"{v10:.0f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    ax3.text(i + 0.2, v25 + 100, f"{v25:.0f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

ax3.set_ylabel("Monthly excess load")
ax3b.set_ylabel("Precipitation\n(mm/day)", labelpad=14)

# Right spine on ax3b
ax3b.spines["right"].set_visible(True)
ax3b.spines["right"].set_linewidth(1)

ax3.set_title(
    "Burden and hydrometeorological background",
    fontweight="bold"
)
ax3.grid(axis="y", linestyle="--", alpha=0.35)

# X-axis month labels on all subplots
for ax in axes:
    ax.tick_params(labelbottom=True)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["month_abbr"])

# Figure-level note
fig.text(
    0.01, 0.01,
    "Shaded months indicate the Stage 4 focus windows: February, May, and July.",
    fontsize=9
)

# Build combined legend for panel 3
handles1, labels1 = ax3.get_legend_handles_labels()
handles2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(handles1 + handles2, labels1 + labels2, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.26), frameon=True)

out_fig = OUT_FIG / "stage4_seasonal_synthesis_monthly_story.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_seasonal_synthesis_monthly_story.png


### 6.2 Dry-season signal: late winter and early spring

In [45]:
# Focus on January–March to represent late winter / early spring transition
dry_months = [1, 2, 3]

df_dry_signal = (
    df_stage3_monthly_story[df_stage3_monthly_story["month"].isin(dry_months)]
    .copy()
    .sort_values("month")
    .reset_index(drop=True)
)

# Add ranks within this subset for quick interpretation
df_dry_signal["rank_sig_PM10"] = df_dry_signal["sigfrac_PM10"].rank(ascending=False, method="min").astype(int)
df_dry_signal["rank_sig_PM25"] = df_dry_signal["sigfrac_PM25"].rank(ascending=False, method="min").astype(int)
df_dry_signal["rank_load_PM10"] = df_dry_signal["PM10_excess_load_sum"].rank(ascending=False, method="min").astype(int)
df_dry_signal["rank_load_PM25"] = df_dry_signal["PM25_excess_load_sum"].rank(ascending=False, method="min").astype(int)

display(df_dry_signal.round(3))

# --- Export table
out_csv = OUT_TAB / "stage4_dryseason_latewinter_earlyspring_summary.csv"
df_dry_signal.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,month,month_name,season,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,precip_mmday,rank_sig_PM10,rank_sig_PM25,rank_load_PM10,rank_load_PM25
0,1,January,DJF,0.42,0.39,74.26,19.80,"5,444.50",87.67,14.87,"4,863.52",0.31,2,3,1,1
1,2,February,DJF,0.64,0.49,72.81,17.19,"4,418.24",88.67,11.71,"3,666.80",0.31,1,2,2,3
2,3,March,MAM,0.38,0.50,72.82,13.80,"3,987.90",91.52,10.11,"3,700.20",0.69,3,1,3,2


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_dryseason_latewinter_earlyspring_summary.csv


In [61]:
# =========================================================
# Figure: dry-season signal (Jan-Feb-Mar)
# =========================================================

plot_df = df_dry_signal.copy()
plot_df["month_abbr"] = plot_df["month"].map(lambda m: calendar.month_abbr[m])
x = np.arange(len(plot_df))

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), dpi=250, sharex=True)
fig.subplots_adjust(top=0.84, bottom=0.23, left=0.10, right=0.90, hspace=0.62)
fig.suptitle("Dry-season signal: late winter and early spring", fontsize=16, fontweight="bold", y=0.965)

# Slightly smaller subplot typography so the global title clearly dominates
subplot_title_fs = 11
axis_label_fs = 10
tick_fs = 9
legend_fs = 8.8

# ---------------------------------------------------------
# Panel 1: significance-area contrast
# ---------------------------------------------------------
w = 0.36
axes[0].bar(
    x - w/2, plot_df["sigfrac_PM10"],
    width=w, edgecolor="black", linewidth=0.6, label="PM10"
)
axes[0].bar(
    x + w/2, plot_df["sigfrac_PM25"],
    width=w, edgecolor="black", linewidth=0.6, label="PM2.5"
)

# Highlight February
feb_idx = plot_df.index[plot_df["month"] == 2]
if len(feb_idx) > 0:
    i = feb_idx[0]
    axes[0].axvspan(i - 0.5, i + 0.5, alpha=0.12)

axes[0].set_ylabel("Fraction of area\n(p < 0.05)", fontsize=axis_label_fs)
axes[0].set_title(
    "February emerges as the clearest high-signal benchmark month",
    fontweight="bold", fontsize=subplot_title_fs
)
axes[0].legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.36), fontsize=legend_fs, frameon=True)
axes[0].grid(axis="y", linestyle="--", alpha=0.35)
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df["month_abbr"])
axes[0].tick_params(axis="x", labelbottom=True, labelsize=tick_fs)
axes[0].tick_params(axis="y", labelsize=tick_fs)

# ---------------------------------------------------------
# Panel 2: exceedance burden and dry background
# ---------------------------------------------------------
ax2 = axes[1]
ax2b = ax2.twinx()

ax2.plot(
    x, plot_df["PM10_exceed_pct"],
    marker="o", linewidth=2, label="PM10 exceedance frequency (%)"
)
ax2.plot(
    x, plot_df["PM25_exceed_pct"],
    marker="s", linewidth=2, label="PM2.5 exceedance frequency (%)"
)
ax2b.plot(
    x, plot_df["precip_mmday"],
    marker="^", linewidth=2, linestyle="--", color="#25DBB6", label="Precipitation climatology (mm/day)"
)

if len(feb_idx) > 0:
    i = feb_idx[0]
    ax2.axvspan(i - 0.5, i + 0.5, alpha=0.12)
    ax2b.axvline(i, color="0.35", linestyle=":", linewidth=1.7, alpha=0.95, zorder=6)

ax2.set_ylabel("WHO exceedance\nfrequency (%)", fontsize=axis_label_fs)
ax2b.set_ylabel("Precipitation\n(mm/day)", fontsize=axis_label_fs)
ax2.set_title(
    "Dry background conditions remain strong through February",
    fontweight="bold", fontsize=subplot_title_fs
)
ax2.grid(axis="y", linestyle="--", alpha=0.35)

axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df["month_abbr"])
axes[1].tick_params(axis="x", labelsize=tick_fs)
axes[1].tick_params(axis="y", labelsize=tick_fs)
ax2b.tick_params(axis="y", labelsize=tick_fs)

# Right spine on ax2b
ax2b.spines["right"].set_visible(True)
ax2b.spines["right"].set_linewidth(1)

handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(handles1 + handles2, labels1 + labels2, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.38), fontsize=legend_fs, frameon=True)

out_fig = OUT_FIG / "stage4_dryseason_signal_jfm.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_dryseason_signal_jfm.png


### 6.3 Transitional signal: the role of May

In [47]:
# Use April-May-June to frame May as a transition month
transition_months = [4, 5, 6]

df_transition = (
    df_stage3_monthly_story[df_stage3_monthly_story["month"].isin(transition_months)]
    .copy()
    .sort_values("month")
    .reset_index(drop=True)
)

# Add simple month-to-month deltas centered on May
may_row = df_transition[df_transition["month"] == 5].iloc[0]
apr_row = df_transition[df_transition["month"] == 4].iloc[0]
jun_row = df_transition[df_transition["month"] == 6].iloc[0]

df_may_transition_summary = pd.DataFrame([
    {
        "focus_month": "May",
        "PM10_sigfrac": may_row["sigfrac_PM10"],
        "PM25_sigfrac": may_row["sigfrac_PM25"],
        "PM10_exceed_pct": may_row["PM10_exceed_pct"],
        "PM25_exceed_pct": may_row["PM25_exceed_pct"],
        "PM10_excess_load_sum": may_row["PM10_excess_load_sum"],
        "PM25_excess_load_sum": may_row["PM25_excess_load_sum"],
        "precip_mmday": may_row["precip_mmday"],
        "delta_sigfrac_PM10_vs_Apr": may_row["sigfrac_PM10"] - apr_row["sigfrac_PM10"],
        "delta_sigfrac_PM25_vs_Apr": may_row["sigfrac_PM25"] - apr_row["sigfrac_PM25"],
        "delta_precip_vs_Apr": may_row["precip_mmday"] - apr_row["precip_mmday"],
        "delta_precip_vs_Jun": may_row["precip_mmday"] - jun_row["precip_mmday"],
        "interpretive_note": (
            "May retains meaningful synoptic sensitivity while showing a wetter background "
            "than late winter and strong PM2.5 burden, supporting its role as a transition month."
        )
    }
])

display(df_transition.round(3))
display(df_may_transition_summary.round(3))

# --- Export tables
out_csv1 = OUT_TAB / "stage4_transition_apr_may_jun_summary.csv"
out_csv2 = OUT_TAB / "stage4_may_transition_focus_table.csv"
df_transition.to_csv(out_csv1, index=False)
df_may_transition_summary.to_csv(out_csv2, index=False)
print(f"saved: {out_csv1}")
print(f"saved: {out_csv2}")

,month,month_name,season,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,precip_mmday
0,4,April,MAM,0.29,0.35,69.03,15.01,"3,902.51",96.59,12.70,"4,634.77",0.80
1,5,May,MAM,0.34,0.35,57.80,15.55,"3,297.54",97.04,14.28,"5,155.66",1.91
2,6,June,JJA,0.43,0.39,22.22,7.16,551.32,73.06,7.89,"2,058.33",4.38


,focus_month,PM10_sigfrac,PM25_sigfrac,PM10_exceed_pct,PM25_exceed_pct,PM10_excess_load_sum,PM25_excess_load_sum,precip_mmday,delta_sigfrac_PM10_vs_Apr,delta_sigfrac_PM25_vs_Apr,delta_precip_vs_Apr,delta_precip_vs_Jun,interpretive_note
0,May,0.34,0.35,57.80,97.04,"3,297.54","5,155.66",1.91,0.05,0.00,1.11,-2.47,May retains meaningful synoptic sensitivity wh...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_transition_apr_may_jun_summary.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_may_transition_focus_table.csv


In [66]:
# =========================================================
# Figure: transitional role of May
# =========================================================

plot_df = df_transition.copy()
plot_df["month_abbr"] = plot_df["month"].map(lambda m: calendar.month_abbr[m])
x = np.arange(len(plot_df))

fig, axes = plt.subplots(2, 1, figsize=(11, 7.8), dpi=250, sharex=True)
fig.subplots_adjust(top=0.84, bottom=0.23, left=0.10, right=0.90, hspace=0.62)
fig.suptitle("Transition-season signal: the role of May", fontsize=17, fontweight="bold", y=0.965)

# Smaller subplot typography so the global title clearly dominates
subplot_title_fs = 11
axis_label_fs = 10
tick_fs = 9
legend_fs = 8.8

# Panel 1
w = 0.36
axes[0].bar(
    x - w/2, plot_df["sigfrac_PM10"], width=w,
    edgecolor="black", linewidth=0.6, label="PM10"
)
axes[0].bar(
    x + w/2, plot_df["sigfrac_PM25"], width=w,
    edgecolor="black", linewidth=0.6, label="PM2.5"
)

# Highlight May
may_idx = plot_df.index[plot_df["month"] == 5]
if len(may_idx) > 0:
    i = may_idx[0]
    axes[0].axvspan(i - 0.5, i + 0.5, alpha=0.12)

axes[0].set_ylabel("Fraction of area\n(p < 0.05)", fontsize=axis_label_fs)
axes[0].set_title(
    "May retains synoptic sensitivity while departing from the late-winter benchmark",
    fontweight="bold", fontsize=subplot_title_fs
)
axes[0].legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.36), fontsize=legend_fs, frameon=True)
axes[0].grid(axis="y", linestyle="--", alpha=0.35)

# Force x labels to appear in subplot 1 even with sharex=True
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df["month_abbr"])
axes[0].tick_params(axis="x", labelbottom=True, labelsize=tick_fs)
axes[0].tick_params(axis="y", labelsize=tick_fs)

# Panel 2
ax2 = axes[1]
ax2b = ax2.twinx()

ax2.plot(
    x, plot_df["PM10_excess_load_sum"],
    marker="o", linewidth=2, label="PM10 excess load sum"
)
ax2.plot(
    x, plot_df["PM25_excess_load_sum"],
    marker="s", linewidth=2, label="PM2.5 excess load sum"
)
ax2b.bar(
    x, plot_df["precip_mmday"],
    width=0.55, alpha=0.25, color="#25DBB6", edgecolor="black", linewidth=0.5,
    label="Precipitation climatology (mm/day)"
)

if len(may_idx) > 0:
    i = may_idx[0]
    ax2.axvspan(i - 0.5, i + 0.5, alpha=0.12)

ax2.set_ylabel("Monthly excess load", fontsize=axis_label_fs)
ax2b.set_ylabel("Precipitation\n(mm/day)", fontsize=axis_label_fs)
ax2.set_title(
    "May combines elevated PM2.5 burden with an intermediate precipitation background",
    fontweight="bold", fontsize=subplot_title_fs
)
ax2.grid(axis="y", linestyle="--", alpha=0.35)

# Right spine on ax2b
ax2b.spines["right"].set_visible(True)
ax2b.spines["right"].set_linewidth(1)

axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df["month_abbr"])
axes[1].tick_params(axis="x", labelsize=tick_fs)
axes[1].tick_params(axis="y", labelsize=tick_fs)
ax2b.tick_params(axis="y", labelsize=tick_fs)

handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(handles1 + handles2, labels1 + labels2, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.38), fontsize=legend_fs, frameon=True)

out_fig = OUT_FIG / "stage4_transition_role_of_may.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_transition_role_of_may.png


### 6.4 Wet-season contrast: July and signal weakening

In [49]:
# Use June-July-August to frame July in the wet season
wet_months = [6, 7, 8]

df_wet_contrast = (
    df_stage3_monthly_story[df_stage3_monthly_story["month"].isin(wet_months)]
    .copy()
    .sort_values("month")
    .reset_index(drop=True)
)

july_row = df_wet_contrast[df_wet_contrast["month"] == 7].iloc[0]

df_july_contrast_summary = pd.DataFrame([
    {
        "focus_month": "July",
        "PM10_sigfrac": july_row["sigfrac_PM10"],
        "PM25_sigfrac": july_row["sigfrac_PM25"],
        "PM10_exceed_pct": july_row["PM10_exceed_pct"],
        "PM25_exceed_pct": july_row["PM25_exceed_pct"],
        "PM10_excess_load_sum": july_row["PM10_excess_load_sum"],
        "PM25_excess_load_sum": july_row["PM25_excess_load_sum"],
        "precip_mmday": july_row["precip_mmday"],
        "interpretive_note": (
            "July acts as a controlled wet-season contrast: PM10 synoptic signal nearly collapses, "
            "while PM2.5 retains some sensitivity under a much wetter background."
        )
    }
])

display(df_wet_contrast.round(3))
display(df_july_contrast_summary.round(3))

# --- Export tables
out_csv1 = OUT_TAB / "stage4_wetseason_jja_summary.csv"
out_csv2 = OUT_TAB / "stage4_july_contrast_focus_table.csv"
df_wet_contrast.to_csv(out_csv1, index=False)
df_july_contrast_summary.to_csv(out_csv2, index=False)
print(f"saved: {out_csv1}")
print(f"saved: {out_csv2}")

,month,month_name,season,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum,precip_mmday
0,6,June,JJA,0.43,0.39,22.22,7.16,551.32,73.06,7.89,"2,058.33",4.38
1,7,July,JJA,0.02,0.30,14.82,5.23,308.32,78.14,6.18,"1,883.80",4.63
2,8,August,JJA,0.18,0.20,7.32,6.71,181.17,63.38,5.05,"1,221.80",4.97


,focus_month,PM10_sigfrac,PM25_sigfrac,PM10_exceed_pct,PM25_exceed_pct,PM10_excess_load_sum,PM25_excess_load_sum,precip_mmday,interpretive_note
0,July,0.02,0.30,14.82,78.14,308.32,"1,883.80",4.63,July acts as a controlled wet-season contrast:...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_wetseason_jja_summary.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_july_contrast_focus_table.csv


In [68]:
# =========================================================
# Figure: July wet-season contrast
# =========================================================

plot_df = df_wet_contrast.copy()
plot_df["month_abbr"] = plot_df["month"].map(lambda m: calendar.month_abbr[m])
x = np.arange(len(plot_df))

fig, axes = plt.subplots(2, 1, figsize=(11, 7.8), dpi=250, sharex=True)
fig.subplots_adjust(top=0.84, bottom=0.23, left=0.10, right=0.90, hspace=0.62)
fig.suptitle("Wet-season contrast: July and signal weakening", fontsize=17, fontweight="bold", y=0.965)

# Smaller subplot typography so the global title clearly dominates
subplot_title_fs = 11
axis_label_fs = 10
tick_fs = 9
legend_fs = 8.8

# Panel 1
w = 0.36
axes[0].bar(
    x - w/2, plot_df["sigfrac_PM10"], width=w,
    edgecolor="black", linewidth=0.6, label="PM10"
)
axes[0].bar(
    x + w/2, plot_df["sigfrac_PM25"], width=w,
    edgecolor="black", linewidth=0.6, label="PM2.5"
)

# Highlight July
jul_idx = plot_df.index[plot_df["month"] == 7]
if len(jul_idx) > 0:
    i = jul_idx[0]
    axes[0].axvspan(i - 0.5, i + 0.5, alpha=0.12)

axes[0].set_ylabel("Fraction of area\n(p < 0.05)", fontsize=axis_label_fs)
axes[0].set_title(
    "PM10 loses robustness more strongly than PM2.5 under wetter conditions",
    fontweight="bold", fontsize=subplot_title_fs
)
axes[0].legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.36), fontsize=legend_fs, frameon=True)
axes[0].grid(axis="y", linestyle="--", alpha=0.35)

# Force x labels to appear in subplot 1 even with sharex=True
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df["month_abbr"])
axes[0].tick_params(axis="x", labelbottom=True, labelsize=tick_fs)
axes[0].tick_params(axis="y", labelsize=tick_fs)

# Panel 2
ax2 = axes[1]
ax2b = ax2.twinx()

ax2.plot(
    x, plot_df["PM10_exceed_pct"],
    marker="o", linewidth=2, label="PM10 exceedance frequency (%)"
)
ax2.plot(
    x, plot_df["PM25_exceed_pct"],
    marker="s", linewidth=2, label="PM2.5 exceedance frequency (%)"
)
ax2b.bar(
    x, plot_df["precip_mmday"],
    width=0.55, alpha=0.25, color="#25DBB6", edgecolor="black", linewidth=0.5,
    label="Precipitation climatology (mm/day)"
)

if len(jul_idx) > 0:
    i = jul_idx[0]
    ax2.axvspan(i - 0.5, i + 0.5, alpha=0.12)

ax2.set_ylabel("WHO exceedance\nfrequency (%)", fontsize=axis_label_fs)
ax2b.set_ylabel("Precipitation\n(mm/day)", fontsize=axis_label_fs)
ax2.set_title(
    "July remains polluted enough to matter, but under a much stronger precipitation regime",
    fontweight="bold", fontsize=subplot_title_fs
)
ax2.grid(axis="y", linestyle="--", alpha=0.35)

# Right spine on ax2b
ax2b.spines["right"].set_visible(True)
ax2b.spines["right"].set_linewidth(1)

axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df["month_abbr"])
axes[1].tick_params(axis="x", labelsize=tick_fs)
axes[1].tick_params(axis="y", labelsize=tick_fs)
ax2b.tick_params(axis="y", labelsize=tick_fs)

handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(handles1 + handles2, labels1 + labels2, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.38), fontsize=legend_fs, frameon=True)

out_fig = OUT_FIG / "stage4_wetseason_july_contrast.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_wetseason_july_contrast.png


### 6.5 Synthesis for Stage 4 analytical focus

In [51]:
df_stage4_analysis_focus = pd.DataFrame([
    {
        "analysis_window": "February",
        "stage4_role": "Primary high-signal case",
        "why_it_stays": (
            "Highest and cleanest synoptic-control benchmark, especially for PM10, "
            "with strong significance-area coverage under a very dry background."
        ),
        "planned_use_in_stage4": (
            "Core month for regime extraction, lag composites, and mechanism interpretation."
        )
    },
    {
        "analysis_window": "May",
        "stage4_role": "Primary transition case",
        "why_it_stays": (
            "Transition month with meaningful synoptic sensitivity and especially strong PM2.5 burden, "
            "suggesting a physically distinct story from late winter."
        ),
        "planned_use_in_stage4": (
            "Core month for transition-season regime comparison and lag evolution."
        )
    },
    {
        "analysis_window": "July",
        "stage4_role": "Wet-season contrast case",
        "why_it_stays": (
            "Useful controlled contrast month: PM10 synoptic signal weakens sharply while PM2.5 retains some sensitivity "
            "under a much wetter seasonal background."
        ),
        "planned_use_in_stage4": (
            "Contrast case for wet-season weakening, altered mechanism, and precipitation-sensitive interpretation."
        )
    }
])

display(df_stage4_analysis_focus)

# --- Export table
out_csv = OUT_TAB / "stage4_final_analysis_focus_table.csv"
df_stage4_analysis_focus.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,analysis_window,stage4_role,why_it_stays,planned_use_in_stage4
0,February,Primary high-signal case,Highest and cleanest synoptic-control benchmar...,"Core month for regime extraction, lag composit..."
1,May,Primary transition case,Transition month with meaningful synoptic sens...,Core month for transition-season regime compar...
2,July,Wet-season contrast case,Useful controlled contrast month: PM10 synopti...,"Contrast case for wet-season weakening, altere..."


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_final_analysis_focus_table.csv


## 7. Light k-means regime extraction for extreme-event days

- What kinds of synoptic prototypes exist?

### 7.1 Objective and Stage 4 regime-design choice

In [73]:
df_regime_design = pd.DataFrame([
    {
        "component": "Stage 4 regime extraction",
        "objective": (
            "Identify a small number of recurrent synoptic prototypes for extreme-event onset days "
            "using a light k-means framework."
        ),
        "sample_definition": (
            "One representative day per p90 episode, defined as day 0 (episode onset)."
        ),
        "clustering_field": "H500_ANOM (daily 500 hPa geopotential height anomaly)",
        "domain": "Mexico-centered Stage 4 analysis domain",
        "candidate_k_values": ", ".join(map(str, CLUSTER_K_VALUES)),
        "design_choice": (
            "Regime extraction is performed separately for PM10 and PM2.5, rather than on a combined sample, "
            "to preserve pollutant-specific synoptic signatures and support later regime-conditioned lag analysis."
        ),
        "what_it_is_not": (
            "This is not intended to be a full climatological weather-pattern classification for all days."
        )
    }
])

display(df_regime_design)

out_csv = OUT_TAB / "stage4_regime_design_table.csv"
df_regime_design.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,component,objective,sample_definition,clustering_field,domain,candidate_k_values,design_choice,what_it_is_not
0,Stage 4 regime extraction,Identify a small number of recurrent synoptic ...,"One representative day per p90 episode, define...",H500_ANOM (daily 500 hPa geopotential height a...,Mexico-centered Stage 4 analysis domain,"2, 3, 4",Regime extraction is performed separately for ...,This is not intended to be a full climatologic...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_regime_design_table.csv


### 7.2 Reusable pipeline for pollutant-specific regime extraction

In [75]:
# --- Shared cluster field
CLUSTER_DA = H500_ANOM.copy().rename("H500_ANOM_CLUSTER")
LON2D = CLUSTER_DA["lon"]
LAT2D = CLUSTER_DA["lat"]
VALID_MASK = np.isfinite(CLUSTER_DA.isel(time=0))

In [76]:
def run_light_kmeans_regime_pipeline(
    pollutant,
    onset_df,
    cluster_da,
    valid_mask,
    k_values=(2, 3, 4),
    default_k=3,
    random_state=42,
    n_init=50,
    max_iter=500,
):
    """
    Run the full light k-means regime-extraction workflow for one pollutant.

    Returns a dictionary with:
    - onset sample
    - preprocessing summary
    - candidate k diagnostics
    - preferred solution
    - regime assignments
    - regime composites
    - regime support tables
    - month counts
    """

    # -----------------------------------------------------
    # 1) Define onset-day sample
    # -----------------------------------------------------
    df_onsets = onset_df[onset_df["start_month"].isin(FOCUS_MONTHS)].copy()
    df_onsets = df_onsets.sort_values("day0").reset_index(drop=True)
    df_onsets["month_name"] = df_onsets["start_month"].map(lambda m: calendar.month_name[m])

    available_times = pd.to_datetime(cluster_da["time"].values).normalize()
    df_onsets = df_onsets[df_onsets["day0"].isin(available_times)].copy()
    df_onsets = df_onsets.sort_values("day0").reset_index(drop=True)

    # -----------------------------------------------------
    # 2) Subset anomaly field on onset days
    # -----------------------------------------------------
    cluster_days = pd.to_datetime(df_onsets["day0"]).values
    da_cluster = cluster_da.sel(time=cluster_days)

    # -----------------------------------------------------
    # 3) Flatten valid grid cells only
    # -----------------------------------------------------
    X_raw = da_cluster.values[:, valid_mask.values]

    scaler = StandardScaler(with_mean=True, with_std=True)
    X_std = scaler.fit_transform(X_raw)

    df_preproc = pd.DataFrame([
        {
            "pollutant": pollutant,
            "n_samples": X_std.shape[0],
            "n_features_valid_gridcells": X_std.shape[1],
            "standardized": True,
            "sample_definition": "Unique episode-onset days (day 0) in February, May, and July",
            "clustering_field": "H500_ANOM"
        }
    ])

    # -----------------------------------------------------
    # 4) Test candidate k values
    # -----------------------------------------------------
    k_rows = []
    k_models = {}

    for k in k_values:
        km = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=n_init,
            max_iter=max_iter
        )

        labels = km.fit_predict(X_std)
        unique_labels, counts = np.unique(labels, return_counts=True)
        counts_sorted = np.sort(counts)[::-1]

        try:
            sil = silhouette_score(X_std, labels)
        except Exception:
            sil = np.nan

        k_rows.append({
            "pollutant": pollutant,
            "k": k,
            "inertia": km.inertia_,
            "silhouette_score": sil,
            "min_cluster_size": int(counts.min()),
            "max_cluster_size": int(counts.max()),
            "cluster_sizes_desc": ", ".join(map(str, counts_sorted.tolist()))
        })

        k_models[k] = {
            "model": km,
            "labels": labels,
            "counts": counts
        }

    df_k_test = pd.DataFrame(k_rows).sort_values("k").reset_index(drop=True)

    # -----------------------------------------------------
    # 5) Select preferred k
    # -----------------------------------------------------
    preferred_k = default_k if default_k in k_values else list(k_values)[0]
    labels_final = k_models[preferred_k]["labels"]
    model_final = k_models[preferred_k]["model"]

    df_assign = df_onsets.copy()
    df_assign["regime_id"] = labels_final + 1
    df_assign["regime_label"] = df_assign["regime_id"].map(lambda r: f"R{r}")

    # -----------------------------------------------------
    # 6) Regime composites
    # -----------------------------------------------------
    H500_reg = H500_ANOM.sel(time=cluster_days).assign_coords(regime=("time", df_assign["regime_id"].values))
    U500_reg = U500_ANOM.sel(time=cluster_days).assign_coords(regime=("time", df_assign["regime_id"].values))
    V500_reg = V500_ANOM.sel(time=cluster_days).assign_coords(regime=("time", df_assign["regime_id"].values))

    H500_comp = H500_reg.groupby("regime").mean("time")
    U500_comp = U500_reg.groupby("regime").mean("time")
    V500_comp = V500_reg.groupby("regime").mean("time")

    # -----------------------------------------------------
    # 7) Regime size summary
    # -----------------------------------------------------
    df_sizes = (
        df_assign.groupby(["regime_id", "regime_label"], as_index=False)
        .size()
        .rename(columns={"size": "n_onset_days"})
        .sort_values("regime_id")
        .reset_index(drop=True)
    )
    df_sizes["fraction_of_sample"] = df_sizes["n_onset_days"] / len(df_assign)
    df_sizes["pollutant"] = pollutant

    # -----------------------------------------------------
    # 8) Regime interpretation support
    # -----------------------------------------------------
    support_rows = []
    for rid in sorted(df_assign["regime_id"].unique()):
        sub = df_assign[df_assign["regime_id"] == rid].copy()

        dominant_month = sub["month_name"].value_counts().idxmax()
        persistence_mix = sub["persistence_class"].value_counts().idxmax()

        hmean = float(np.nanmean(H500_comp.sel(regime=rid).values))
        umean = float(np.nanmean(U500_comp.sel(regime=rid).values))
        vmean = float(np.nanmean(V500_comp.sel(regime=rid).values))

        support_rows.append({
            "pollutant": pollutant,
            "regime_id": int(rid),
            "regime_label": f"R{int(rid)}",
            "n_onset_days": int(len(sub)),
            "dominant_month": dominant_month,
            "dominant_persistence_class": persistence_mix,
            "domain_mean_H500_anom": hmean,
            "domain_mean_U500_anom": umean,
            "domain_mean_V500_anom": vmean
        })

    df_support = pd.DataFrame(support_rows).sort_values("regime_id").reset_index(drop=True)

    # -----------------------------------------------------
    # 9) Regime frequency by month
    # -----------------------------------------------------
    df_month_counts = (
        df_assign.groupby(["start_month", "month_name", "regime_label"], as_index=False)
        .size()
        .rename(columns={"size": "n_episode_onsets"})
        .sort_values(["start_month", "regime_label"])
        .reset_index(drop=True)
    )
    df_month_counts["pollutant"] = pollutant

    return {
        "pollutant": pollutant,
        "df_onsets": df_onsets,
        "df_preproc": df_preproc,
        "df_k_test": df_k_test,
        "preferred_k": preferred_k,
        "model_final": model_final,
        "df_assign": df_assign,
        "df_sizes": df_sizes,
        "df_support": df_support,
        "df_month_counts": df_month_counts,
        "H500_comp": H500_comp,
        "U500_comp": U500_comp,
        "V500_comp": V500_comp,
        "cluster_days": cluster_days
    }

print("Reusable pollutant-specific regime pipeline loaded.")

Reusable pollutant-specific regime pipeline loaded.


### 7.3 PM10 regime extraction

#### 7.3.1 Run PM10 pipeline

In [77]:
pm10_regime = run_light_kmeans_regime_pipeline(
    pollutant="PM10",
    onset_df=evt_pm10_onsets,
    cluster_da=CLUSTER_DA,
    valid_mask=VALID_MASK,
    k_values=CLUSTER_K_VALUES,
    default_k=DEFAULT_N_CLUSTERS,
    random_state=KMEANS_RANDOM_STATE,
    n_init=KMEANS_N_INIT,
    max_iter=KMEANS_MAX_ITER
)

display(pm10_regime["df_preproc"])
display(pm10_regime["df_k_test"].round(4))
display(pm10_regime["df_sizes"].round(3))

# --- Exports
pm10_regime["df_onsets"].to_csv(OUT_TAB / "stage4_PM10_onset_sample.csv", index=False)
pm10_regime["df_preproc"].to_csv(OUT_TAB / "stage4_PM10_cluster_preprocessing_summary.csv", index=False)
pm10_regime["df_k_test"].to_csv(OUT_TAB / "stage4_PM10_kmeans_candidate_solutions.csv", index=False)
pm10_regime["df_assign"].to_csv(OUT_TAB / "stage4_PM10_regime_assignments.csv", index=False)
pm10_regime["df_sizes"].to_csv(OUT_TAB / "stage4_PM10_regime_size_summary.csv", index=False)

print(f"saved: {OUT_TAB / 'stage4_PM10_onset_sample.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM10_cluster_preprocessing_summary.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM10_kmeans_candidate_solutions.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM10_regime_assignments.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM10_regime_size_summary.csv'}")

c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,pollutant,n_samples,n_features_valid_gridcells,standardized,sample_definition,clustering_field
0,PM10,63,9904,True,"Unique episode-onset days (day 0) in February,...",H500_ANOM


,pollutant,k,inertia,silhouette_score,min_cluster_size,max_cluster_size,cluster_sizes_desc
0,PM10,2,"422,144.59",0.26,30,33,"33, 30"
1,PM10,3,"355,508.19",0.23,10,28,"28, 25, 10"
2,PM10,4,"304,525.41",0.23,3,30,"30, 16, 14, 3"


,regime_id,regime_label,n_onset_days,fraction_of_sample,pollutant
0,1,R1,10,0.16,PM10
1,2,R2,25,0.40,PM10
2,3,R3,28,0.44,PM10


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_onset_sample.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_cluster_preprocessing_summary.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_kmeans_candidate_solutions.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_regime_assignments.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_regime_size_summary.csv


#### 7.3.2 PM10 candidate-k figure

In [ ]:
plot_df = pm10_regime["df_k_test"].copy()

fig, ax1 = plt.subplots(figsize=(8.5, 5.2), dpi=250)
ax2 = ax1.twinx()

# Make the plotting area slightly smaller inside the same canvas for cleaner spacing
fig.subplots_adjust(left=0.14, right=0.86, top=0.84, bottom=0.30)

# Contrasting palette + soft background
line1_color = "#1B4965"   # deep blue
line2_color = "#E76F51"   # warm coral
ax1.set_facecolor("#F8FBFF")

ax1.plot(
    plot_df["k"], plot_df["inertia"],
    marker="o", markersize=6, markerfacecolor="white", markeredgewidth=1.5,
    linewidth=2.4, color=line1_color, label="Inertia", zorder=3
)
ax2.plot(
    plot_df["k"], plot_df["silhouette_score"],
    marker="s", markersize=6, markerfacecolor="white", markeredgewidth=1.5,
    linewidth=2.2, linestyle="--", color=line2_color, label="Silhouette score", zorder=3
)

ax1.set_xlabel("Number of clusters (k)", fontsize=10)
ax1.set_ylabel("Inertia", fontsize=10, color=line1_color)
ax2.set_ylabel("Silhouette score", fontsize=10, color=line2_color)
ax1.set_xticks(plot_df["k"])
ax1.tick_params(axis="both", labelsize=9)
ax2.tick_params(axis="y", labelsize=9)
ax1.tick_params(axis="y", colors=line1_color)
ax2.tick_params(axis="y", colors=line2_color)

ax1.grid(axis="y", linestyle="--", alpha=0.35)
ax1.set_title(
    "PM10 onset-day k-means candidate solutions",
    fontsize=12,
    fontweight="bold"
)

# Right spine on ax2
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_linewidth(1)
ax2.spines["right"].set_color(line2_color)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(
    h1 + h2, l1 + l2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=True,
    fontsize=9
)

out_fig = OUT_FIG / "stage4_PM10_kmeans_candidate_solutions.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_kmeans_candidate_solutions.png


#### 7.3.3 PM10 regime composites and frequency context

In [84]:
display(pm10_regime["df_support"].round(3))
display(pm10_regime["df_month_counts"])

pm10_regime["df_support"].to_csv(OUT_TAB / "stage4_PM10_regime_interpretation_support.csv", index=False)
pm10_regime["df_month_counts"].to_csv(OUT_TAB / "stage4_PM10_regime_month_counts.csv", index=False)

print(f"saved: {OUT_TAB / 'stage4_PM10_regime_interpretation_support.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM10_regime_month_counts.csv'}")

,pollutant,regime_id,regime_label,n_onset_days,dominant_month,dominant_persistence_class,domain_mean_H500_anom,domain_mean_U500_anom,domain_mean_V500_anom
0,PM10,1,R1,10,February,one_day,21.89,-1.59,-3.24
1,PM10,2,R2,25,July,persistent,9.53,-0.15,0.50
2,PM10,3,R3,28,July,one_day,-11.94,0.50,-0.16


,start_month,month_name,regime_label,n_episode_onsets,pollutant
0,2,February,R1,8,PM10
1,2,February,R2,6,PM10
2,2,February,R3,5,PM10
3,5,May,R1,1,PM10
4,5,May,R2,6,PM10
5,5,May,R3,8,PM10
6,7,July,R1,1,PM10
7,7,July,R2,13,PM10
8,7,July,R3,15,PM10


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_regime_interpretation_support.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_regime_month_counts.csv


In [111]:
# =========================================================
# Figure: PM10 regime composites at 500 hPa
# =========================================================

nreg = pm10_regime["preferred_k"]
df_assign = pm10_regime["df_assign"]
H500_comp = pm10_regime["H500_comp"]
U500_comp = pm10_regime["U500_comp"]
V500_comp = pm10_regime["V500_comp"]

max_abs = float(np.nanmax(np.abs(H500_comp.values)))
levels = np.linspace(-max_abs, max_abs, 17)

fig, axes = plt.subplots(
    1, nreg,
    figsize=(5.6 * nreg, 4.8),
    dpi=250,
    subplot_kw={"projection": ccrs.PlateCarree()}
 )
fig.subplots_adjust(top=0.83, bottom=0.14, wspace=0.20)

if nreg == 1:
    axes = [axes]

for ax, rid in zip(axes, sorted(df_assign["regime_id"].unique())):
    h = H500_comp.sel(regime=rid)
    u = U500_comp.sel(regime=rid)
    v = V500_comp.sel(regime=rid)

    cf = ax.contourf(
        LON2D, LAT2D, h,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        transform=ccrs.PlateCarree()
    )

    cs = ax.contour(
        LON2D, LAT2D, h,
        levels=levels[::2],
        colors="k",
        linewidths=0.5,
        alpha=0.55,
        transform=ccrs.PlateCarree()
    )
    ax.clabel(cs, fmt="%.0f", fontsize=7, inline=True)

    step = 7
    ax.quiver(
        LON2D.values[::step, ::step],
        LAT2D.values[::step, ::step],
        u.values[::step, ::step],
        v.values[::step, ::step],
        transform=ccrs.PlateCarree(),
        scale=80,
        width=0.0022
    )

    apply_geo_format(ax, extent=DOMAIN_EXTENT)

    rect = mpatches.Rectangle(
        (VOM_BOX[0], VOM_BOX[1]),
        VOM_BOX[2], VOM_BOX[3],
        fill=False, linewidth=1.2, linestyle="--",
        transform=ccrs.PlateCarree()
    )
    ax.add_patch(rect)

    ax.plot(
        CDMX_REF["lon"], CDMX_REF["lat"],
        marker="o", markersize=4,
        transform=ccrs.PlateCarree()
    )

    n_days = int((df_assign["regime_id"] == rid).sum())
    dominant_month = df_assign.loc[df_assign["regime_id"] == rid, "month_name"].value_counts().idxmax()

    ax.set_title(
        f"R{rid}\nN={n_days} onset days | dominant month: {dominant_month}",
        fontsize=11, fontweight="bold"
    )

cbar = fig.colorbar(cf, ax=axes, orientation="horizontal", fraction=0.05, pad=0.13)
cbar_ticks = np.linspace(-max_abs, max_abs, 7)
cbar.set_ticks(cbar_ticks)
cbar.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
cbar.ax.tick_params(labelsize=10, pad=2)
cbar.set_label("500 hPa geopotential height anomaly (m)", fontsize=14, labelpad=6)

fig.suptitle(
    f"PM10 onset-day regime composites at 500 hPa (k = {nreg})",
    fontsize=16, fontweight="bold", y=0.92
)

out_fig = OUT_FIG / f"stage4_PM10_regime_composites_H500_k{nreg}.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_regime_composites_H500_k3.png


In [102]:
# =========================================================
# Figure: PM10 regime frequency by month
# =========================================================

month_name_order = [calendar.month_name[m] for m in FOCUS_MONTHS]

pm10_pivot = (
    pm10_regime["df_month_counts"]
    .pivot(index="month_name", columns="regime_label", values="n_episode_onsets")
    .fillna(0)
    .reindex(month_name_order)
)

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=250)
fig.subplots_adjust(left=0.10, right=0.95, top=0.86, bottom=0.28)

# Harmonious palette for regimes
regime_palette = ["#264653", "#2A9D8F", "#E9C46A"]
regime_colors = regime_palette[:len(pm10_pivot.columns)]

ax.set_facecolor("#F8FBFA")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

bottom = np.zeros(len(pm10_pivot))
x = np.arange(len(pm10_pivot.index))
totals = pm10_pivot.sum(axis=1).values
min_label_value = 2

for i, regime in enumerate(pm10_pivot.columns):
    vals = pm10_pivot[regime].values
    bars = ax.bar(
        x,
        vals,
        bottom=bottom,
        color=regime_colors[i],
        edgecolor="white",
        linewidth=0.9,
        width=0.62,
        alpha=0.95,
        label=regime
    )

    # Segment labels: count + percentage, only when segment is large enough
    for j, (v, bar) in enumerate(zip(vals, bars)):
        if v >= min_label_value and totals[j] > 0:
            pct = 100.0 * v / totals[j]
            txt_color = "white" if i < 2 else "black"
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[j] + v / 2,
                f"{int(round(v))} ({pct:.0f}%)",
                ha="center",
                va="center",
                fontsize=8,
                color=txt_color,
                fontweight="bold"
            )

    bottom += vals

# Monthly totals on top of each stacked bar
y_offset = max(totals) * 0.04 if np.max(totals) > 0 else 0.3
for j, total in enumerate(totals):
    ax.text(
        x[j],
        total + y_offset,
        f"{int(round(total))}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="0.25"
    )

ax.set_xticks(x)
ax.set_xticklabels([calendar.month_abbr[m] for m in FOCUS_MONTHS], fontsize=10)
ax.set_ylabel("PM10 episode-onset count", fontsize=10)
ax.set_title(
    f"PM10 regime frequency by month (k = {nreg})",
    fontsize=13,
    fontweight="bold",
    pad=10
)
ax.grid(axis="y", linestyle="--", alpha=0.30, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylim(0, np.max(totals) * 1.20 if np.max(totals) > 0 else 1)

# Legend centered below the plot
ax.legend(
    ncol=len(pm10_pivot.columns),
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    frameon=True,
    fancybox=True,
    framealpha=0.95,
    edgecolor="0.8",
    fontsize=9
)

out_fig = OUT_FIG / f"stage4_PM10_regime_frequency_k{nreg}.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_regime_frequency_k3.png


### 7.4 PM2.5 regime extraction

#### 7.4.1 Run PM2.5 pipeline

In [91]:
pm25_regime = run_light_kmeans_regime_pipeline(
    pollutant="PM2.5",
    onset_df=evt_pm25_onsets,
    cluster_da=CLUSTER_DA,
    valid_mask=VALID_MASK,
    k_values=CLUSTER_K_VALUES,
    default_k=DEFAULT_N_CLUSTERS,
    random_state=KMEANS_RANDOM_STATE,
    n_init=KMEANS_N_INIT,
    max_iter=KMEANS_MAX_ITER
)

display(pm25_regime["df_preproc"])
display(pm25_regime["df_k_test"].round(4))
display(pm25_regime["df_sizes"].round(3))

# --- Exports
pm25_regime["df_onsets"].to_csv(OUT_TAB / "stage4_PM25_onset_sample.csv", index=False)
pm25_regime["df_preproc"].to_csv(OUT_TAB / "stage4_PM25_cluster_preprocessing_summary.csv", index=False)
pm25_regime["df_k_test"].to_csv(OUT_TAB / "stage4_PM25_kmeans_candidate_solutions.csv", index=False)
pm25_regime["df_assign"].to_csv(OUT_TAB / "stage4_PM25_regime_assignments.csv", index=False)
pm25_regime["df_sizes"].to_csv(OUT_TAB / "stage4_PM25_regime_size_summary.csv", index=False)

print(f"saved: {OUT_TAB / 'stage4_PM25_onset_sample.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM25_cluster_preprocessing_summary.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM25_kmeans_candidate_solutions.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM25_regime_assignments.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM25_regime_size_summary.csv'}")

c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\DELL\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,pollutant,n_samples,n_features_valid_gridcells,standardized,sample_definition,clustering_field
0,PM2.5,67,9904,True,"Unique episode-onset days (day 0) in February,...",H500_ANOM


,pollutant,k,inertia,silhouette_score,min_cluster_size,max_cluster_size,cluster_sizes_desc
0,PM2.5,2,"447,396.19",0.27,30,37,"37, 30"
1,PM2.5,3,"368,152.44",0.24,7,31,"31, 29, 7"
2,PM2.5,4,"312,286.56",0.25,4,27,"27, 26, 10, 4"


,regime_id,regime_label,n_onset_days,fraction_of_sample,pollutant
0,1,R1,7,0.10,PM2.5
1,2,R2,31,0.46,PM2.5
2,3,R3,29,0.43,PM2.5


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_onset_sample.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_cluster_preprocessing_summary.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_kmeans_candidate_solutions.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_regime_assignments.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_regime_size_summary.csv


#### 7.4.2 PM2.5 candidate-k figure

In [93]:
plot_df = pm25_regime["df_k_test"].copy()

fig, ax1 = plt.subplots(figsize=(8.5, 5.2), dpi=250)
ax2 = ax1.twinx()

# Make the plotting area slightly smaller inside the same canvas for cleaner spacing
fig.subplots_adjust(left=0.14, right=0.86, top=0.84, bottom=0.30)

# Contrasting palette + soft background (matched to PM10 style)
line1_color = "#1B4965"   # deep blue
line2_color = "#E76F51"   # warm coral
ax1.set_facecolor("#F8FBFF")

ax1.plot(
    plot_df["k"], plot_df["inertia"],
    marker="o", markersize=6, markerfacecolor="white", markeredgewidth=1.5,
    linewidth=2.4, color=line1_color, label="Inertia", zorder=3
)
ax2.plot(
    plot_df["k"], plot_df["silhouette_score"],
    marker="s", markersize=6, markerfacecolor="white", markeredgewidth=1.5,
    linewidth=2.2, linestyle="--", color=line2_color, label="Silhouette score", zorder=3
)

ax1.set_xlabel("Number of clusters (k)", fontsize=10)
ax1.set_ylabel("Inertia", fontsize=10, color=line1_color)
ax2.set_ylabel("Silhouette score", fontsize=10, color=line2_color)
ax1.set_xticks(plot_df["k"])
ax1.tick_params(axis="both", labelsize=9)
ax2.tick_params(axis="y", labelsize=9)
ax1.tick_params(axis="y", colors=line1_color)
ax2.tick_params(axis="y", colors=line2_color)

ax1.grid(axis="y", linestyle="--", alpha=0.35)
ax1.set_title(
    "PM2.5 onset-day k-means candidate solutions",
    fontsize=12,
    fontweight="bold"
)

# Right spine on ax2
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_linewidth(1)
ax2.spines["right"].set_color(line2_color)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(
    h1 + h2, l1 + l2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=True,
    fontsize=9
)

out_fig = OUT_FIG / "stage4_PM25_kmeans_candidate_solutions.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_kmeans_candidate_solutions.png


#### 7.4.3 PM2.5 regime composites and frequency context

In [94]:
display(pm25_regime["df_support"].round(3))
display(pm25_regime["df_month_counts"])

pm25_regime["df_support"].to_csv(OUT_TAB / "stage4_PM25_regime_interpretation_support.csv", index=False)
pm25_regime["df_month_counts"].to_csv(OUT_TAB / "stage4_PM25_regime_month_counts.csv", index=False)

print(f"saved: {OUT_TAB / 'stage4_PM25_regime_interpretation_support.csv'}")
print(f"saved: {OUT_TAB / 'stage4_PM25_regime_month_counts.csv'}")

,pollutant,regime_id,regime_label,n_onset_days,dominant_month,dominant_persistence_class,domain_mean_H500_anom,domain_mean_U500_anom,domain_mean_V500_anom
0,PM2.5,1,R1,7,February,one_day,34.99,-2.24,-0.89
1,PM2.5,2,R2,31,July,one_day,4.04,0.99,-0.15
2,PM2.5,3,R3,29,July,one_day,-13.51,0.22,0.02


,start_month,month_name,regime_label,n_episode_onsets,pollutant
0,2,February,R1,5,PM2.5
1,2,February,R2,7,PM2.5
2,2,February,R3,8,PM2.5
3,5,May,R1,1,PM2.5
4,5,May,R2,11,PM2.5
5,5,May,R3,6,PM2.5
6,7,July,R1,1,PM2.5
7,7,July,R2,13,PM2.5
8,7,July,R3,15,PM2.5


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_regime_interpretation_support.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_regime_month_counts.csv


In [112]:
# =========================================================
# Figure: PM2.5 regime composites at 500 hPa
# =========================================================

nreg = pm25_regime["preferred_k"]
df_assign = pm25_regime["df_assign"]
H500_comp = pm25_regime["H500_comp"]
U500_comp = pm25_regime["U500_comp"]
V500_comp = pm25_regime["V500_comp"]

max_abs = float(np.nanmax(np.abs(H500_comp.values)))
levels = np.linspace(-max_abs, max_abs, 17)

fig, axes = plt.subplots(
    1, nreg,
    figsize=(5.6 * nreg, 4.8),
    dpi=250,
    subplot_kw={"projection": ccrs.PlateCarree()}
 )
fig.subplots_adjust(top=0.83, bottom=0.14, wspace=0.20)

if nreg == 1:
    axes = [axes]

for ax, rid in zip(axes, sorted(df_assign["regime_id"].unique())):
    h = H500_comp.sel(regime=rid)
    u = U500_comp.sel(regime=rid)
    v = V500_comp.sel(regime=rid)

    cf = ax.contourf(
        LON2D, LAT2D, h,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        transform=ccrs.PlateCarree()
    )

    cs = ax.contour(
        LON2D, LAT2D, h,
        levels=levels[::2],
        colors="k",
        linewidths=0.5,
        alpha=0.55,
        transform=ccrs.PlateCarree()
    )
    ax.clabel(cs, fmt="%.0f", fontsize=7, inline=True)

    step = 7
    ax.quiver(
        LON2D.values[::step, ::step],
        LAT2D.values[::step, ::step],
        u.values[::step, ::step],
        v.values[::step, ::step],
        transform=ccrs.PlateCarree(),
        scale=80,
        width=0.0022
    )

    apply_geo_format(ax, extent=DOMAIN_EXTENT)

    rect = mpatches.Rectangle(
        (VOM_BOX[0], VOM_BOX[1]),
        VOM_BOX[2], VOM_BOX[3],
        fill=False, linewidth=1.2, linestyle="--",
        transform=ccrs.PlateCarree()
    )
    ax.add_patch(rect)

    ax.plot(
        CDMX_REF["lon"], CDMX_REF["lat"],
        marker="o", markersize=4,
        transform=ccrs.PlateCarree()
    )

    n_days = int((df_assign["regime_id"] == rid).sum())
    dominant_month = df_assign.loc[df_assign["regime_id"] == rid, "month_name"].value_counts().idxmax()

    ax.set_title(
        f"R{rid}\nN={n_days} onset days | dominant month: {dominant_month}",
        fontsize=11, fontweight="bold"
    )

cbar = fig.colorbar(cf, ax=axes, orientation="horizontal", fraction=0.05, pad=0.13)
cbar_ticks = np.linspace(-max_abs, max_abs, 7)
cbar.set_ticks(cbar_ticks)
cbar.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
cbar.ax.tick_params(labelsize=10, pad=2)
cbar.set_label("500 hPa geopotential height anomaly (m)", fontsize=14, labelpad=6)

fig.suptitle(
    f"PM2.5 onset-day regime composites at 500 hPa (k = {nreg})",
    fontsize=16, fontweight="bold", y=0.92
)

out_fig = OUT_FIG / f"stage4_PM25_regime_composites_H500_k{nreg}.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_regime_composites_H500_k3.png


In [103]:
# =========================================================
# Figure: PM2.5 regime frequency by month
# =========================================================

pm25_pivot = (
    pm25_regime["df_month_counts"]
    .pivot(index="month_name", columns="regime_label", values="n_episode_onsets")
    .fillna(0)
    .reindex(month_name_order)
)

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=250)
fig.subplots_adjust(left=0.10, right=0.95, top=0.86, bottom=0.28)

# Harmonious palette for regimes (matched to PM10 style)
regime_palette = ["#264653", "#2A9D8F", "#E9C46A"]
regime_colors = regime_palette[:len(pm25_pivot.columns)]

ax.set_facecolor("#F8FBFA")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

bottom = np.zeros(len(pm25_pivot))
x = np.arange(len(pm25_pivot.index))
totals = pm25_pivot.sum(axis=1).values
min_label_value = 2

for i, regime in enumerate(pm25_pivot.columns):
    vals = pm25_pivot[regime].values
    bars = ax.bar(
        x,
        vals,
        bottom=bottom,
        color=regime_colors[i],
        edgecolor="white",
        linewidth=0.9,
        width=0.62,
        alpha=0.95,
        label=regime
    )

    # Segment labels: count + percentage, only when segment is large enough
    for j, (v, bar) in enumerate(zip(vals, bars)):
        if v >= min_label_value and totals[j] > 0:
            pct = 100.0 * v / totals[j]
            txt_color = "white" if i < 2 else "black"
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[j] + v / 2,
                f"{int(round(v))} ({pct:.0f}%)",
                ha="center",
                va="center",
                fontsize=8,
                color=txt_color,
                fontweight="bold"
            )

    bottom += vals

# Monthly totals on top of each stacked bar
y_offset = max(totals) * 0.04 if np.max(totals) > 0 else 0.3
for j, total in enumerate(totals):
    ax.text(
        x[j],
        total + y_offset,
        f"{int(round(total))}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="0.25"
    )

ax.set_xticks(x)
ax.set_xticklabels([calendar.month_abbr[m] for m in FOCUS_MONTHS], fontsize=10)
ax.set_ylabel("PM2.5 episode-onset count", fontsize=10)
ax.set_title(
    f"PM2.5 regime frequency by month (k = {nreg})",
    fontsize=13,
    fontweight="bold",
    pad=10
)
ax.grid(axis="y", linestyle="--", alpha=0.30, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylim(0, np.max(totals) * 1.20 if np.max(totals) > 0 else 1)

# Legend centered below the plot
ax.legend(
    ncol=len(pm25_pivot.columns),
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    frameon=True,
    fancybox=True,
    framealpha=0.95,
    edgecolor="0.8",
    fontsize=9
)

out_fig = OUT_FIG / f"stage4_PM25_regime_frequency_k{nreg}.png"
savefig(fig, out_fig, close=True)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_regime_frequency_k3.png


### 7.5 Cross-pollutant comparison and Stage 4 selection note

In [99]:
df_cross_pollutant_regime_summary = pd.concat([
    pm10_regime["df_sizes"],
    pm25_regime["df_sizes"]
], ignore_index=True)

display(df_cross_pollutant_regime_summary.round(3))

out_csv = OUT_TAB / "stage4_cross_pollutant_regime_size_summary.csv"
df_cross_pollutant_regime_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,regime_id,regime_label,n_onset_days,fraction_of_sample,pollutant
0,1,R1,10,0.16,PM10
1,2,R2,25,0.40,PM10
2,3,R3,28,0.44,PM10
3,1,R1,7,0.10,PM2.5
4,2,R2,31,0.46,PM2.5
5,3,R3,29,0.43,PM2.5


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_cross_pollutant_regime_size_summary.csv


In [101]:
df_stage4_regime_selection_note = pd.DataFrame([
    {
        "pollutant": "PM10",
        "preferred_k": pm10_regime["preferred_k"],
        "selection_note": (
            "For PM10, k = 3 was retained as the preferred solution because it provides a useful physical "
            "separation between a compact late-winter benchmark regime and two broader transition/wet-season "
            "structures, while preserving acceptable cluster sizes. Although k = 2 yielded a slightly higher "
            "silhouette score, it was considered too coarse for the Stage 4 objective. The k = 4 option was "
            "not favored because it introduced an undersampled cluster with limited added interpretive value."
        )
    },
    {
        "pollutant": "PM2.5",
        "preferred_k": pm25_regime["preferred_k"],
        "selection_note": (
            "For PM2.5, k = 3 was also retained as the preferred solution because it isolates a small but coherent "
            "late-winter regime and separates it from two broader May–July structures, which is more useful for "
            "Stage 4 interpretation than the coarser k = 2 solution. Although k = 2 achieved the highest silhouette "
            "score, k = 3 better preserves the seasonal contrast and remains acceptable in terms of cluster size. "
            "The k = 4 option was rejected because it produced an undersampled cluster with little practical benefit."
        )
    }
])

display(df_stage4_regime_selection_note)

out_csv = OUT_TAB / "stage4_regime_selection_note.csv"
df_stage4_regime_selection_note.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,pollutant,preferred_k,selection_note
0,PM10,3,"For PM10, k = 3 was retained as the preferred ..."
1,PM2.5,3,"For PM2.5, k = 3 was also retained as the pref..."


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_regime_selection_note.csv


## 8. Regime-conditioned lag composites for key windows

- How do those prototype-related events evolve around onset?

### 8.1 Shared lag framework and episode-persistence summary

In [104]:
# --- Shared lag setup
LAG_WINDOW = list(LAG_DAYS)   # expected: [-3, -2, -1, 0, 1, 2]
LAG_LABEL_MAP = {lag: f"{lag:+d}" if lag != 0 else "0" for lag in LAG_WINDOW}

print("Lag framework loaded:")
print(f"  Day 0 definition: {DAY0_DEFINITION}")
print(f"  Lag window: {LAG_WINDOW}")

# --- Focus-month persistence summary for both pollutants
pm10_persist_summary = (
    evt_pm10_onsets[evt_pm10_onsets["start_month"].isin(FOCUS_MONTHS)]
    .groupby(["start_month", "start_season", "persistence_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_episode_onsets"})
)
pm10_persist_summary["pollutant"] = "PM10"

pm25_persist_summary = (
    evt_pm25_onsets[evt_pm25_onsets["start_month"].isin(FOCUS_MONTHS)]
    .groupby(["start_month", "start_season", "persistence_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_episode_onsets"})
)
pm25_persist_summary["pollutant"] = "PM2.5"

df_lag_persistence_summary = pd.concat(
    [pm10_persist_summary, pm25_persist_summary],
    ignore_index=True
)

df_lag_persistence_summary["month_name"] = df_lag_persistence_summary["start_month"].map(lambda m: calendar.month_name[m])
df_lag_persistence_summary = df_lag_persistence_summary[
    ["pollutant", "start_month", "month_name", "start_season", "persistence_class", "n_episode_onsets"]
].sort_values(["pollutant", "start_month", "persistence_class"]).reset_index(drop=True)

display(df_lag_persistence_summary)

out_csv = OUT_TAB / "stage4_lag_persistence_summary_focusmonths.csv"
df_lag_persistence_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

Lag framework loaded:
  Day 0 definition: episode_onset
  Lag window: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]


,pollutant,start_month,month_name,start_season,persistence_class,n_episode_onsets
0,PM10,2,February,DJF,one_day,9
1,PM10,2,February,DJF,persistent,10
2,PM10,5,May,MAM,one_day,7
3,PM10,5,May,MAM,persistent,8
4,PM10,7,July,JJA,one_day,18
5,PM10,7,July,JJA,persistent,11
6,PM2.5,2,February,DJF,one_day,10
7,PM2.5,2,February,DJF,persistent,10
8,PM2.5,5,May,MAM,one_day,11
9,PM2.5,5,May,MAM,persistent,7


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_lag_persistence_summary_focusmonths.csv


In [164]:
# =========================================================
# Reusable helpers for monthly lag composites
# =========================================================

def _normalize_ts_list(values):
    """Return a normalized DatetimeIndex from an iterable of dates."""
    return pd.DatetimeIndex(pd.to_datetime(values)).normalize()


def _available_time_index(da):
    """Normalized DatetimeIndex from a DataArray time coordinate."""
    return pd.DatetimeIndex(pd.to_datetime(da["time"].values)).normalize()


def _safe_month_name(month_num):
    return calendar.month_name[int(month_num)]


def compute_monthly_lag_composites(
    onset_df,
    month_num,
    pollutant_label,
    H_da,
    U_da,
    V_da,
    lag_window=None
):
    """
    Compute lag composites for one pollutant and one focus month.

    Returns
    -------
    result : dict with
        - month_meta
        - onset_dates
        - H_comp, U_comp, V_comp
        - lag_counts
        - onset_subset
    """
    lag_window = lag_window or list(LAG_DAYS)

    # Restrict onset table
    sub = onset_df[onset_df["start_month"] == month_num].copy()
    sub = sub.sort_values("day0").reset_index(drop=True)

    onset_dates = _normalize_ts_list(sub["day0"].unique())
    avail = _available_time_index(H_da)

    # Pre-allocate composites
    H_list, U_list, V_list = [], [], []
    count_rows = []

    empty_template_H = xr.full_like(H_da.isel(time=0), np.nan)
    empty_template_U = xr.full_like(U_da.isel(time=0), np.nan)
    empty_template_V = xr.full_like(V_da.isel(time=0), np.nan)

    for lag in lag_window:
        lag_dates = onset_dates + pd.to_timedelta(lag, unit="D")
        valid_dates = lag_dates[lag_dates.isin(avail)]

        if len(valid_dates) > 0:
            Hc = H_da.sel(time=valid_dates).mean("time")
            Uc = U_da.sel(time=valid_dates).mean("time")
            Vc = V_da.sel(time=valid_dates).mean("time")
        else:
            Hc = empty_template_H.copy()
            Uc = empty_template_U.copy()
            Vc = empty_template_V.copy()

        H_list.append(Hc.expand_dims(lag=[lag]))
        U_list.append(Uc.expand_dims(lag=[lag]))
        V_list.append(Vc.expand_dims(lag=[lag]))

        count_rows.append({
            "pollutant": pollutant_label,
            "month": month_num,
            "month_name": _safe_month_name(month_num),
            "lag": lag,
            "lag_label": LAG_LABEL_MAP[lag],
            "n_valid_maps": len(valid_dates),
            "n_total_onsets_in_month": len(onset_dates)
        })

    H_comp = xr.concat(H_list, dim="lag")
    U_comp = xr.concat(U_list, dim="lag")
    V_comp = xr.concat(V_list, dim="lag")

    lag_counts = pd.DataFrame(count_rows)

    month_meta = pd.DataFrame([{
        "pollutant": pollutant_label,
        "month": month_num,
        "month_name": _safe_month_name(month_num),
        "n_onset_days": len(onset_dates),
        "n_one_day": int((sub["persistence_class"] == "one_day").sum()),
        "n_persistent": int((sub["persistence_class"] == "persistent").sum()),
        "lag_window": ", ".join([LAG_LABEL_MAP[l] for l in lag_window])
    }])

    return {
        "month_meta": month_meta,
        "onset_dates": onset_dates,
        "onset_subset": sub,
        "H_comp": H_comp,
        "U_comp": U_comp,
        "V_comp": V_comp,
        "lag_counts": lag_counts
    }


def export_monthly_lag_tables(result, out_prefix):
    """
    Export month-level lag metadata and lag-count tables.
    """
    out_meta = OUT_TAB / f"{out_prefix}_month_meta.csv"
    out_counts = OUT_TAB / f"{out_prefix}_lag_counts.csv"

    result["month_meta"].to_csv(out_meta, index=False)
    result["lag_counts"].to_csv(out_counts, index=False)

    print(f"saved: {out_meta}")
    print(f"saved: {out_counts}")


def plot_monthly_lag_composite_6panel(
    result,
    pollutant_label,
    month_num,
    out_fig_path,
    vector_step=6,
    contour_step=10
):
    """
    Export a 6-panel lag-composite figure (2x3) for one pollutant and one month.
    Style aligned with Stage 3 monthly multipanel figures.

    Shading: H500 anomaly
    Contours: H500 anomaly (positive solid / negative dashed)
    Vectors: U500/V500 anomalies
    """
    H_comp = result["H_comp"]
    U_comp = result["U_comp"]
    V_comp = result["V_comp"]
    lag_counts = result["lag_counts"]

    month_name = _safe_month_name(month_num)

    lon2d = H_comp["lon"].values
    lat2d = H_comp["lat"].values

    max_abs = float(np.nanmax(np.abs(H_comp.values)))
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        2, 3,
        figsize=(16, 10.4),
        dpi=250,
        subplot_kw={"projection": proj}
    )
    axes = axes.flatten()

    fig.subplots_adjust(
    left=0.05,
    right=0.90,
    bottom=0.18,
    top=0.82,
    wspace=0.10,
    hspace=0.24
)

    pcm_ref = None

    for i, lag in enumerate(LAG_WINDOW):
        ax = axes[i]
        ax.set_extent(
            [DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
             DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        h = H_comp.sel(lag=lag).values
        u = U_comp.sel(lag=lag).values
        v = V_comp.sel(lag=lag).values

        # --- Shading
        pcm = ax.pcolormesh(
            lon2d, lat2d, h,
            cmap="RdBu_r",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        # --- Contours (positive solid / negative dashed)
        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(
                lon2d, lat2d, h,
                levels=pos,
                colors="k",
                linewidths=0.7,
                transform=proj
            )
        if len(neg) > 0:
            ax.contour(
                lon2d, lat2d, h,
                levels=neg,
                colors="k",
                linewidths=0.7,
                linestyles="--",
                transform=proj
            )

         # --- Vectors
        yy = np.arange(0, h.shape[0], vector_step)
        xx = np.arange(0, h.shape[1], vector_step)

        q = ax.quiver(
            lon2d[np.ix_(yy, xx)],
            lat2d[np.ix_(yy, xx)],
            u[np.ix_(yy, xx)],
            v[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,          # smaller scale => larger visible arrows
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        # --- Valley of Mexico box
        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        # --- CDMX star
        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        # --- Title with lag and N
        n_valid = int(lag_counts.loc[lag_counts["lag"] == lag, "n_valid_maps"].iloc[0])
        ax.set_title(
            f"Lag {LAG_LABEL_MAP[lag]} (N={n_valid})",
            fontsize=12,
            weight="bold"
        )

        # --- Gridlines like Stage 3
        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if (i % 3) != 0:
            gl.left_labels = False

    # --- Bottom colorbar
    cbar_ax = fig.add_axes([0.32, 0.11, 0.36, 0.035])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("Z500′ anomaly (m)")

    # --- Titles
    n_onsets = int(result["month_meta"]["n_onset_days"].iloc[0])
    n_one_day = int(result["month_meta"]["n_one_day"].iloc[0])
    n_persistent = int(result["month_meta"]["n_persistent"].iloc[0])

    fig.suptitle(
        f"{pollutant_label}: {month_name} lag composites around episode onset (2012–2024)",
        fontsize=18,
        weight="bold",
        y=0.975
    )

    fig.text(
        0.5, 0.93,
        "Shading/contours: Z500′ anomaly | Vectors: Δ(U′,V′) at 500 hPa | Day 0 = onset",
        ha="center",
        fontsize=14,
        style="italic"
    )

    fig.text(
        0.5, 0.902,
        f"6-panel lag sequence (−3 to +2 days) | Onset sample: N={n_onsets} ({n_one_day} one-day, {n_persistent} persistent)",
        ha="center",
        fontsize=11
    )

    # --- Legend
    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(
        handles=legend_elements,
        loc="upper left",
        fontsize=11,
        frameon=False,
        bbox_to_anchor=(0.01, 0.95)
    )

    # Layout already controlled with fig.subplots_adjust(...)
    plt.savefig(out_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {out_fig_path}")


print("Monthly lag-composite helpers loaded.")

Monthly lag-composite helpers loaded.


### 8.2 PM10 lag composites

#### 8.2.1 February total

In [153]:
# =========================================================
# PM10 February total lag composites
# =========================================================

pm10_feb_lag = compute_monthly_lag_composites(
    onset_df=evt_pm10_onsets,
    month_num=2,
    pollutant_label="PM10",
    H_da=H500_ANOM,
    U_da=U500_ANOM,
    V_da=V500_ANOM,
    lag_window=LAG_WINDOW
)

display(pm10_feb_lag["month_meta"])
display(pm10_feb_lag["lag_counts"])

export_monthly_lag_tables(pm10_feb_lag, "stage4_PM10_February_lag")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent,lag_window
0,PM10,2,February,19,9,10,"-3, -2, -1, 0, +1, +2"


,pollutant,month,month_name,lag,lag_label,n_valid_maps,n_total_onsets_in_month
0,PM10,2,February,-3,-3,19,19
1,PM10,2,February,-2,-2,19,19
2,PM10,2,February,-1,-1,19,19
3,PM10,2,February,0,0,19,19
4,PM10,2,February,1,+1,19,19
5,PM10,2,February,2,+2,19,19


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_February_lag_month_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_February_lag_lag_counts.csv


In [165]:
plot_monthly_lag_composite_6panel(
    result=pm10_feb_lag,
    pollutant_label="PM10",
    month_num=2,
    out_fig_path=OUT_FIG / "stage4_PM10_February_lag_composites_6panel.png",
    vector_step=6,
    contour_step=10
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_February_lag_composites_6panel.png


#### 8.2.2 PM10 May total

In [155]:
# =========================================================
# PM10 May total lag composites
# =========================================================

pm10_may_lag = compute_monthly_lag_composites(
    onset_df=evt_pm10_onsets,
    month_num=5,
    pollutant_label="PM10",
    H_da=H500_ANOM,
    U_da=U500_ANOM,
    V_da=V500_ANOM,
    lag_window=LAG_WINDOW
)

display(pm10_may_lag["month_meta"])
display(pm10_may_lag["lag_counts"])

export_monthly_lag_tables(pm10_may_lag, "stage4_PM10_May_lag")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent,lag_window
0,PM10,5,May,15,7,8,"-3, -2, -1, 0, +1, +2"


,pollutant,month,month_name,lag,lag_label,n_valid_maps,n_total_onsets_in_month
0,PM10,5,May,-3,-3,15,15
1,PM10,5,May,-2,-2,15,15
2,PM10,5,May,-1,-1,15,15
3,PM10,5,May,0,0,15,15
4,PM10,5,May,1,+1,15,15
5,PM10,5,May,2,+2,15,15


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_May_lag_month_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_May_lag_lag_counts.csv


In [166]:
plot_monthly_lag_composite_6panel(
    result=pm10_may_lag,
    pollutant_label="PM10",
    month_num=5,
    out_fig_path=OUT_FIG / "stage4_PM10_May_lag_composites_6panel.png",
    vector_step=6,
    contour_step=10
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_May_lag_composites_6panel.png


#### 8.2.3 PM10 July total

In [157]:
# =========================================================
# PM10 July total lag composites
# =========================================================

pm10_jul_lag = compute_monthly_lag_composites(
    onset_df=evt_pm10_onsets,
    month_num=7,
    pollutant_label="PM10",
    H_da=H500_ANOM,
    U_da=U500_ANOM,
    V_da=V500_ANOM,
    lag_window=LAG_WINDOW
)

display(pm10_jul_lag["month_meta"])
display(pm10_jul_lag["lag_counts"])

export_monthly_lag_tables(pm10_jul_lag, "stage4_PM10_July_lag")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent,lag_window
0,PM10,7,July,29,18,11,"-3, -2, -1, 0, +1, +2"


,pollutant,month,month_name,lag,lag_label,n_valid_maps,n_total_onsets_in_month
0,PM10,7,July,-3,-3,29,29
1,PM10,7,July,-2,-2,29,29
2,PM10,7,July,-1,-1,29,29
3,PM10,7,July,0,0,29,29
4,PM10,7,July,1,+1,29,29
5,PM10,7,July,2,+2,29,29


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_July_lag_month_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_July_lag_lag_counts.csv


In [167]:
plot_monthly_lag_composite_6panel(
    result=pm10_jul_lag,
    pollutant_label="PM10",
    month_num=7,
    out_fig_path=OUT_FIG / "stage4_PM10_July_lag_composites_6panel.png",
    vector_step=6,
    contour_step=10
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_July_lag_composites_6panel.png


#### 8.2.4 PM10 regime-conditioned lag behavior

- Do the different PM10 regimes really represent synoptic structures distinct from the onset?
- If instead of looking at it by month, I look at it by regime, what is the average structure of each type of onset?

In [168]:
# =========================================================
# Regime summary table at lag 0
# =========================================================

# --- Start from Section 7 outputs
pm10_regime_summary_824 = pm10_regime["df_support"].copy()

# --- Add month-specific counts from Section 7
pm10_month_counts_wide = (
    pm10_regime["df_month_counts"]
    .pivot(index="regime_label", columns="month_name", values="n_episode_onsets")
    .fillna(0)
    .reset_index()
)

# Ensure expected columns exist
for mname in ["February", "May", "July"]:
    if mname not in pm10_month_counts_wide.columns:
        pm10_month_counts_wide[mname] = 0

pm10_regime_summary_824 = pm10_regime_summary_824.merge(
    pm10_month_counts_wide[["regime_label", "February", "May", "July"]],
    on="regime_label",
    how="left"
)

pm10_regime_summary_824 = pm10_regime_summary_824.rename(columns={
    "February": "n_February",
    "May": "n_May",
    "July": "n_July"
})

# --- Add concise interpretation tags
interpretation_map = {
    "R1": "Late-winter positive-height benchmark regime",
    "R2": "Weak positive non-winter structure",
    "R3": "Weak negative non-winter structure"
}
pm10_regime_summary_824["interpretation_tag"] = pm10_regime_summary_824["regime_label"].map(interpretation_map)

# --- Reorder columns
pm10_regime_summary_824 = pm10_regime_summary_824[
    [
        "pollutant", "regime_id", "regime_label", "n_onset_days",
        "dominant_month", "dominant_persistence_class",
        "n_February", "n_May", "n_July",
        "domain_mean_H500_anom", "domain_mean_U500_anom", "domain_mean_V500_anom",
        "interpretation_tag"
    ]
].sort_values("regime_id").reset_index(drop=True)

display(pm10_regime_summary_824.round(3))

out_csv = OUT_TAB / "stage4_PM10_regime_conditioned_lag0_summary.csv"
pm10_regime_summary_824.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,pollutant,regime_id,regime_label,n_onset_days,dominant_month,dominant_persistence_class,n_February,n_May,n_July,domain_mean_H500_anom,domain_mean_U500_anom,domain_mean_V500_anom,interpretation_tag
0,PM10,1,R1,10,February,one_day,8,1,1,21.89,-1.59,-3.24,Late-winter positive-height benchmark regime
1,PM10,2,R2,25,July,persistent,6,6,13,9.53,-0.15,0.50,Weak positive non-winter structure
2,PM10,3,R3,28,July,one_day,5,8,15,-11.94,0.50,-0.16,Weak negative non-winter structure


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_regime_conditioned_lag0_summary.csv


In [224]:
# =========================================================
# Figure: Improved PM10 lag-0 regime fingerprints
# =========================================================

# --- Build lag-0 composites for each PM10 regime using all onset days in that regime
df_assign_pm10 = pm10_regime["df_assign"].copy()

regime_ids = sorted(df_assign_pm10["regime_id"].unique())

H0_list, U0_list, V0_list = [], [], []
rows_info = []

for rid in regime_ids:
    dates_r = pd.DatetimeIndex(df_assign_pm10.loc[df_assign_pm10["regime_id"] == rid, "day0"]).normalize()

    H0 = H500_ANOM.sel(time=dates_r).mean("time").expand_dims(regime=[rid])
    U0 = U500_ANOM.sel(time=dates_r).mean("time").expand_dims(regime=[rid])
    V0 = V500_ANOM.sel(time=dates_r).mean("time").expand_dims(regime=[rid])

    H0_list.append(H0)
    U0_list.append(U0)
    V0_list.append(V0)

    rows_info.append({
        "regime_id": rid,
        "regime_label": f"R{rid}",
        "n_onset_days": int(len(dates_r)),
        "dominant_month": df_assign_pm10.loc[df_assign_pm10["regime_id"] == rid, "month_name"].value_counts().idxmax()
    })

H0_comp = xr.concat(H0_list, dim="regime")
U0_comp = xr.concat(U0_list, dim="regime")
V0_comp = xr.concat(V0_list, dim="regime")

df_pm10_regime_lag0_info = pd.DataFrame(rows_info).sort_values("regime_id").reset_index(drop=True)

# --- Shared plotting scale from regime composites themselves
max_abs = float(np.nanmax(np.abs(H0_comp.values)))
if not np.isfinite(max_abs) or max_abs == 0:
    max_abs = 1.0

# keep simple and readable
max_abs = float(np.ceil(max_abs / 10.0) * 10)
norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(
    1, len(regime_ids),
    figsize=(5.7 * len(regime_ids), 5.8),
    dpi=250,
    subplot_kw={"projection": proj}
)

if len(regime_ids) == 1:
    axes = [axes]

pcm_ref = None

for ax, rid in zip(axes, regime_ids):
    h = H0_comp.sel(regime=rid).values
    u = U0_comp.sel(regime=rid).values
    v = V0_comp.sel(regime=rid).values

    lon2d = H0_comp["lon"].values
    lat2d = H0_comp["lat"].values

    ax.set_extent(
        [DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
         DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]],
        crs=proj
    )

    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

    # Shading
    pcm = ax.pcolormesh(
        lon2d, lat2d, h,
        cmap="RdBu_r",
        norm=norm,
        shading="auto",
        transform=proj
    )
    pcm_ref = pcm

    # Contours
    contour_step = 10
    lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
    pos = lev[lev > 0]
    neg = lev[lev < 0]

    if len(pos) > 0:
        ax.contour(
            lon2d, lat2d, h,
            levels=pos,
            colors="k",
            linewidths=0.7,
            transform=proj
        )
    if len(neg) > 0:
        ax.contour(
            lon2d, lat2d, h,
            levels=neg,
            colors="k",
            linewidths=0.7,
            linestyles="--",
            transform=proj
        )

    # Vectors
    vector_step = 6
    yy = np.arange(0, h.shape[0], vector_step)
    xx = np.arange(0, h.shape[1], vector_step)

    ax.quiver(
        lon2d[np.ix_(yy, xx)],
        lat2d[np.ix_(yy, xx)],
        u[np.ix_(yy, xx)],
        v[np.ix_(yy, xx)],
        transform=proj,
        color="black",
        scale=80,
        width=0.0022,
        headwidth=3.5,
        headlength=4.5,
        headaxislength=4.0,
        minlength=0.05,
        pivot="middle",
        zorder=4
    )

    # VOM box
    rect = mpatches.Rectangle(
        (VOM_BOX[0], VOM_BOX[1]),
        VOM_BOX[2], VOM_BOX[3],
        fill=False,
        edgecolor="k",
        linewidth=1.0,
        linestyle="-",
        transform=proj
    )
    ax.add_patch(rect)

    # CDMX star
    ax.plot(
        CDMX_REF["lon"], CDMX_REF["lat"],
        marker="*",
        color="gold",
        markersize=9,
        markeredgecolor="k",
        transform=proj
    )

    info_row = df_pm10_regime_lag0_info[df_pm10_regime_lag0_info["regime_id"] == rid].iloc[0]
    ax.set_title(
        f"R{rid}\nLag 0 | N={int(info_row['n_onset_days'])} | dominant month: {info_row['dominant_month']}",
        fontsize=12,
        weight="bold"
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    if rid != regime_ids[0]:
        gl.left_labels = False

# Colorbar below
cbar_ax = fig.add_axes([0.32, 0.11, 0.36, 0.04])
cb = fig.colorbar(
    pcm_ref,
    cax=cbar_ax,
    orientation="horizontal",
    extend="both"
)
cb.set_label("Z500′ anomaly (m)")

# Titles
fig.suptitle(
    "PM10: regime-conditioned onset fingerprints at lag 0 (2012–2024)",
    fontsize=18,
    weight="bold",
    y=0.98
)

fig.text(
    0.5, 0.915,
    "Shading/contours: Z500′ anomaly | Vectors: Δ(U′,V′) at 500 hPa | Each panel averages all onset days assigned to the regime",
    ha="center",
    fontsize=13,
    style="italic"
)

fig.text(
    0.5, 0.88,
    "Used together with Section 7 regime frequencies to compare the February benchmark against broader May–July structures",
    ha="center",
    fontsize=11
)

legend_elements = [
    mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
    plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
               markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
]
fig.legend(
    handles=legend_elements,
    loc="upper left",
    fontsize=11,
    frameon=False,
    bbox_to_anchor=(0.01, 0.98)
)

fig.subplots_adjust(
    left=0.05,
    right=0.96,
    bottom=0.20,
    top=0.82,
    wspace=0.10
)

out_fig = OUT_FIG / "stage4_PM10_regime_conditioned_lag0_fingerprints.png"
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_regime_conditioned_lag0_fingerprints.png


### 8.3 PM2.5 lag composites

#### 8.3.1 February total

In [173]:
# =========================================================
# PM2.5 February total lag composites
# =========================================================

pm25_feb_lag = compute_monthly_lag_composites(
    onset_df=evt_pm25_onsets,
    month_num=2,
    pollutant_label="PM2.5",
    H_da=H500_ANOM,
    U_da=U500_ANOM,
    V_da=V500_ANOM,
    lag_window=LAG_WINDOW
)

display(pm25_feb_lag["month_meta"])
display(pm25_feb_lag["lag_counts"])

export_monthly_lag_tables(pm25_feb_lag, "stage4_PM25_February_lag")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent,lag_window
0,PM2.5,2,February,20,10,10,"-3, -2, -1, 0, +1, +2"


,pollutant,month,month_name,lag,lag_label,n_valid_maps,n_total_onsets_in_month
0,PM2.5,2,February,-3,-3,20,20
1,PM2.5,2,February,-2,-2,20,20
2,PM2.5,2,February,-1,-1,20,20
3,PM2.5,2,February,0,0,20,20
4,PM2.5,2,February,1,+1,20,20
5,PM2.5,2,February,2,+2,20,20


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_February_lag_month_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_February_lag_lag_counts.csv


In [174]:
plot_monthly_lag_composite_6panel(
    result=pm25_feb_lag,
    pollutant_label="PM2.5",
    month_num=2,
    out_fig_path=OUT_FIG / "stage4_PM25_February_lag_composites_6panel.png",
    vector_step=6,
    contour_step=10
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_February_lag_composites_6panel.png


#### 8.3.2 May total

In [175]:
# =========================================================
# PM2.5 May total lag composites
# =========================================================

pm25_may_lag = compute_monthly_lag_composites(
    onset_df=evt_pm25_onsets,
    month_num=5,
    pollutant_label="PM2.5",
    H_da=H500_ANOM,
    U_da=U500_ANOM,
    V_da=V500_ANOM,
    lag_window=LAG_WINDOW
)

display(pm25_may_lag["month_meta"])
display(pm25_may_lag["lag_counts"])

export_monthly_lag_tables(pm25_may_lag, "stage4_PM25_May_lag")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent,lag_window
0,PM2.5,5,May,18,11,7,"-3, -2, -1, 0, +1, +2"


,pollutant,month,month_name,lag,lag_label,n_valid_maps,n_total_onsets_in_month
0,PM2.5,5,May,-3,-3,18,18
1,PM2.5,5,May,-2,-2,18,18
2,PM2.5,5,May,-1,-1,18,18
3,PM2.5,5,May,0,0,18,18
4,PM2.5,5,May,1,+1,18,18
5,PM2.5,5,May,2,+2,18,18


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_May_lag_month_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_May_lag_lag_counts.csv


In [176]:
plot_monthly_lag_composite_6panel(
    result=pm25_may_lag,
    pollutant_label="PM2.5",
    month_num=5,
    out_fig_path=OUT_FIG / "stage4_PM25_May_lag_composites_6panel.png",
    vector_step=6,
    contour_step=10
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_May_lag_composites_6panel.png


#### 8.3.3 July total

In [177]:
# =========================================================
# PM2.5 July total lag composites
# =========================================================

pm25_jul_lag = compute_monthly_lag_composites(
    onset_df=evt_pm25_onsets,
    month_num=7,
    pollutant_label="PM2.5",
    H_da=H500_ANOM,
    U_da=U500_ANOM,
    V_da=V500_ANOM,
    lag_window=LAG_WINDOW
)

display(pm25_jul_lag["month_meta"])
display(pm25_jul_lag["lag_counts"])

export_monthly_lag_tables(pm25_jul_lag, "stage4_PM25_July_lag")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent,lag_window
0,PM2.5,7,July,29,21,8,"-3, -2, -1, 0, +1, +2"


,pollutant,month,month_name,lag,lag_label,n_valid_maps,n_total_onsets_in_month
0,PM2.5,7,July,-3,-3,29,29
1,PM2.5,7,July,-2,-2,29,29
2,PM2.5,7,July,-1,-1,29,29
3,PM2.5,7,July,0,0,29,29
4,PM2.5,7,July,1,+1,29,29
5,PM2.5,7,July,2,+2,29,29


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_July_lag_month_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_July_lag_lag_counts.csv


In [178]:
plot_monthly_lag_composite_6panel(
    result=pm25_jul_lag,
    pollutant_label="PM2.5",
    month_num=7,
    out_fig_path=OUT_FIG / "stage4_PM25_July_lag_composites_6panel.png",
    vector_step=6,
    contour_step=10
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_July_lag_composites_6panel.png


#### 8.3.4 PM2.5 regime-conditioned lag behavior

In [179]:
# =========================================================
# Regime summary table at lag 0
# =========================================================

pm25_regime_summary_834 = pm25_regime["df_support"].copy()

pm25_month_counts_wide = (
    pm25_regime["df_month_counts"]
    .pivot(index="regime_label", columns="month_name", values="n_episode_onsets")
    .fillna(0)
    .reset_index()
)

for mname in ["February", "May", "July"]:
    if mname not in pm25_month_counts_wide.columns:
        pm25_month_counts_wide[mname] = 0

pm25_regime_summary_834 = pm25_regime_summary_834.merge(
    pm25_month_counts_wide[["regime_label", "February", "May", "July"]],
    on="regime_label",
    how="left"
)

pm25_regime_summary_834 = pm25_regime_summary_834.rename(columns={
    "February": "n_February",
    "May": "n_May",
    "July": "n_July"
})

interpretation_map_pm25 = {
    "R1": "Late-winter positive-height benchmark regime",
    "R2": "Weak positive non-winter structure",
    "R3": "Weak negative non-winter structure"
}
pm25_regime_summary_834["interpretation_tag"] = pm25_regime_summary_834["regime_label"].map(interpretation_map_pm25)

pm25_regime_summary_834 = pm25_regime_summary_834[
    [
        "pollutant", "regime_id", "regime_label", "n_onset_days",
        "dominant_month", "dominant_persistence_class",
        "n_February", "n_May", "n_July",
        "domain_mean_H500_anom", "domain_mean_U500_anom", "domain_mean_V500_anom",
        "interpretation_tag"
    ]
].sort_values("regime_id").reset_index(drop=True)

display(pm25_regime_summary_834.round(3))

out_csv = OUT_TAB / "stage4_PM25_regime_conditioned_lag0_summary.csv"
pm25_regime_summary_834.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,pollutant,regime_id,regime_label,n_onset_days,dominant_month,dominant_persistence_class,n_February,n_May,n_July,domain_mean_H500_anom,domain_mean_U500_anom,domain_mean_V500_anom,interpretation_tag
0,PM2.5,1,R1,7,February,one_day,5,1,1,34.99,-2.24,-0.89,Late-winter positive-height benchmark regime
1,PM2.5,2,R2,31,July,one_day,7,11,13,4.04,0.99,-0.15,Weak positive non-winter structure
2,PM2.5,3,R3,29,July,one_day,8,6,15,-13.51,0.22,0.02,Weak negative non-winter structure


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_regime_conditioned_lag0_summary.csv


In [223]:
# =========================================================
# Figure: Improved PM2.5 lag-0 regime fingerprints
# =========================================================

df_assign_pm25 = pm25_regime["df_assign"].copy()

regime_ids = sorted(df_assign_pm25["regime_id"].unique())

H0_list, U0_list, V0_list = [], [], []
rows_info = []

for rid in regime_ids:
    dates_r = pd.DatetimeIndex(df_assign_pm25.loc[df_assign_pm25["regime_id"] == rid, "day0"]).normalize()

    H0 = H500_ANOM.sel(time=dates_r).mean("time").expand_dims(regime=[rid])
    U0 = U500_ANOM.sel(time=dates_r).mean("time").expand_dims(regime=[rid])
    V0 = V500_ANOM.sel(time=dates_r).mean("time").expand_dims(regime=[rid])

    H0_list.append(H0)
    U0_list.append(U0)
    V0_list.append(V0)

    rows_info.append({
        "regime_id": rid,
        "regime_label": f"R{rid}",
        "n_onset_days": int(len(dates_r)),
        "dominant_month": df_assign_pm25.loc[df_assign_pm25["regime_id"] == rid, "month_name"].value_counts().idxmax()
    })

H0_comp = xr.concat(H0_list, dim="regime")
U0_comp = xr.concat(U0_list, dim="regime")
V0_comp = xr.concat(V0_list, dim="regime")

df_pm25_regime_lag0_info = pd.DataFrame(rows_info).sort_values("regime_id").reset_index(drop=True)

max_abs = float(np.nanmax(np.abs(H0_comp.values)))
if not np.isfinite(max_abs) or max_abs == 0:
    max_abs = 1.0
max_abs = float(np.ceil(max_abs / 10.0) * 10)
norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(
    1, len(regime_ids),
    figsize=(5.7 * len(regime_ids), 5.8),
    dpi=250,
    subplot_kw={"projection": proj}
)

if len(regime_ids) == 1:
    axes = [axes]

pcm_ref = None

for ax, rid in zip(axes, regime_ids):
    h = H0_comp.sel(regime=rid).values
    u = U0_comp.sel(regime=rid).values
    v = V0_comp.sel(regime=rid).values

    lon2d = H0_comp["lon"].values
    lat2d = H0_comp["lat"].values

    ax.set_extent(
        [DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
         DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]],
        crs=proj
    )

    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

    pcm = ax.pcolormesh(
        lon2d, lat2d, h,
        cmap="RdBu_r",
        norm=norm,
        shading="auto",
        transform=proj
    )
    pcm_ref = pcm

    contour_step = 10
    lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
    pos = lev[lev > 0]
    neg = lev[lev < 0]

    if len(pos) > 0:
        ax.contour(
            lon2d, lat2d, h,
            levels=pos,
            colors="k",
            linewidths=0.7,
            transform=proj
        )
    if len(neg) > 0:
        ax.contour(
            lon2d, lat2d, h,
            levels=neg,
            colors="k",
            linewidths=0.7,
            linestyles="--",
            transform=proj
        )

    vector_step = 6
    yy = np.arange(0, h.shape[0], vector_step)
    xx = np.arange(0, h.shape[1], vector_step)

    ax.quiver(
        lon2d[np.ix_(yy, xx)],
        lat2d[np.ix_(yy, xx)],
        u[np.ix_(yy, xx)],
        v[np.ix_(yy, xx)],
        transform=proj,
        color="black",
        scale=80,
        width=0.0022,
        headwidth=3.5,
        headlength=4.5,
        headaxislength=4.0,
        minlength=0.05,
        pivot="middle",
        zorder=4
    )

    rect = mpatches.Rectangle(
        (VOM_BOX[0], VOM_BOX[1]),
        VOM_BOX[2], VOM_BOX[3],
        fill=False,
        edgecolor="k",
        linewidth=1.0,
        linestyle="-",
        transform=proj
    )
    ax.add_patch(rect)

    ax.plot(
        CDMX_REF["lon"], CDMX_REF["lat"],
        marker="*",
        color="gold",
        markersize=9,
        markeredgecolor="k",
        transform=proj
    )

    info_row = df_pm25_regime_lag0_info[df_pm25_regime_lag0_info["regime_id"] == rid].iloc[0]
    ax.set_title(
        f"R{rid}\nLag 0 | N={int(info_row['n_onset_days'])} | dominant month: {info_row['dominant_month']}",
        fontsize=12,
        weight="bold"
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    if rid != regime_ids[0]:
        gl.left_labels = False

cbar_ax = fig.add_axes([0.32, 0.11, 0.36, 0.04])
cb = fig.colorbar(
    pcm_ref,
    cax=cbar_ax,
    orientation="horizontal",
    extend="both"
)
cb.set_label("Z500′ anomaly (m)")

fig.suptitle(
    "PM2.5: regime-conditioned onset fingerprints at lag 0 (2012–2024)",
    fontsize=18,
    weight="bold",
    y=0.98
)

fig.text(
    0.5, 0.915,
    "Shading/contours: Z500′ anomaly | Vectors: Δ(U′,V′) at 500 hPa | Each panel averages all onset days assigned to the regime",
    ha="center",
    fontsize=13,
    style="italic"
)

fig.text(
    0.5, 0.88,
    "Used together with Section 7 regime frequencies to compare the February benchmark against broader May–July structures",
    ha="center",
    fontsize=11
)

legend_elements = [
    mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
    plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
               markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
]
fig.legend(
    handles=legend_elements,
    loc="upper left",
    fontsize=11,
    frameon=False,
    bbox_to_anchor=(0.01, 0.98)
)

fig.subplots_adjust(
    left=0.05,
    right=0.96,
    bottom=0.20,
    top=0.82,
    wspace=0.10
)

out_fig = OUT_FIG / "stage4_PM25_regime_conditioned_lag0_fingerprints.png"
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_regime_conditioned_lag0_fingerprints.png


### 8.4 Summary of lag-based physical insights

#### 8.4.1 Insight table

In [184]:
df_stage4_lag_insights = pd.DataFrame([
    {
        "insight_id": 1,
        "theme": "February benchmark",
        "statement": (
            "February shows the clearest preconditioning-to-onset evolution, especially for PM10, "
            "with a coherent late-winter positive-height structure building into day 0 and persisting afterward."
        )
    },
    {
        "insight_id": 2,
        "theme": "May transition",
        "statement": (
            "May retains meaningful lag evolution but does not reproduce the February benchmark cleanly, "
            "supporting its role as a transition month rather than as a second winter-type peak."
        )
    },
    {
        "insight_id": 3,
        "theme": "July wet-season contrast",
        "statement": (
            "July behaves as a weaker and noisier contrast case for PM10, whereas PM2.5 retains a clearer and more coherent "
            "lag structure under the wetter seasonal background, especially through a persistent negative-height anomaly pattern."
        )
    },
    {
        "insight_id": 4,
        "theme": "PM10 regime structure",
        "statement": (
            "For PM10, the strongest onset fingerprint is concentrated in the late-winter benchmark regime (R1), "
            "whereas May and July are distributed mainly across broader non-winter structures (R2–R3)."
        )
    },
    {
        "insight_id": 5,
        "theme": "PM2.5 regime structure",
        "statement": (
            "For PM2.5, onset days separate into a compact February benchmark regime and two broader May–July structures, "
            "confirming that PM2.5 retains synoptic sensitivity beyond late winter rather than collapsing into a purely weak wet-season signal."
        )
    },
    {
        "insight_id": 6,
        "theme": "Stage 4 synthesis",
        "statement": (
            "The lag analysis supports a reduced Stage 4 narrative built around three physical stories: "
            "a robust February benchmark, a transitional May case, and a wetter July contrast."
        )
    }
])

display(df_stage4_lag_insights)

out_csv = OUT_TAB / "stage4_lag_based_physical_insights_summary.csv"
df_stage4_lag_insights.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,insight_id,theme,statement
0,1,February benchmark,February shows the clearest preconditioning-to...
1,2,May transition,May retains meaningful lag evolution but does ...
2,3,July wet-season contrast,July behaves as a weaker and noisier contrast ...
3,4,PM10 regime structure,"For PM10, the strongest onset fingerprint is c..."
4,5,PM2.5 regime structure,"For PM2.5, onset days separate into a compact ..."
5,6,Stage 4 synthesis,The lag analysis supports a reduced Stage 4 na...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_lag_based_physical_insights_summary.csv


#### 8.4.2 Cross-pollutant summary table

In [ ]:
df_stage4_cross_pollutant_lag_summary = pd.DataFrame([
    {
        "pollutant": "PM10",
        "February": "Strongest and clearest lag evolution",
        "May": "Transition behavior; weaker than February",
        "July": "Weak/noisy wet-season contrast with fragmented signal",
        "regime_note": "R1 captures the late-winter benchmark; R2–R3 dominate non-winter onset days"
    },
    {
        "pollutant": "PM2.5",
        "February": "Strong late-winter benchmark structure",
        "May": "Transition behavior remains synoptically meaningful",
        "July": "Retains a clearer wet-season structure than PM10",
        "regime_note": "Broader May–July regime occupancy confirms retained synoptic sensitivity outside late winter"
    }
])

display(df_stage4_cross_pollutant_lag_summary)

out_csv = OUT_TAB / "stage4_cross_pollutant_lag_summary.csv"
df_stage4_cross_pollutant_lag_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,pollutant,February,May,July,regime_note
0,PM10,Strongest and clearest lag evolution,Transition behavior; weaker than February,Weak/noisy wet-season contrast,R1 captures the late-winter benchmark; R2–R3 d...
1,PM2.5,Strong late-winter benchmark structure,Transition behavior remains synoptically meani...,Retains a clearer wet-season structure than PM10,Broader May–July regime occupancy confirms ret...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_cross_pollutant_lag_summary.csv


## 9. Targeted physical mechanism check 

- Are those stories physically consistent with precipitation and low-level flow/moisture?
- Do extreme events occur under conditions physically consistent with rainfall suppression, anomalous wet/dry structure, and low tropospheric flow patterns?

### 9.0 One-time setup for SH850 mini-file

#### 9.0.1 Download SHUM850 file via OPeNDAP

In [186]:
# =========================================================
# One-time setup: build SH850 mini-file from NOAA/PSL
# =========================================================

BASE = "https://psl.noaa.gov/thredds/dodsC/Datasets/NARR/Dailies/pressure/"
years  = range(2012, 2025)               # 2012–2024
months = [f"{m:02d}" for m in range(1, 13)]
twin   = (pd.Timestamp("2012-01-01"), pd.Timestamp("2024-12-31"))

# --- Detect bbox once from a sample file
sample = xr.open_dataset(f"{BASE}hgt.201201.nc", engine="netcdf4")
lat2d, lon2d = sample["lat"], sample["lon"]

LON_MIN, LON_MAX = -120, -85
LAT_MIN, LAT_MAX = 10, 35

mask = (lon2d >= LON_MIN) & (lon2d <= LON_MAX) & (lat2d >= LAT_MIN) & (lat2d <= LAT_MAX)
yy, xx = np.where(mask)
y0, y1 = yy.min(), yy.max()
x0, x1 = xx.min(), xx.max()
sample.close()

print(f"Detected bbox indices: y={y0}:{y1}, x={x0}:{x1}")

Detected bbox indices: y=0:99, x=120:264


In [187]:
def build_mini_level(var_folder_name: str, varname: str, level_value: int, out_path: str):
    """
    Open month by month via OPeNDAP, subset:
      - requested pressure level (e.g., 850 hPa)
      - bbox by indices y,x
      - time window 2012–2024
    Concatenate by 'time' and save compressed.
    """
    pieces = []

    for y in years:
        for m in months:
            url = f"{BASE}{var_folder_name}.{y}{m}.nc"
            try:
                ds = xr.open_dataset(url, engine="netcdf4", decode_cf=True)
                da = ds[varname]

                # Time filter
                da = da.sel(time=slice(twin[0], twin[1]))

                # Pressure level
                if "level" in da.dims:
                    da = da.sel(level=level_value)

                # Spatial cropping using detected bbox indices
                da = da.isel(y=slice(y0, y1 + 1), x=slice(x0, x1 + 1))

                # Skip empty months if any
                if da.sizes.get("time", 0) == 0:
                    ds.close()
                    continue

                # Modest chunking
                da = da.chunk({"time": min(da.sizes["time"], 31)})

                pieces.append(da)
                ds.close()

                print(f"OK {y}-{m} ({varname}@{level_value})")
            except Exception as e:
                print(f"Skip {y}-{m} ({varname}@{level_value}): {str(e)[:120]}")
                continue

    if not pieces:
        raise RuntimeError(f"Nothing was gathered for {varname}@{level_value}.")

    out = xr.concat(
        pieces,
        dim="time",
        compat="override",
        join="override",
        coords="minimal"
    )

    comp = dict(zlib=True, complevel=3)
    out.to_netcdf(out_path, encoding={varname: comp})
    print(f"Saved: {out_path} ({out.sizes['time']} days)")

In [188]:
# --- Build SH850
build_mini_level(
    var_folder_name="shum",
    varname="shum",
    level_value=850,
    out_path="shum850_mex_2012_2024.nc"
)

OK 2012-01 (shum@850)
OK 2012-02 (shum@850)
OK 2012-03 (shum@850)
OK 2012-04 (shum@850)
OK 2012-05 (shum@850)
OK 2012-06 (shum@850)
OK 2012-07 (shum@850)
OK 2012-08 (shum@850)
OK 2012-09 (shum@850)
OK 2012-10 (shum@850)
OK 2012-11 (shum@850)
OK 2012-12 (shum@850)
OK 2013-01 (shum@850)
OK 2013-02 (shum@850)
OK 2013-03 (shum@850)
OK 2013-04 (shum@850)
OK 2013-05 (shum@850)
OK 2013-06 (shum@850)
OK 2013-07 (shum@850)
OK 2013-08 (shum@850)
OK 2013-09 (shum@850)
OK 2013-10 (shum@850)
OK 2013-11 (shum@850)
OK 2013-12 (shum@850)
OK 2014-01 (shum@850)
OK 2014-02 (shum@850)
OK 2014-03 (shum@850)
OK 2014-04 (shum@850)
OK 2014-05 (shum@850)
OK 2014-06 (shum@850)
OK 2014-07 (shum@850)
OK 2014-08 (shum@850)
OK 2014-09 (shum@850)
OK 2014-10 (shum@850)
OK 2014-11 (shum@850)
OK 2014-12 (shum@850)
OK 2015-01 (shum@850)
OK 2015-02 (shum@850)
OK 2015-03 (shum@850)
OK 2015-04 (shum@850)
OK 2015-05 (shum@850)
OK 2015-06 (shum@850)
OK 2015-07 (shum@850)
OK 2015-08 (shum@850)
OK 2015-09 (shum@850)
OK 2015-10

#### 9.0.2 Sanity check after download

In [189]:
ds_sh_test = xr.open_dataset("shum850_mex_2012_2024.nc")
print(ds_sh_test)

da_sh_test = ds_sh_test["shum"]
print(da_sh_test)
print("time range:", pd.to_datetime(da_sh_test["time"].values[0]).date(), "->", pd.to_datetime(da_sh_test["time"].values[-1]).date())
print("shape:", da_sh_test.shape)
print("min/max:", float(np.nanmin(da_sh_test.values)), float(np.nanmax(da_sh_test.values)))

<xarray.Dataset> Size: 276MB
Dimensions:  (time: 4749, y: 100, x: 145)
Coordinates:
  * time     (time) datetime64[ns] 38kB 2012-01-01 2012-01-02 ... 2024-12-31
  * y        (y) float32 400B 0.0 3.246e+04 6.493e+04 ... 3.181e+06 3.214e+06
  * x        (x) float32 580B 3.896e+06 3.928e+06 ... 8.538e+06 8.57e+06
    level    float32 4B ...
    lat      (y, x) float32 58kB ...
    lon      (y, x) float32 58kB ...
Data variables:
    shum     (time, y, x) float32 275MB ...
<xarray.DataArray 'shum' (time: 4749, y: 100, x: 145)> Size: 275MB
[68860500 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 38kB 2012-01-01 2012-01-02 ... 2024-12-31
  * y        (y) float32 400B 0.0 3.246e+04 6.493e+04 ... 3.181e+06 3.214e+06
  * x        (x) float32 580B 3.896e+06 3.928e+06 ... 8.538e+06 8.57e+06
    level    float32 4B ...
    lat      (y, x) float32 58kB ...
    lon      (y, x) float32 58kB ...
Attributes: (12/14)
    GRIB_id:        51
    GRIB_name:      SPFH
    grid_ma

### 9.1 Objective of the mechanism check

In [190]:
df_stage45_objective = pd.DataFrame([
    {
        "component": "Stage 4.5 mechanism check inside Stage 4",
        "objective": (
            "Provide a focused physical consistency test for the Stage 4 synoptic story, "
            "rather than opening a new parallel analysis branch."
        ),
        "core_question": (
            "Assess whether the selected high-signal windows are physically consistent with "
            "reduced precipitation, low-level flow patterns linked to transport or stagnation, "
            "and distinct 850 hPa moisture structure."
        ),
        "scope_control": (
            "This section is intentionally limited to February, May, and July for monthly composites, "
            "with lagged mechanism diagnostics restricted to February and May."
        ),
        "link_to_stage4": (
            "The mechanism check is interpreted only as support for the February benchmark, the May transition case, "
            "and the July wet-season contrast already established in Sections 6–8."
        ),
        "variables_used": (
            "Daily precipitation, H850/U850/V850, and SH850."
        )
    }
])

display(df_stage45_objective)

out_csv = OUT_TAB / "stage4_stage45_mechanism_check_objective.csv"
df_stage45_objective.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,component,objective,core_question,scope_control,link_to_stage4,variables_used
0,Stage 4.5 mechanism check inside Stage 4,Provide a focused physical consistency test fo...,Assess whether the selected high-signal window...,This section is intentionally limited to Febru...,The mechanism check is interpreted only as sup...,"Daily precipitation, H850/U850/V850, and SH850."


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_stage45_mechanism_check_objective.csv


### 9.2 Daily precipitation composites for selected windows

- Do the most polluted cases come with less rainfall and more persistent dry days?

#### 9.2.1 Helpers for precipitation mechanism diagnostics

In [191]:
def area_mean_over_vom_box(da):
    """
    Compute simple spatial mean over the Valley of Mexico box
    using the 2-D lat/lon coordinates.
    """
    lon2d = da["lon"]
    lat2d = da["lat"]

    mask = (
        (lon2d >= DOMAIN_VOM["lon_min"]) & (lon2d <= DOMAIN_VOM["lon_max"]) &
        (lat2d >= DOMAIN_VOM["lat_min"]) & (lat2d <= DOMAIN_VOM["lat_max"])
    )

    return da.where(mask).mean(dim=("y", "x"), skipna=True)


def compute_precip_lag_series(onset_df, month_num, pollutant_label, prcp_da, lag_window=None):
    """
    Compute lagged daily precipitation diagnostics over the Valley of Mexico box
    for one pollutant and one month.
    """
    lag_window = lag_window or list(LAG_DAYS)

    sub = onset_df[onset_df["start_month"] == month_num].copy()
    sub = sub.sort_values("day0").reset_index(drop=True)

    onset_dates = pd.DatetimeIndex(pd.to_datetime(sub["day0"].unique())).normalize()
    avail = pd.DatetimeIndex(pd.to_datetime(prcp_da["time"].values)).normalize()

    rows = []

    # area-mean precipitation time series over VOM box
    prcp_vom = area_mean_over_vom_box(prcp_da)

    for lag in lag_window:
        lag_dates = onset_dates + pd.to_timedelta(lag, unit="D")
        valid_dates = lag_dates[lag_dates.isin(avail)]

        if len(valid_dates) > 0:
            vals = prcp_vom.sel(time=valid_dates).values
            mean_mmday = float(np.nanmean(vals))
            median_mmday = float(np.nanmedian(vals))
            dry_frac = float(np.mean(vals < 1.0))  # fraction of days with <1 mm/day
        else:
            mean_mmday = np.nan
            median_mmday = np.nan
            dry_frac = np.nan

        rows.append({
            "pollutant": pollutant_label,
            "month": month_num,
            "month_name": calendar.month_name[month_num],
            "lag": lag,
            "lag_label": LAG_LABEL_MAP[lag],
            "n_valid_days": len(valid_dates),
            "mean_precip_mmday_vom": mean_mmday,
            "median_precip_mmday_vom": median_mmday,
            "dryday_frac_lt1mm": dry_frac
        })

    df_lag = pd.DataFrame(rows)

    meta = pd.DataFrame([{
        "pollutant": pollutant_label,
        "month": month_num,
        "month_name": calendar.month_name[month_num],
        "n_onset_days": len(onset_dates),
        "n_one_day": int((sub["persistence_class"] == "one_day").sum()),
        "n_persistent": int((sub["persistence_class"] == "persistent").sum())
    }])

    return {"meta": meta, "lag_df": df_lag}


print("Precipitation mechanism helpers loaded.")

Precipitation mechanism helpers loaded.


#### 9.2.2 Precipitation lag summaries for February, May, July

In [192]:
precip_results = []

for pollutant_label, onset_df in [("PM10", evt_pm10_onsets), ("PM2.5", evt_pm25_onsets)]:
    for month_num in FOCUS_MONTHS:
        out = compute_precip_lag_series(
            onset_df=onset_df,
            month_num=month_num,
            pollutant_label=pollutant_label,
            prcp_da=PRCP,
            lag_window=LAG_WINDOW
        )
        precip_results.append(out)

df_precip_mech_meta = pd.concat([r["meta"] for r in precip_results], ignore_index=True)
df_precip_mech_lags = pd.concat([r["lag_df"] for r in precip_results], ignore_index=True)

display(df_precip_mech_meta)
display(df_precip_mech_lags.round(3))

out_csv1 = OUT_TAB / "stage4_precip_mechanism_meta.csv"
out_csv2 = OUT_TAB / "stage4_precip_mechanism_lag_summary.csv"
df_precip_mech_meta.to_csv(out_csv1, index=False)
df_precip_mech_lags.to_csv(out_csv2, index=False)
print(f"saved: {out_csv1}")
print(f"saved: {out_csv2}")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent
0,PM10,2,February,19,9,10
1,PM10,5,May,15,7,8
2,PM10,7,July,29,18,11
3,PM2.5,2,February,20,10,10
4,PM2.5,5,May,18,11,7
5,PM2.5,7,July,29,21,8


,pollutant,month,month_name,lag,lag_label,n_valid_days,mean_precip_mmday_vom,median_precip_mmday_vom,dryday_frac_lt1mm
0,PM10,2,February,-3,-3,19,0.19,0.06,0.90
1,PM10,2,February,-2,-2,19,0.16,0.06,0.95
2,PM10,2,February,-1,-1,19,0.15,0.02,0.95
3,PM10,2,February,0,0,19,0.11,0.06,1.00
4,PM10,2,February,1,+1,19,0.11,0.05,1.00
5,PM10,2,February,2,+2,19,0.21,0.07,0.95
6,PM10,5,May,-3,-3,15,1.44,0.15,0.60
7,PM10,5,May,-2,-2,15,1.22,0.31,0.60
8,PM10,5,May,-1,-1,15,0.96,0.62,0.60
9,PM10,5,May,0,0,15,0.89,0.39,0.60


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_precip_mechanism_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_precip_mechanism_lag_summary.csv


#### 9.2.3 Lagged precipitation mechanism check

In [251]:
month_order = [2, 5, 7]
month_labels = [calendar.month_name[m] for m in month_order]

month_colors = {
    2: "#D37259",  
    5: "#71C2A0",  
    7: "#9999CC"   
}

fig, axes = plt.subplots(2, 1, figsize=(11, 7.8), dpi=250, sharex=True)
fig.subplots_adjust(top=0.84, bottom=0.23, left=0.10, right=0.90, hspace=0.50)

for ax, pollutant_label in zip(axes, ["PM10", "PM2.5"]):
    sub = df_precip_mech_lags[df_precip_mech_lags["pollutant"] == pollutant_label].copy()

    for month_num in month_order:
        d = sub[sub["month"] == month_num].copy().sort_values("lag")

        ax.plot(
            d["lag"], d["mean_precip_mmday_vom"],
            marker="o", linewidth=2,
            color=month_colors[month_num],
            label=calendar.month_abbr[month_num]
        )

    ax.axvline(0, color="k", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.set_ylabel("Mean precipitation over\nValley of Mexico (mm/day)", fontsize=9)
    ax.set_title(
        f"{pollutant_label}: lagged daily precipitation around episode onset",
        fontsize=11,
        fontweight="bold"
    )
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.tick_params(axis="both", labelsize=9)

# Force x labels to appear in subplot 1 even with sharex=True
axes[0].set_xticks(LAG_WINDOW)
axes[0].set_xticklabels([LAG_LABEL_MAP[l] for l in LAG_WINDOW])
axes[0].tick_params(axis="x", labelbottom=True, labelsize=8)

# Set x ticks and labels in subplot 2
axes[1].set_xticks(LAG_WINDOW)
axes[1].set_xticklabels([LAG_LABEL_MAP[l] for l in LAG_WINDOW])
axes[1].set_xlabel("Lag relative to episode onset (days)", fontsize=9)
axes[1].tick_params(axis="x", labelsize=8)

# Get legend from first subplot and place it below, outside both subplots
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.32),
               fontsize=9, frameon=True)

fig.suptitle(
    "Precipitation consistency around onset",
    fontsize=14,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.5, 0.94,
    "Area-mean daily precipitation over the Valley of Mexico | February, May, and July | Day 0 = onset",
    ha="center",
    fontsize=9,
    style="italic"
)

out_fig = OUT_FIG / "stage4_precip_mechanism_lag_check.png"
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_precip_mechanism_lag_check.png


#### 9.2.4 Dry-day persistence summary table

In [216]:
df_precip_dry_summary = (
    df_precip_mech_lags[df_precip_mech_lags["lag"].isin([-1, 0, 1])]
    .groupby(["pollutant", "month", "month_name"], as_index=False)
    .agg({
        "mean_precip_mmday_vom": "mean",
        "dryday_frac_lt1mm": "mean"
    })
    .rename(columns={
        "mean_precip_mmday_vom": "mean_precip_mmday_vom_lagm1_to_p1",
        "dryday_frac_lt1mm": "mean_dryday_fraction_lagm1_to_p1"
    })
    .sort_values(["pollutant", "month"])
    .reset_index(drop=True)
)

display(df_precip_dry_summary.round(3))

out_csv = OUT_TAB / "stage4_precip_dryday_persistence_summary.csv"
df_precip_dry_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,pollutant,month,month_name,mean_precip_mmday_vom_lagm1_to_p1,mean_dryday_fraction_lagm1_to_p1
0,PM10,2,February,0.12,0.98
1,PM10,5,May,1.06,0.58
2,PM10,7,July,3.17,0.22
3,PM2.5,2,February,0.26,0.92
4,PM2.5,5,May,1.55,0.52
5,PM2.5,7,July,5.11,0.09


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_precip_dryday_persistence_summary.csv


### 9.3 850 hPa circulation composites

#### 9.3.1 Helpers for monthly 850 hPa circulation composites

In [225]:
def compute_monthly_field_composites(
    onset_df,
    month_num,
    pollutant_label,
    field_dict
):
    """
    Compute monthly onset-day composites for a set of fields.
    
    Parameters
    ----------
    onset_df : pd.DataFrame
        Episode-onset table (e.g., evt_pm10_onsets)
    month_num : int
        Target month
    pollutant_label : str
        "PM10" or "PM2.5"
    field_dict : dict
        Example:
        {
            "H850": H850_ANOM,
            "U850": U850_ANOM,
            "V850": V850_ANOM,
        }
    """
    sub = onset_df[onset_df["start_month"] == month_num].copy()
    sub = sub.sort_values("day0").reset_index(drop=True)

    onset_dates = pd.DatetimeIndex(pd.to_datetime(sub["day0"].unique())).normalize()

    out = {}
    for name, da in field_dict.items():
        avail = pd.DatetimeIndex(pd.to_datetime(da["time"].values)).normalize()
        valid_dates = onset_dates[onset_dates.isin(avail)]

        if len(valid_dates) > 0:
            out[name] = da.sel(time=valid_dates).mean("time")
        else:
            out[name] = xr.full_like(da.isel(time=0), np.nan)

    meta = pd.DataFrame([{
        "pollutant": pollutant_label,
        "month": month_num,
        "month_name": calendar.month_name[month_num],
        "n_onset_days": len(onset_dates),
        "n_one_day": int((sub["persistence_class"] == "one_day").sum()),
        "n_persistent": int((sub["persistence_class"] == "persistent").sum())
    }])

    return {
        "meta": meta,
        "fields": out,
        "onset_subset": sub,
        "onset_dates": onset_dates
    }


def plot_3month_850_circulation(
    monthly_results,
    pollutant_label,
    out_fig_path,
    contour_step=5,
    vector_step=6
):
    """
    Plot February/May/July onset-day composites at 850 hPa.
    Shading: H850 anomaly
    Contours: H850 anomaly
    Vectors: U850/V850 anomalies
    """
    month_order = [2, 5, 7]

    # Compute shared color scale across the 3 panels
    vals = []
    for m in month_order:
        vals.append(monthly_results[m]["fields"]["H850"].values)
    vals = np.concatenate([v.ravel() for v in vals])
    vals = vals[np.isfinite(vals)]

    max_abs = float(np.nanmax(np.abs(vals))) if vals.size > 0 else 1.0
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        1, 3,
        figsize=(16.5, 5.9),
        dpi=250,
        subplot_kw={"projection": proj}
    )

    pcm_ref = None
    lon2d = monthly_results[2]["fields"]["H850"]["lon"].values
    lat2d = monthly_results[2]["fields"]["H850"]["lat"].values
    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    for ax, m in zip(axes, month_order):
        h = monthly_results[m]["fields"]["H850"].values
        u = monthly_results[m]["fields"]["U850"].values
        v = monthly_results[m]["fields"]["V850"].values
        meta = monthly_results[m]["meta"].iloc[0]

        ax.set_extent(
            [DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
             DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        pcm = ax.pcolormesh(
            lon2d, lat2d, h,
            cmap="RdBu_r",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(lon2d, lat2d, h, levels=pos, colors="k", linewidths=0.7, transform=proj)
        if len(neg) > 0:
            ax.contour(lon2d, lat2d, h, levels=neg, colors="k", linewidths=0.7, linestyles="--", transform=proj)

        yy = np.arange(0, h.shape[0], vector_step)
        xx = np.arange(0, h.shape[1], vector_step)

        ax.quiver(
            lon2d[np.ix_(yy, xx)],
            lat2d[np.ix_(yy, xx)],
            u[np.ix_(yy, xx)],
            v[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        ax.set_title(
            f"{calendar.month_abbr[m]} (N={int(meta['n_onset_days'])})",
            fontsize=12,
            weight="bold"
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if m != 2:
            gl.left_labels = False

    cbar_ax = fig.add_axes([0.32, 0.12, 0.36, 0.04])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("H850′ anomaly (m)")

    fig.suptitle(
        f"{pollutant_label}: 850 hPa circulation composites for selected windows (2012–2024)",
        fontsize=18,
        weight="bold",
        y=0.98
    )
    fig.text(
        0.5, 0.915,
        "Shading/contours: H850′ anomaly | Vectors: Δ(U,V) at 850 hPa | Onset-day composites",
        ha="center",
        fontsize=14,
        style="italic"
    )
    fig.text(
        0.5, 0.88,
        "February, May, and July | Mechanism-oriented consistency check for transport, ventilation, and stagnation",
        ha="center",
        fontsize=11
    )

    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(handles=legend_elements, loc="upper left", fontsize=11, frameon=False, bbox_to_anchor=(0.01, 0.98))

    fig.subplots_adjust(left=0.05, right=0.96, bottom=0.20, top=0.82, wspace=0.10)

    plt.savefig(out_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {out_fig_path}")


print("850 hPa circulation helpers loaded.")

850 hPa circulation helpers loaded.


#### 9.3.2 PM10 / PM2.5 monthly 850 hPa composites

In [218]:
field_dict_850 = {
    "H850": H850_ANOM,
    "U850": U850_ANOM,
    "V850": V850_ANOM,
}

pm10_850_monthly = {}
pm25_850_monthly = {}

for m in FOCUS_MONTHS:
    pm10_850_monthly[m] = compute_monthly_field_composites(
        onset_df=evt_pm10_onsets,
        month_num=m,
        pollutant_label="PM10",
        field_dict=field_dict_850
    )
    pm25_850_monthly[m] = compute_monthly_field_composites(
        onset_df=evt_pm25_onsets,
        month_num=m,
        pollutant_label="PM2.5",
        field_dict=field_dict_850
    )

df_pm10_850_meta = pd.concat([pm10_850_monthly[m]["meta"] for m in FOCUS_MONTHS], ignore_index=True)
df_pm25_850_meta = pd.concat([pm25_850_monthly[m]["meta"] for m in FOCUS_MONTHS], ignore_index=True)

display(df_pm10_850_meta)
display(df_pm25_850_meta)

out_csv1 = OUT_TAB / "stage4_PM10_850_monthly_meta.csv"
out_csv2 = OUT_TAB / "stage4_PM25_850_monthly_meta.csv"
df_pm10_850_meta.to_csv(out_csv1, index=False)
df_pm25_850_meta.to_csv(out_csv2, index=False)
print(f"saved: {out_csv1}")
print(f"saved: {out_csv2}")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent
0,PM10,2,February,19,9,10
1,PM10,5,May,15,7,8
2,PM10,7,July,29,18,11


,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent
0,PM2.5,2,February,20,10,10
1,PM2.5,5,May,18,11,7
2,PM2.5,7,July,29,21,8


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_850_monthly_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_850_monthly_meta.csv


#### 9.3.3 PM10 and PM2.5 850 hPa circulation

In [226]:
plot_3month_850_circulation(
    monthly_results=pm10_850_monthly,
    pollutant_label="PM10",
    out_fig_path=OUT_FIG / "stage4_PM10_850_circulation_selected_windows.png",
    contour_step=5,
    vector_step=6
)

plot_3month_850_circulation(
    monthly_results=pm25_850_monthly,
    pollutant_label="PM2.5",
    out_fig_path=OUT_FIG / "stage4_PM25_850_circulation_selected_windows.png",
    contour_step=5,
    vector_step=6
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_850_circulation_selected_windows.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_850_circulation_selected_windows.png


### 9.4 850 hPa moisture composites

- On the days when extreme weather events begin, is the lower troposphere wetter or drier than normal for that time of year?

#### 9.4.1 Load and prepare SH850

In [242]:
# If SH850 is still missing in memory, load it from the local mini-file.
if SH850 is None:
    ds_sh850_local, var_sh850_local = _open_field_dataset(HERE / "shum850_mex_2012_2024.nc", preferred_keywords=["shum", "q", "spfh"])
    SH850 = ds_sh850_local[var_sh850_local].rename("SH850")
    SH850 = _normalize_time_daily(SH850)

    # Align to common analysis dates and match reanalysis footprint
    SH850 = SH850.sel(time=np.isin(pd.to_datetime(SH850["time"].values).normalize(), pd.DatetimeIndex(df_pm.index).normalize()))
    SH850 = SH850.where(np.isfinite(H500))

    # Derive anomaly if missing
    SH850_ANOM, SH850_CLIM_DOY = compute_daily_anomaly(SH850)

    FIELDS_RAW["SH850"] = SH850
    FIELDS_ANOM["SH850"] = SH850_ANOM
    FIELDS_CLIM_DOY["SH850"] = SH850_CLIM_DOY

    print("SH850 loaded and integrated into Stage 4 memory.")
else:
    print("SH850 already available in memory.")

print(SH850)
print(SH850_ANOM)

SH850 already available in memory.
<xarray.DataArray 'SH850' (time: 4555, y: 90, x: 140)> Size: 230MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..

In [230]:
print("SH850 valid fraction:", float(np.isfinite(SH850.values).mean()))
print("SH850 min/max:", float(np.nanmin(SH850.values)), float(np.nanmax(SH850.values)))
print("SH850_ANOM min/max:", float(np.nanmin(SH850_ANOM.values)), float(np.nanmax(SH850_ANOM.values)))

SH850 valid fraction: 0.7860317460317461
SH850 min/max: 9.282422297474113e-07 0.017886413261294365
SH850_ANOM min/max: -0.010106063447892666 0.008767789229750633


In [231]:
print(SH850.shape, SH850_ANOM.shape)

(4555, 90, 140) (4555, 90, 140)


#### 9.4.2 Helpers for monthly SH850 composites

In [241]:
def plot_3month_850_moisture(
    monthly_results,
    pollutant_label,
    out_fig_path,
    contour_step=0.001,
    vector_step=6,
    use_anomaly=True
):
    """
    Plot February/May/July onset-day moisture composites at 850 hPa.
    Shading: SH850 anomaly
    Contours: SH850 anomaly
    Vectors: U850/V850 anomalies
    """
    month_order = [2, 5, 7]

    vals = []
    for m in month_order:
        vals.append(monthly_results[m]["fields"]["SH850"].values)
    vals = np.concatenate([v.ravel() for v in vals])
    vals = vals[np.isfinite(vals)]

    max_abs = float(np.nanmax(np.abs(vals))) if vals.size > 0 else 1.0
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0

    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        1, 3,
        figsize=(16.5, 5.9),
        dpi=250,
        subplot_kw={"projection": proj}
    )

    pcm_ref = None
    lon2d = monthly_results[2]["fields"]["SH850"]["lon"].values
    lat2d = monthly_results[2]["fields"]["SH850"]["lat"].values
    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)

    for ax, m in zip(axes, month_order):
        sh = monthly_results[m]["fields"]["SH850"].values
        u = monthly_results[m]["fields"]["U850"].values
        v = monthly_results[m]["fields"]["V850"].values
        meta = monthly_results[m]["meta"].iloc[0]

        ax.set_extent(
            [DOMAIN_MAIN["lon_min"], DOMAIN_MAIN["lon_max"],
             DOMAIN_MAIN["lat_min"], DOMAIN_MAIN["lat_max"]],
            crs=proj
        )

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        pcm = ax.pcolormesh(
            lon2d, lat2d, sh,
            cmap="BrBG",
            norm=norm,
            shading="auto",
            transform=proj
        )
        pcm_ref = pcm

        lev = np.arange(-max_abs, max_abs + contour_step, contour_step)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(lon2d, lat2d, sh, levels=pos, colors="k", linewidths=0.7, transform=proj)
        if len(neg) > 0:
            ax.contour(lon2d, lat2d, sh, levels=neg, colors="k", linewidths=0.7, linestyles="--", transform=proj)

        yy = np.arange(0, sh.shape[0], vector_step)
        xx = np.arange(0, sh.shape[1], vector_step)

        ax.quiver(
            lon2d[np.ix_(yy, xx)],
            lat2d[np.ix_(yy, xx)],
            u[np.ix_(yy, xx)],
            v[np.ix_(yy, xx)],
            transform=proj,
            color="black",
            scale=80,
            width=0.0022,
            headwidth=3.5,
            headlength=4.5,
            headaxislength=4.0,
            minlength=0.05,
            pivot="middle",
            zorder=4
        )

        rect = mpatches.Rectangle(
            (VOM_BOX[0], VOM_BOX[1]),
            VOM_BOX[2], VOM_BOX[3],
            fill=False,
            edgecolor="k",
            linewidth=1.0,
            linestyle="-",
            transform=proj
        )
        ax.add_patch(rect)

        ax.plot(
            CDMX_REF["lon"], CDMX_REF["lat"],
            marker="*",
            color="gold",
            markersize=9,
            markeredgecolor="k",
            transform=proj
        )

        ax.set_title(
            f"{calendar.month_abbr[m]} (N={int(meta['n_onset_days'])})",
            fontsize=12,
            weight="bold"
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if m != 2:
            gl.left_labels = False

    cbar_ax = fig.add_axes([0.32, 0.12, 0.36, 0.04])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax, orientation="horizontal", extend="both")
    cb.set_label("SH850′ anomaly (kg/kg)")

    fig.suptitle(
        f"{pollutant_label}: 850 hPa moisture composites for selected windows (2012–2024)",
        fontsize=18,
        weight="bold",
        y=0.98
    )
    fig.text(
        0.5, 0.915,
        "Shading/contours: SH850′ anomaly | Vectors: Δ(U,V) at 850 hPa | Onset-day composites",
        ha="center",
        fontsize=14,
        style="italic"
    )
    fig.text(
        0.5, 0.88,
        "February, May, and July | Mechanism-oriented consistency check for moisture structure around polluted onset",
        ha="center",
        fontsize=11
    )

    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(handles=legend_elements, loc="upper left", fontsize=11, frameon=False, bbox_to_anchor=(0.01, 0.98))

    fig.subplots_adjust(left=0.05, right=0.96, bottom=0.20, top=0.82, wspace=0.10)

    plt.savefig(out_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {out_fig_path}")


print("850 hPa moisture helpers loaded.")

850 hPa moisture helpers loaded.


#### 9.4.3 PM10 / PM2.5 monthly SH850 composites

In [240]:
field_dict_sh850 = {
    "SH850": SH850_ANOM,
    "U850": U850_ANOM,
    "V850": V850_ANOM,
}

pm10_sh850_monthly = {}
pm25_sh850_monthly = {}

for m in FOCUS_MONTHS:
    pm10_sh850_monthly[m] = compute_monthly_field_composites(
        onset_df=evt_pm10_onsets,
        month_num=m,
        pollutant_label="PM10",
        field_dict=field_dict_sh850
    )
    pm25_sh850_monthly[m] = compute_monthly_field_composites(
        onset_df=evt_pm25_onsets,
        month_num=m,
        pollutant_label="PM2.5",
        field_dict=field_dict_sh850
    )

df_pm10_sh850_meta = pd.concat([pm10_sh850_monthly[m]["meta"] for m in FOCUS_MONTHS], ignore_index=True)
df_pm25_sh850_meta = pd.concat([pm25_sh850_monthly[m]["meta"] for m in FOCUS_MONTHS], ignore_index=True)

display(df_pm10_sh850_meta)
display(df_pm25_sh850_meta)

out_csv1 = OUT_TAB / "stage4_PM10_sh850_monthly_meta.csv"
out_csv2 = OUT_TAB / "stage4_PM25_sh850_monthly_meta.csv"
df_pm10_sh850_meta.to_csv(out_csv1, index=False)
df_pm25_sh850_meta.to_csv(out_csv2, index=False)
print(f"saved: {out_csv1}")
print(f"saved: {out_csv2}")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent
0,PM10,2,February,19,9,10
1,PM10,5,May,15,7,8
2,PM10,7,July,29,18,11


,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent
0,PM2.5,2,February,20,10,10
1,PM2.5,5,May,18,11,7
2,PM2.5,7,July,29,21,8


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM10_sh850_monthly_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_PM25_sh850_monthly_meta.csv


#### 9.4.4 PM10 and PM2.5 SH850 composites

In [243]:
plot_3month_850_moisture(
    monthly_results=pm10_sh850_monthly,
    pollutant_label="PM10",
    out_fig_path=OUT_FIG / "stage4_PM10_sh850_selected_windows.png",
    contour_step=0.001,
    vector_step=6
)

plot_3month_850_moisture(
    monthly_results=pm25_sh850_monthly,
    pollutant_label="PM2.5",
    out_fig_path=OUT_FIG / "stage4_PM25_sh850_selected_windows.png",
    contour_step=0.001,
    vector_step=6
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_sh850_selected_windows.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_sh850_selected_windows.png


### 9.5 Lagged mechanism diagnostics

#### 9.5.1 Helpers for lagged mechanism diagnostics

In [269]:
def compute_area_mean_series(da):
    """
    Area mean over the Valley of Mexico box using 2-D lat/lon coords.
    """
    lon2d = da["lon"]
    lat2d = da["lat"]

    mask = (
        (lon2d >= DOMAIN_VOM["lon_min"]) & (lon2d <= DOMAIN_VOM["lon_max"]) &
        (lat2d >= DOMAIN_VOM["lat_min"]) & (lat2d <= DOMAIN_VOM["lat_max"])
    )

    return da.where(mask).mean(dim=("y", "x"), skipna=True)


def compute_lagged_mechanism_series(
    onset_df,
    month_num,
    pollutant_label,
    prcp_da,
    u850_da,
    v850_da,
    sh850_anom_da,
    lag_window=None
):
    """
    Compute lagged mechanism diagnostics over the VOM box
    for one pollutant and one month.

    Returns a tidy DataFrame with:
    - mean precipitation (mm/day)
    - mean 850 hPa wind-anomaly magnitude (same units as U/V, likely m/s)
    - mean SH850 anomaly (kg/kg)
    """
    lag_window = lag_window or list(LAG_DAYS)

    sub = onset_df[onset_df["start_month"] == month_num].copy()
    sub = sub.sort_values("day0").reset_index(drop=True)

    onset_dates = pd.DatetimeIndex(pd.to_datetime(sub["day0"].unique())).normalize()

    prcp_vom = compute_area_mean_series(prcp_da)
    u850_vom = compute_area_mean_series(u850_da)
    v850_vom = compute_area_mean_series(v850_da)
    sh850_vom = compute_area_mean_series(sh850_anom_da)

    # Wind anomaly magnitude over VOM box
    wspd850_vom = np.sqrt(u850_vom**2 + v850_vom**2).rename("WSPD850_ANOM")

    avail = pd.DatetimeIndex(pd.to_datetime(prcp_da["time"].values)).normalize()

    rows = []

    for lag in lag_window:
        lag_dates = onset_dates + pd.to_timedelta(lag, unit="D")
        valid_dates = lag_dates[lag_dates.isin(avail)]

        if len(valid_dates) > 0:
            pr_vals = prcp_vom.sel(time=valid_dates).values
            ws_vals = wspd850_vom.sel(time=valid_dates).values
            sh_vals = sh850_vom.sel(time=valid_dates).values

            rows.append({
                "pollutant": pollutant_label,
                "month": month_num,
                "month_name": calendar.month_name[month_num],
                "lag": lag,
                "lag_label": LAG_LABEL_MAP[lag],
                "n_valid_days": len(valid_dates),
                "mean_precip_mmday_vom": float(np.nanmean(pr_vals)),
                "mean_wspd850_anom_vom": float(np.nanmean(ws_vals)),
                "mean_sh850_anom_vom": float(np.nanmean(sh_vals)),
                "median_precip_mmday_vom": float(np.nanmedian(pr_vals)),
                "median_wspd850_anom_vom": float(np.nanmedian(ws_vals)),
                "median_sh850_anom_vom": float(np.nanmedian(sh_vals)),
            })
        else:
            rows.append({
                "pollutant": pollutant_label,
                "month": month_num,
                "month_name": calendar.month_name[month_num],
                "lag": lag,
                "lag_label": LAG_LABEL_MAP[lag],
                "n_valid_days": 0,
                "mean_precip_mmday_vom": np.nan,
                "mean_wspd850_anom_vom": np.nan,
                "mean_sh850_anom_vom": np.nan,
                "median_precip_mmday_vom": np.nan,
                "median_wspd850_anom_vom": np.nan,
                "median_sh850_anom_vom": np.nan,
            })

    meta = pd.DataFrame([{
        "pollutant": pollutant_label,
        "month": month_num,
        "month_name": calendar.month_name[month_num],
        "n_onset_days": len(onset_dates),
        "n_one_day": int((sub["persistence_class"] == "one_day").sum()),
        "n_persistent": int((sub["persistence_class"] == "persistent").sum())
    }])

    return {
        "meta": meta,
        "lag_df": pd.DataFrame(rows)
    }


def plot_lagged_mechanism_panels(df_mech, pollutant_label, out_fig_path):
    """
    Three-panel lagged mechanism figure for one pollutant:
    - precipitation
    - 850 hPa wind-anomaly magnitude
    - SH850 anomaly
    comparing February vs May vs July
    """
    month_order = [2, 5, 7]
    month_colors = {
        2: "#D37259",   # Feb
        5: "#71C2A0",   # May
        7: "#9999CC",   # Jul
    }

    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8), dpi=250, sharex=True)

    metrics = [
        ("mean_precip_mmday_vom", "Precipitation\n(mm/day)", "Lagged precipitation over VOM"),
        ("mean_wspd850_anom_vom", "850 hPa wind anomaly\nmagnitude Δ(U,V) (m/s)", "Lagged low-level flow over VOM"),
        ("mean_sh850_anom_vom", "SH850′ (kg/kg)", "Lagged moisture anomaly over VOM"),
    ]

    sub = df_mech[df_mech["pollutant"] == pollutant_label].copy()

    for ax, (metric, ylabel, title) in zip(axes, metrics):
        for m in month_order:
            d = sub[sub["month"] == m].sort_values("lag")
            ax.plot(
                d["lag"],
                d[metric],
                marker="o",
                linewidth=2,
                label=calendar.month_abbr[m],
                color=month_colors[m]
            )

        ax.axvline(0, color="k", linestyle="--", linewidth=1.0, alpha=0.7)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", linestyle="--", alpha=0.35)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        ncol=3,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.04),
        frameon=True
    )

    axes[1].set_xlabel("Lag relative to episode onset (days)")
    axes[0].set_xticks(LAG_WINDOW)
    axes[0].set_xticklabels([LAG_LABEL_MAP[l] for l in LAG_WINDOW])

    fig.suptitle(
        f"{pollutant_label}: lagged mechanism diagnostics",
        fontsize=17,
        fontweight="bold",
        y=0.98
    )
    fig.text(
        0.5, 0.895,
        "Valley of Mexico area-mean diagnostics | Day 0 = onset | February benchmark, May transition, and July wet-season contrast",
        ha="center",
        fontsize=11,
        style="italic"
    )

    fig.subplots_adjust(top=0.80, bottom=0.24, wspace=0.28)

    plt.savefig(out_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {out_fig_path}")


print("Lagged mechanism helpers loaded.")

Lagged mechanism helpers loaded.


#### 9.5.2 Compute lagged mechanism diagnostics

In [247]:
mech_results = []

for pollutant_label, onset_df in [("PM10", evt_pm10_onsets), ("PM2.5", evt_pm25_onsets)]:
    for month_num in [2, 5, 7]:
        out = compute_lagged_mechanism_series(
            onset_df=onset_df,
            month_num=month_num,
            pollutant_label=pollutant_label,
            prcp_da=PRCP,
            u850_da=U850_ANOM,
            v850_da=V850_ANOM,
            sh850_anom_da=SH850_ANOM,
            lag_window=LAG_WINDOW
        )
        mech_results.append(out)

df_mech_meta = pd.concat([r["meta"] for r in mech_results], ignore_index=True)
df_mech_lags = pd.concat([r["lag_df"] for r in mech_results], ignore_index=True)

display(df_mech_meta)
display(df_mech_lags.round(4))

out_csv1 = OUT_TAB / "stage4_lagged_mechanism_meta.csv"
out_csv2 = OUT_TAB / "stage4_lagged_mechanism_summary.csv"
df_mech_meta.to_csv(out_csv1, index=False)
df_mech_lags.to_csv(out_csv2, index=False)

print(f"saved: {out_csv1}")
print(f"saved: {out_csv2}")

,pollutant,month,month_name,n_onset_days,n_one_day,n_persistent
0,PM10,2,February,19,9,10
1,PM10,5,May,15,7,8
2,PM10,7,July,29,18,11
3,PM2.5,2,February,20,10,10
4,PM2.5,5,May,18,11,7
5,PM2.5,7,July,29,21,8


,pollutant,month,month_name,lag,lag_label,n_valid_days,mean_precip_mmday_vom,mean_wspd850_anom_vom,mean_sh850_anom_vom,median_precip_mmday_vom,median_wspd850_anom_vom,median_sh850_anom_vom
0,PM10,2,February,-3,-3,19,0.19,1.34,-0.00,0.06,1.18,-0.00
1,PM10,2,February,-2,-2,19,0.16,1.68,-0.00,0.06,1.69,0.00
2,PM10,2,February,-1,-1,19,0.15,1.83,-0.00,0.02,1.71,-0.00
3,PM10,2,February,0,0,19,0.11,1.53,-0.00,0.06,1.67,-0.00
4,PM10,2,February,1,+1,19,0.11,1.62,-0.00,0.05,1.50,-0.00
5,PM10,2,February,2,+2,19,0.21,1.65,-0.00,0.06,1.68,0.00
6,PM10,5,May,-3,-3,15,1.44,1.66,-0.00,0.15,1.41,-0.00
7,PM10,5,May,-2,-2,15,1.22,1.52,-0.00,0.31,1.24,-0.00
8,PM10,5,May,-1,-1,15,0.96,1.75,-0.00,0.62,1.39,-0.00
9,PM10,5,May,0,0,15,0.89,1.23,-0.00,0.39,1.14,-0.00


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_lagged_mechanism_meta.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_lagged_mechanism_summary.csv


#### 9.5.3 PM10 and PM2.5 lagged mechanism diagnostics

In [270]:
plot_lagged_mechanism_panels(
    df_mech=df_mech_lags,
    pollutant_label="PM10",
    out_fig_path=OUT_FIG / "stage4_PM10_lagged_mechanism.png"
)

plot_lagged_mechanism_panels(
    df_mech=df_mech_lags,
    pollutant_label="PM2.5",
    out_fig_path=OUT_FIG / "stage4_PM25_lagged_mechanism.png"
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM10_lagged_mechanism.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\figures\stage4_PM25_lagged_mechanism.png


### 10. Export summary tables

In [272]:
# ---------------------------------------------------------
# 1. Focus-window sample sizes
# ---------------------------------------------------------
df_focus_window_sample_sizes = pd.concat([
    df_focus_months[[
        "month", "month_name", "season", "stage4_role",
        "PM10_episode_count", "PM25_episode_count",
        "sigfrac_PM10", "sigfrac_PM25",
        "PM10_exceed_pct", "PM25_exceed_pct",
        "precip_mmday"
    ]]
], ignore_index=True).copy()

display(df_focus_window_sample_sizes.round(3))

out_csv = OUT_TAB / "stage4_focus_window_sample_sizes_summary.csv"
df_focus_window_sample_sizes.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")


# ---------------------------------------------------------
# 2. Regime counts summary
# ---------------------------------------------------------
df_regime_counts_summary = pd.concat([
    pm10_regime_summary_824.assign(pollutant_label="PM10"),
    pm25_regime_summary_834.assign(pollutant_label="PM2.5")
], ignore_index=True)

df_regime_counts_summary = df_regime_counts_summary[
    [
        "pollutant", "regime_label", "n_onset_days",
        "dominant_month", "dominant_persistence_class",
        "n_February", "n_May", "n_July",
        "interpretation_tag"
    ]
].sort_values(["pollutant", "regime_label"]).reset_index(drop=True)

display(df_regime_counts_summary)

out_csv = OUT_TAB / "stage4_regime_counts_summary.csv"
df_regime_counts_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")


# ---------------------------------------------------------
# 3. Lag-based core findings summary
# ---------------------------------------------------------
df_lag_core_summary = pd.DataFrame([
    {
        "pollutant": "PM10",
        "February_story": "Strongest and clearest lag evolution; robust late-winter benchmark",
        "May_story": "Transition case; weaker and less clean than February",
        "July_story": "Weak/noisy wet-season contrast",
        "regime_story": "R1 captures February benchmark; R2-R3 dominate non-winter onset days"
    },
    {
        "pollutant": "PM2.5",
        "February_story": "Strong late-winter benchmark structure",
        "May_story": "Transition behavior remains meaningful",
        "July_story": "Retains clearer wet-season structure than PM10",
        "regime_story": "Compact February regime plus broader May-July structures"
    }
])

display(df_lag_core_summary)

out_csv = OUT_TAB / "stage4_lag_core_findings_summary.csv"
df_lag_core_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")


# ---------------------------------------------------------
# 4. Mechanism-check summary
# ---------------------------------------------------------
df_mechanism_summary = pd.DataFrame([
    {
        "pollutant": "PM10",
        "February_mechanism": "Persistently dry benchmark; precipitation nearly absent around onset",
        "May_mechanism": "Intermediate transition case with modest rainfall and changing low-level structure",
        "July_mechanism": "Onset linked to temporary rainfall reduction and weaker low-level flow",
        "SH850_note": "No single universal dry-signal pattern; May dryness is more pronounced than February"
    },
    {
        "pollutant": "PM2.5",
        "February_mechanism": "Dry benchmark with coherent circulation support",
        "May_mechanism": "Transition case; onset can occur even as rainfall becomes more relevant",
        "July_mechanism": "Onset persists under wet conditions without a strong rainfall-break signal",
        "SH850_note": "Broader seasonal moisture dependence; May shows clearer negative SH850 anomaly than February"
    }
])

display(df_mechanism_summary)

out_csv = OUT_TAB / "stage4_mechanism_check_summary.csv"
df_mechanism_summary.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

,month,month_name,season,stage4_role,PM10_episode_count,PM25_episode_count,sigfrac_PM10,sigfrac_PM25,PM10_exceed_pct,PM25_exceed_pct,precip_mmday
0,2,February,DJF,Primary high-signal case,19,20,0.64,0.49,72.81,88.67,0.31
1,5,May,MAM,Primary transition case,15,18,0.34,0.35,57.80,97.04,1.91
2,7,July,JJA,Wet-season contrast case,29,29,0.02,0.30,14.82,78.14,4.63


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_focus_window_sample_sizes_summary.csv


,pollutant,regime_label,n_onset_days,dominant_month,dominant_persistence_class,n_February,n_May,n_July,interpretation_tag
0,PM10,R1,10,February,one_day,8,1,1,Late-winter positive-height benchmark regime
1,PM10,R2,25,July,persistent,6,6,13,Weak positive non-winter structure
2,PM10,R3,28,July,one_day,5,8,15,Weak negative non-winter structure
3,PM2.5,R1,7,February,one_day,5,1,1,Late-winter positive-height benchmark regime
4,PM2.5,R2,31,July,one_day,7,11,13,Weak positive non-winter structure
5,PM2.5,R3,29,July,one_day,8,6,15,Weak negative non-winter structure


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_regime_counts_summary.csv


,pollutant,February_story,May_story,July_story,regime_story
0,PM10,Strongest and clearest lag evolution; robust l...,Transition case; weaker and less clean than Fe...,Weak/noisy wet-season contrast,R1 captures February benchmark; R2-R3 dominate...
1,PM2.5,Strong late-winter benchmark structure,Transition behavior remains meaningful,Retains clearer wet-season structure than PM10,Compact February regime plus broader May-July ...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_lag_core_findings_summary.csv


,pollutant,February_mechanism,May_mechanism,July_mechanism,SH850_note
0,PM10,Persistently dry benchmark; precipitation near...,Intermediate transition case with modest rainf...,Onset linked to temporary rainfall reduction a...,No single universal dry-signal pattern; May dr...
1,PM2.5,Dry benchmark with coherent circulation support,Transition case; onset can occur even as rainf...,Onset persists under wet conditions without a ...,Broader seasonal moisture dependence; May show...


saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 4\outputs\tables\stage4_mechanism_check_summary.csv
